In [1]:
# ============================================================
# Clears workspace, clones LabelBench, installs dependencies
# ============================================================

!rm -rf /kaggle/working/*
!git clone https://github.com/EfficientTraining/LabelBench.git
!pip install -r /kaggle/working/LabelBench/requirements.txt
%cd /kaggle/working/LabelBench

Cloning into 'LabelBench'...
remote: Enumerating objects: 2858, done.
remote: Counting objects: 100% (479/479), done.
remote: Compressing objects: 100% (278/278), done.
remote: Total 2858 (delta 248), reused 326 (delta 185), pack-reused 2379 (from 1)
Receiving objects: 100% (2858/2858), 111.50 MiB | 32.24 MiB/s, done.
Resolving deltas: 100% (1483/1483), done.
Updating files: 100% (1415/1415), done.
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-ekxqnf3h
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-ekxqnf3h
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 32.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
ls

configs/        LabelBench/  mp_eval_launcher.py  README.md
docs/           LICENSE      mp_launcher.py       requirements.txt
example_run.sh  main.py      point_evaluation.py  results/


In [ ]:
# import os
# import random
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import numpy as np
# import matplotlib.pyplot as plt
# from collections import defaultdict, Counter
# from torch.utils.data import DataLoader, TensorDataset, Dataset
# from sklearn.manifold import TSNE
# import torchvision.transforms as T
# from torchvision.models import resnet18
# import hdbscan
# import copy
# from torchvision import datasets as tv_datasets, transforms

# # LabelBench Imports
# from LabelBench.skeleton.dataset_skeleton import datasets as DATASET_REGISTRY
# from LabelBench.skeleton.dataset_skeleton import register_dataset, LabelType, TransformDataset

# # ============================================================
# # BULLETPROOF REPRODUCIBILITY LOCK
# # ============================================================
# def set_seed(seed=42):
#     """Forces strict deterministic behavior across different machines/Colab accounts."""
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed(seed)
#         torch.cuda.manual_seed_all(seed)
    
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False
#     os.environ['PYTHONHASHSEED'] = str(seed)
#     os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
#     torch.use_deterministic_algorithms(True, warn_only=False)

# def get_locked_generator(seed=42):
#     g = torch.Generator()
#     g.manual_seed(seed)
#     return g

import os
# 🚨 THESE MUST BE SET BEFORE IMPORTING TORCH
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.manifold import TSNE
import torchvision.transforms as T
from torchvision.models import resnet18
import hdbscan
import copy
from torchvision import datasets as tv_datasets, transforms

# LabelBench Imports
from LabelBench.skeleton.dataset_skeleton import datasets as DATASET_REGISTRY
from LabelBench.skeleton.dataset_skeleton import register_dataset, LabelType, TransformDataset

# ============================================================
# BULLETPROOF REPRODUCIBILITY LOCK
# ============================================================
def set_seed(seed=42):
    """Forces strict deterministic behavior across different machines/Colab accounts."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # 🚨 STRICT LOCK: Crash if a non-deterministic operation is attempted
    torch.use_deterministic_algorithms(True, warn_only=False)

def get_locked_generator(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# ============================================================
# STABLE METRIC LEARNING LOSSES
# ============================================================
def soft_center_loss(z, labels, margin=0.15):
    loss = torch.tensor(0.0, device=z.device)
    for c in torch.unique(labels):
        mask = (labels == c)
        if mask.sum() > 1:
            centroid = z[mask].mean(dim=0, keepdim=True).detach()
            centroid = F.normalize(centroid, dim=1)
            dist = 1 - F.cosine_similarity(z[mask], centroid)
            loss += F.relu(dist - margin).mean()
    return loss

def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

def triplet_loss_batch(z, labels, margin=1.0):
    loss = torch.tensor(0.0, device=z.device)
    valid_triplets = 0
    for i in range(len(z)):
        anchor = z[i]
        pos_mask = (labels == labels[i])
        pos_mask[i] = False
        neg_mask = (labels != labels[i])
        if pos_mask.sum() > 0 and neg_mask.sum() > 0:
            pos = z[pos_mask][torch.randint(0, pos_mask.sum(), (1,))].squeeze(0)
            neg = z[neg_mask][torch.randint(0, neg_mask.sum(), (1,))].squeeze(0)
            loss += F.triplet_margin_loss(anchor.unsqueeze(0), pos.unsqueeze(0), neg.unsqueeze(0), margin=margin)
            valid_triplets += 1
    if valid_triplets > 0:
        loss /= valid_triplets
    return loss

# ============================================================
# DIRECTORIES & CONFIGURATION (STRICT HPTs ONLY)
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
global BASE_DIR

# 🚀 IMPROVEMENT: Increased Max Novelty Buffer from 2000 to 2500
P = 1                              
TOPK_NOVELTY = 400              
MAX_NOVELTY_BUFFER = 2500        

# LOCKED STRICT HPT CONFIGURATION
ALPHA = 0.35              
BETA = 0.01                
DELTA = 15                 
MIN_CLUSTER_SIZE = 150
EPSILON = 0.00
METHOD = "margin_contrastive"

CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat",
    4: "deer", 5: "dog", 6: "frog", 7: "horse",
    8: "ship", 9: "truck"
}

# ============================================================
# DYNAMIC 41-TASK DATASET DEFINITION 
# ============================================================
NUM_TASKS = 41   

class CIFARStream(Dataset):
    def __init__(self, base_ds, indices):
        self.base_ds = base_ds      
        self.indices = indices      

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.base_ds[self.indices[idx]]
        return x, y

def one_hot(y, n=10):
    return F.one_hot(torch.tensor(y), num_classes=n).float()

@register_dataset("splitcifar10", LabelType.MULTI_CLASS)
def get_splitcifar10(_):
    raise RuntimeError("Use splitcifar10_<id>")

base_train_global = tv_datasets.CIFAR10(root="./data", train=True, download=True)
base_train_targets = np.array(base_train_global.targets)

for split_id in range(NUM_TASKS):
    @register_dataset(f"splitcifar10_{split_id}", LabelType.MULTI_CLASS)
    def _make_split(data_dir, split_id=split_id):
        tf = transforms.Compose([transforms.ToTensor()])
        base_train = tv_datasets.CIFAR10(root=data_dir, train=True, download=True)
        base_test  = tv_datasets.CIFAR10(root=data_dir, train=False, download=True)
        
        if split_id == 0:
            indices = [i for i, y in enumerate(base_train_targets) if y in [0, 1]]
        else:
            novel_class_idx = (split_id - 1) // 5
            target_class = novel_class_idx + 2  
            chunk_idx = (split_id - 1) % 5
            
            class_indices = np.where(base_train_targets == target_class)[0]
            start_idx = chunk_idx * 1000
            end_idx = start_idx + 1000
            indices = class_indices[start_idx:end_idx].tolist()

        train_ds = CIFARStream(base_train, indices)
        train_ds = TransformDataset(train_ds, transform=tf, target_transform=lambda y: one_hot(y,10))
        test_ds = TransformDataset(base_test, transform=tf, target_transform=lambda y: one_hot(y,10))
        return train_ds, test_ds, test_ds, None, None, None, 10, [str(i) for i in range(10)]

# ============================================================
# MODEL DEFINITION
# ============================================================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = resnet18(weights='DEFAULT')
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes, bias=False)
        self.scale = 20.0  

    def expand_head(self, new_classes):
        old_w = self.classifier.weight.data.clone()
        old_n = old_w.shape[0]
        new_classifier = nn.Linear(512, new_classes, bias=False).to(old_w.device)
        new_classifier.weight.data[:old_n] = old_w
        self.classifier = new_classifier

    def forward(self, x, labels=None):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = self.embed(z)
        z = F.normalize(z, dim=1)
        W = F.normalize(self.classifier.weight, dim=1)
        cosine = torch.matmul(z, W.t())
        if labels is not None: 
            m = 0.2 
            theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
            target_logits = torch.cos(theta + m)
            one_hot = F.one_hot(labels, num_classes=W.size(0)).float()
            logits = cosine * (1 - one_hot) + target_logits * one_hot
        else:
            logits = cosine
        logits = logits * self.scale
        return logits, z

# ============================================================
# TRACKERS & MEMORY BUFFER
# ============================================================
class DebugCentroidTracker:
    def __init__(self):
        self.reference = {}
        self.history = defaultdict(list)

    def compute_centroids(self, model, memory):
        model.eval()
        centroids = {}
        with torch.no_grad():
            for cls, Xs in memory.items():
                if len(Xs) == 0: continue
                X = torch.stack([x for x, _ in Xs]).to(DEVICE)
                _, Z = model(X)
                mu = F.normalize(Z.mean(0), dim=0)
                centroids[cls] = mu.detach().cpu()
        return centroids

    def snapshot(self, model, memory):
        self.reference = self.compute_centroids(model, memory)

    def measure_drift(self, model, memory, epoch, tag=""):
        current = self.compute_centroids(model, memory)
        for cls in self.reference:
            if cls in current:
                drift = 1 - torch.dot(self.reference[cls], current[cls])
                self.history[(tag, cls)].append(drift.item())

class MemoryBuffer:
    def __init__(self, max_per_class=400):
        self.data = defaultdict(list)
        self.max_per_class = max_per_class
        self.aug = T.Compose([
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip()
        ])

    @torch.no_grad()
    def build_memory_herding(self, X_all, y_label, model):
        model.eval()
        logits_all, Z_all = [], []
        for i in range(0, len(X_all), 128):
            batch = X_all[i:i+128].to(DEVICE)
            logits, z = model(batch)
            logits_all.append(logits.cpu())
            Z_all.append(z.cpu())
        logits_all = torch.cat(logits_all)
        Z_all = torch.cat(Z_all)
        class_mean = F.normalize(Z_all.mean(0), dim=0)
        selected_idx = []
        features = Z_all.clone()

        for k in range(min(self.max_per_class, len(X_all))):
            if k > 0:
                S = Z_all[selected_idx].sum(0)
            else:
                S = torch.zeros_like(class_mean)
            target = (k + 1) * class_mean - S
            distances = torch.norm(features - target, dim=1)
            for idx in selected_idx:
                distances[idx] = float('inf')
            best = distances.argmin().item()
            selected_idx.append(best)

        self.data[int(y_label)] = []
        for idx in selected_idx:
            self.data[int(y_label)].append((X_all[idx].detach().cpu(), logits_all[idx].detach().cpu()))
    
    def get(self): return self.data

    def sample_balanced(self, batch_size, model):
        classes = list(self.data.keys())
        if not classes: return None, None, None
        samples_per_class = max(1, batch_size // len(classes))
        X_mem, Y_mem, L_mem = [], [], []
        current_dim = model.classifier.out_features
        for cls in classes:
            samples = self.data[cls]
            if len(samples) == 0: continue
            replace = len(samples) < samples_per_class
            idx = np.random.choice(len(samples), samples_per_class, replace=replace)
            for i in idx:
                x, logit = samples[i]
                if logit.shape[0] < current_dim:
                    padded = torch.zeros(current_dim)
                    padded[:logit.shape[0]] = logit
                    logit = padded
                X_mem.append(x)
                Y_mem.append(cls)
                L_mem.append(logit)
        if not X_mem: return None, None, None
        X_tensor = torch.stack(X_mem)
        X_tensor = self.aug(X_tensor) 
        return X_tensor, torch.tensor(Y_mem), torch.stack(L_mem)

class HypersphereNovelty:
    def __init__(self, q=0.90):
        self.q = q
        self.mu, self.r = {}, {}

    def update(self, memory, model):
        self.mu, self.r = {}, {}
        for k, X_tuples in memory.items():
            if len(X_tuples) == 0: continue
            X = torch.stack([x for x, _ in X_tuples]).to(DEVICE)
            with torch.no_grad(): 
                _, Z = model(X)
            mu = F.normalize(Z.mean(0), dim=0)
            d = 1 - torch.matmul(Z, mu)
            self.mu[k] = mu
            self.r[k] = torch.quantile(d, self.q)

    def score(self, z):
        if not self.mu: return torch.tensor(0.0)
        return min([(1 - torch.dot(z, self.mu[k].to(z.device))) - self.r[k].to(z.device) for k in self.mu])

# ============================================================
# TRAINING LOOPS
# ============================================================
def train_supervised(model, loader, test_loader, task_id, method="baseline_ce"):
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    
    for epoch in range(30):
        for x, y in loader:
            x_device, y_device = x.to(DEVICE), y.argmax(1).to(DEVICE)
            x_aug = aug(x_device)
            logits, z = model(x_aug, y_device)
            loss_ce = F.cross_entropy(logits, y_device)
            z_norm = F.normalize(z, dim=1)
            
            if method == "baseline_ce": loss = loss_ce
            elif method == "soft_center_loss": loss = loss_ce + 1.0 * soft_center_loss(z_norm, y_device, margin=0.15)
            elif method == "margin_contrastive": loss = loss_ce + 1.0 * margin_contrastive_loss(z_norm, y_device)
            elif method == "triplet": loss = loss_ce + 1.0 * triplet_loss_batch(z_norm, y_device, margin=1.0)
                
            opt.zero_grad()
            loss.backward()
            opt.step()

def finetune(model, memory, X_new, new_label, task_id, test_loader, locked_gen): 
    old_model = copy.deepcopy(model).eval()
    for p in old_model.parameters(): p.requires_grad = False 
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()
            
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    Y_new = torch.full((len(X_new),), new_label, dtype=torch.long)
    
    loader = DataLoader(TensorDataset(X_new, Y_new), batch_size=32, shuffle=True, generator=locked_gen)

    for p in model.parameters(): p.requires_grad = True
    opt = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

    # 🚀 IMPROVEMENT: Increased Finetuning Epochs from 15 to 25
    for epoch in range(25):
        for xb, yb in loader:
            xb = aug(xb) 
            X_mem, Y_mem, L_mem = memory.sample_balanced(32, model)
            if X_mem is not None:
                xb = torch.cat([xb, X_mem], dim=0)
                yb = torch.cat([yb, Y_mem], dim=0)

            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits_margin, Z = model(xb, yb)
            loss_ce = F.cross_entropy(logits_margin, yb)
            
            if X_mem is not None:
                pure_logits, _ = model(xb) 
                logits_mem = pure_logits[-len(X_mem):]
                loss_der = F.mse_loss(logits_mem, L_mem.to(DEVICE))
            else:
                loss_der = torch.tensor(0.0, device=DEVICE)

            with torch.no_grad(): 
                logits_old, Z_old = old_model(xb)

            loss_feat = (1 - F.cosine_similarity(Z, Z_old)).mean()
            loss = loss_ce + 0.5 * loss_der + 1.0 * loss_feat
            
            opt.zero_grad()
            loss.backward()
            opt.step()
    model.eval()

def evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies):
    correct_per_class = defaultdict(int)
    total_per_class = defaultdict(int)
    model.eval()
    with torch.no_grad():
        for x_test, y_test in DataLoader(test_ds, batch_size=128):
            x_test = x_test.to(DEVICE)
            y_test = y_test.argmax(1).to(DEVICE)
            logits, _ = model(x_test)
            preds = logits.argmax(1)
            
            for i in range(len(y_test)):
                sem = int(y_test[i])
                if sem in semantic_to_internal:
                    internal_gt = semantic_to_internal[sem]
                    total_per_class[sem] += 1
                    if preds[i].item() == internal_gt:
                        correct_per_class[sem] += 1

    current_class_accs = {}
    for sem in semantic_to_internal.keys():
        current_class_accs[sem] = correct_per_class[sem] / max(total_per_class[sem], 1)
        
    class_accuracies[t] = current_class_accs
    total_correct = sum(correct_per_class.values())
    total_eval = sum(total_per_class.values())
    acc = total_correct / max(total_eval, 1)
    
    if t > 0:
        forgetting_list = []
        for sem in current_class_accs.keys():
            past_accs = [class_accuracies[k].get(sem, None) for k in range(t)]
            past_accs = [a for a in past_accs if a is not None]
            if past_accs:
                max_past = max(past_accs)
                forgetting = max_past - current_class_accs[sem]
                forgetting_list.append(forgetting)
        avg_forg = np.mean(forgetting_list) if forgetting_list else 0.0
    else:
        avg_forg = 0.0
        
    return acc, avg_forg, current_class_accs


# ============================================================
# GRAND SINGLE RUN: OPTIMIZED STRICT + MARGIN CONTRASTIVE
# ============================================================

print("\n" + "="*80)
print(f"🚀 RUNNING OPTIMIZED PIPELINE: [Strict] | METHOD [{METHOD.upper()}]")
print("="*80)

set_seed(42)
locked_generator = get_locked_generator(42)

BASE_DIR = f"debug_OptimizedStrict_{METHOD}"
os.makedirs(BASE_DIR, exist_ok=True)

task_kca, task_cpr = [], []
average_forgetting = [] 
class_accuracies = {}   
learned_classes_over_time = []
promoted_classes_log = []

TASKS = [f"splitcifar10_{i}" for i in range(NUM_TASKS)]

model = CNN(num_classes=2).to(DEVICE)

# 🚀 IMPROVEMENT: Increased Replay Memory max_per_class from 200 to 400
memory = MemoryBuffer(max_per_class=400) 
detector = HypersphereNovelty()
novelty_buffer = []

semantic_to_internal = {0: 0, 1: 1}
internal_to_semantic = {0: 0, 1: 1}
known_classes = 2

for t, task in enumerate(TASKS):
    _, dataset_fn = DATASET_REGISTRY[task]
    train_ds, test_ds, *_ = dataset_fn("./data")
    loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=locked_generator) 

    print(f"\n{'='*20} 🚀 TASK {t} {'='*20}")
    all_semantics = []
    for _, y in loader: all_semantics.extend(y.argmax(1).tolist())
    unique_semantics = list(set(all_semantics))
    class_names = [CIFAR10_LABELS[sem] for sem in unique_semantics]
    print(f"📦 [STREAM] Task {t} stream contains class(es): {class_names} (Label IDs: {unique_semantics})")
    print(f"📊 [STREAM] Total images in this chunk: {len(train_ds)}")

    if t == 0:
        print(f"🎓 [INIT] Running '{METHOD}' initialization on Task 0...")
        train_supervised(model, loader, DataLoader(test_ds, batch_size=128, shuffle=False), t, method=METHOD)
        for cls in [0, 1]:
            Xc = torch.cat([x[y.argmax(1) == cls] for x, y in loader])
            memory.build_memory_herding(Xc, cls, model)
        detector.update(memory.get(), model)
        learned_classes_over_time.append({CIFAR10_LABELS[0], CIFAR10_LABELS[1]})
        print(f"✅ [INIT COMPLETE] Known classes: {learned_classes_over_time[-1]}")
        
        acc, avg_forg, current_class_accs = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
        task_kca.append(acc)
        average_forgetting.append(avg_forg)
        print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")
        for sem, c_acc in current_class_accs.items():
            print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")
        continue

    model.eval()
    novelty_candidates = []
    novel, false_novel, total = 0, 0, 0
    
    with torch.no_grad():
        for x, y in loader:
            _, z = model(x.to(DEVICE))
            y_labels = y.argmax(1)
            scores = [detector.score(z[i]).item() for i in range(len(z))]
            thr = np.percentile(scores, 30)
            for i in range(len(z)):
                total += 1
                if scores[i] > thr:
                    novelty_candidates.append((scores[i], x[i].cpu(), y_labels[i].item()))
                    novel += 1
                    if y_labels[i].item() in semantic_to_internal:
                        false_novel += 1

    print(f"🔍 [NOVELTY] Flagged Novel: {novel}/{total} | False Novelty (Knowns flagged): {false_novel}")

    novelty_candidates.sort(reverse=True, key=lambda x: x[0])
    novelty_buffer.extend([(img, y) for _, img, y in novelty_candidates[:TOPK_NOVELTY]])
    novelty_buffer = novelty_buffer[-MAX_NOVELTY_BUFFER:]

    if t % P == 0 and len(novelty_buffer) >= 20: 
        print(f"\n🧠 [CLUSTERING] Running HDBSCAN on novelty buffer (Size: {len(novelty_buffer)})...")
        
        Z = []
        with torch.no_grad():
            for img, _ in novelty_buffer:
                _, z = model(img.unsqueeze(0).to(DEVICE))
                Z.append(z.squeeze().cpu().numpy())

        Z = np.stack(Z)
        labels = hdbscan.HDBSCAN(metric='euclidean', min_cluster_size=20, cluster_selection_epsilon=EPSILON).fit_predict(Z)
        
        new_buffer = []
        found, promoted = 0, 0

        for cid in sorted(set(labels)):
            idxs = np.where(labels == cid)[0]
            if cid == -1:
                print(f"  🗑️  [CLUSTER -1] NOISE: Found {len(idxs)} noise samples.")
                for i in idxs: new_buffer.append(novelty_buffer[i])
                continue

            found += 1
            Xc = torch.stack([novelty_buffer[i][0] for i in idxs])
            with torch.no_grad(): _, Zc = model(Xc.to(DEVICE))
            
            mu = F.normalize(Zc.mean(0), dim=0)
            n = len(idxs)
            S_intra = torch.mean(1 - torch.matmul(Zc, mu))
            
            if len(detector.mu) > 0:
                S_known = min([1 - torch.dot(mu, detector.mu[k].to(mu.device)) for k in detector.mu])
            else:
                S_known = torch.tensor(1.0)

            density = n / (S_intra.item() + 1e-6)
            margin = S_known - S_intra
            
            labels_true = [novelty_buffer[i][1] for i in idxs]
            sem_label, cnt = Counter(labels_true).most_common(1)[0]
            purity = cnt / len(labels_true) 
            
            if sem_label in semantic_to_internal:
                print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Dominant class '{CIFAR10_LABELS[sem_label]}' is already known.")
                continue 

            cond_intra = S_intra.item() <= ALPHA
            cond_density = density >= DELTA
            cond_known = S_known.item() >= BETA
            cond_margin = margin.item() > -0.10
            cond_size = n >= MIN_CLUSTER_SIZE

            print(f"\n  📊 [CLUSTER {cid} EVALUATION] Dominant: '{CIFAR10_LABELS[sem_label]}' (Purity: {purity:.2f})")
            print(f"     ➔ Size:    {n:3d}   (Req: >= {MIN_CLUSTER_SIZE:2d})   {'✅' if cond_size else '❌'}")
            print(f"     ➔ S_intra: {S_intra.item():.3f} (Req: <= {ALPHA:.2f}) {'✅' if cond_intra else '❌'}")
            print(f"     ➔ S_known: {S_known.item():.3f} (Req: >= {BETA:.2f}) {'✅' if cond_known else '❌'}")
            print(f"     ➔ Density: {density:.1f} (Req: >= {DELTA})   {'✅' if cond_density else '❌'}")
            print(f"     ➔ Margin:  {margin.item():.3f} (Req: > -0.10) {'✅' if cond_margin else '❌'}")

            if cond_intra and cond_density and cond_known and cond_margin and cond_size:
                promoted += 1
                new_label = known_classes
                semantic_to_internal[sem_label] = new_label
                internal_to_semantic[new_label] = sem_label
                known_classes += 1
                
                # ---> ADD THIS LINE HERE <---
                torch.save(Xc.cpu(), f"{BASE_DIR}/promoted_class_{sem_label}.pt")
                
                model.expand_head(known_classes)
                model.to(DEVICE)
                
                print(f"     🎉 [PROMOTION SUCCESS] -> Preparing to Finetune '{CIFAR10_LABELS[sem_label]}'...")
                finetune(model, memory, Xc, new_label, t, DataLoader(test_ds, batch_size=128, shuffle=False), locked_generator)
                print(f"     ✨ [LEARNED] -> Network successfully adapted to '{CIFAR10_LABELS[sem_label]}'")
                
                memory.build_memory_herding(Xc, new_label, model)
                detector.update(memory.get(), model)
                
                promoted_classes_log.append({"task": t, "semantic": CIFAR10_LABELS[sem_label]})
            else:
                print(f"     🛑 [PROMOTION FAILED] -> Conditions not met. Retaining samples in buffer.")
                for i in idxs: new_buffer.append(novelty_buffer[i])
                
        novelty_buffer = new_buffer
        task_cpr.append(promoted / max(found, 1))
    else:
        task_cpr.append(0.0)

    # FLUSH BUFFER AFTER EVERY 5 TASKS
    if t > 0 and t % 5 == 0:
        novelty_buffer = [] 
        print(f"\n🧹 [BUFFER CLEAR] Task {t} marks the end of a class stream! Flushed the novelty buffer for the next class.")

    learned_classes_over_time.append(set(learned_classes_over_time[-1]) if len(learned_classes_over_time) > 0 else set())
    for p in promoted_classes_log:
        if p["task"] == t and p["semantic"] not in learned_classes_over_time[-1]:
            learned_classes_over_time[-1].add(p["semantic"])

    acc, avg_forg, current_class_accs = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
    task_kca.append(acc)
    average_forgetting.append(avg_forg)
    
    print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")
    for sem, c_acc in current_class_accs.items():
        print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")

# Store final results
final_accs = class_accuracies[NUM_TASKS - 1]
row_result = {
    "HPT": "Strict (Optimized)",
    "Method": METHOD,
    "Final_KCA": f"{task_kca[-1]:.3f}",
    "Avg_Forg": f"{average_forgetting[-1]:.3f}",
    "Classes_Learned": str(len(learned_classes_over_time[-1]))
}
for cls_id in range(10):
    row_result[f"Cls{cls_id}_{CIFAR10_LABELS[cls_id][:3]}"] = f"{final_accs.get(cls_id, 0.0):.2f}"

final_results_summary = [row_result]

# ============================================================
# CREATE & SAVE THE DESCRIPTIVE RESULT TABLE (PNG)
# ============================================================
print("\n" + "="*120)
print("📊 FINAL OPTIMIZED RUN SUMMARY (10-Class Partial Open World)")
print("="*120)

columns = ["HPT", "Method", "Final KCA", "Avg Forg", "Classes Learned"] + [f"Cls{i}" for i in range(10)]
cell_text = []

# Print to console
header_format = "{:<18} | {:<18} | {:<9} | {:<8} | {:<15} | " + " | ".join([f"{{:<4}}"]*10)
print(header_format.format(*columns))
print("-" * 140)

for res in final_results_summary:
    row = [res["HPT"], res["Method"], res["Final_KCA"], res["Avg_Forg"], res["Classes_Learned"]]
    row += [res[k] for k in list(res.keys())[5:]]
    cell_text.append(row)
    print(header_format.format(*row))

# Generate PNG Image of the Table
fig, ax = plt.subplots(figsize=(24, len(cell_text) * 0.5 + 2))
ax.axis('off')
ax.axis('tight')

table = ax.table(cellText=cell_text, colLabels=columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.8)

for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#4c72b0')
    else:
        if i % 2 == 0:
            cell.set_facecolor('#f2f2f2')

plt.title("Optimized Run Summary (Strict + Margin Contrastive)", fontsize=16, fontweight='bold', pad=20)
plt.savefig("final_optimized_summary_table.png", bbox_inches='tight', dpi=300)
plt.close()

print("\n✅ Script complete. Descriptive table saved to 'final_optimized_margincontrastive_stricthpt_summary_table.png'")

In [ ]:
import os
import glob
import zipfile

# ============================================================
# ZIP RESULTS FOR MANUAL DOWNLOAD
# ============================================================
print("\n📦 Zipping all 'debug_OptimizedStrict_*' folders and summaries for manual download...")
zip_filename = "latest_optimized_tensors_and_results.zip"

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    # 1. Zip the main output directories (which now contain the .pt files)
    for folder in glob.glob("debug_OptimizedStrict_*"):
        for root, dirs, files in os.walk(folder):
            for file in files:
                file_path = os.path.join(root, file)
                zipf.write(file_path, arcname=file_path)
    
    # 2. Zip the generated descriptive table PNG if it exists
    summary_png = "final_optimized_summary_table.png"
    if os.path.exists(summary_png):
        zipf.write(summary_png, arcname=summary_png)

print(f"✅ Successfully created '{zip_filename}'!")
print("You can now download this ZIP file to your disk.")

### joint upper bound of strict margin contrastive

In [ ]:
import os
import glob
# 🚨 THESE MUST BE SET BEFORE IMPORTING TORCH
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as T
from torchvision.models import resnet18
from torchvision import datasets as tv_datasets

# ============================================================
# BULLETPROOF REPRODUCIBILITY LOCK
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=False)

def get_locked_generator(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
METHOD = "margin_contrastive"
BASE_DIR = f"debug_OptimizedStrict_{METHOD}"

CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat",
    4: "deer", 5: "dog", 6: "frog", 7: "horse",
    8: "ship", 9: "truck"
}

# ============================================================
# STABLE METRIC LEARNING LOSSES
# ============================================================
def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

# ============================================================
# MODEL DEFINITION (Native 10-Class for Upper Bound)
# ============================================================
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        base = resnet18(weights='DEFAULT')
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes, bias=False)
        self.scale = 20.0  

    def forward(self, x, labels=None):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = self.embed(z)
        z = F.normalize(z, dim=1)
        W = F.normalize(self.classifier.weight, dim=1)
        cosine = torch.matmul(z, W.t())
        if labels is not None: 
            m = 0.2 
            theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
            target_logits = torch.cos(theta + m)
            one_hot = F.one_hot(labels, num_classes=W.size(0)).float()
            logits = cosine * (1 - one_hot) + target_logits * one_hot
        else:
            logits = cosine
        logits = logits * self.scale
        return logits, z

# ============================================================
# TRAINING HELPER
# ============================================================
def train_supervised(model, loader):
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    
    for epoch in range(30):
        total_loss = 0
        for x, y in loader:
            x_device, y_device = x.to(DEVICE), y.to(DEVICE)
            x_aug = aug(x_device)
            logits, z = model(x_aug, y_device)
            loss_ce = F.cross_entropy(logits, y_device)
            z_norm = F.normalize(z, dim=1)
            
            loss = loss_ce + 1.0 * margin_contrastive_loss(z_norm, y_device)
                
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
            
        if (epoch + 1) % 5 == 0:
            print(f"   ➔ Epoch {epoch + 1}/30 | Loss: {total_loss/len(loader):.4f}")

# ============================================================
# DYNAMIC JOINT DATASET PREPARATION
# ============================================================
print("\n" + "="*80)
print(f"🚀 STARTING ALGORITHM-SPECIFIC JOINT UPPER BOUND | {METHOD.upper()}")
print("="*80)

set_seed(42) 
locked_generator = get_locked_generator(42)

print("📦 Fetching Base Datasets (Airplanes & Autos)...")
tf = T.ToTensor()
base_train = tv_datasets.CIFAR10(root="./data", train=True, download=True)
base_test  = tv_datasets.CIFAR10(root="./data", train=False, download=True)

X_train_joint, Y_train_joint = [], []

# 1. Add Base Classes (0 and 1)
for img, target in base_train:
    if target in [0, 1]:
        X_train_joint.append(tf(img))
        Y_train_joint.append(target)

X_train_joint = torch.stack(X_train_joint)
Y_train_joint = torch.tensor(Y_train_joint, dtype=torch.long)
print(f"   ➔ Base Data Loaded: {len(X_train_joint)} images (Classes 0 & 1)")

# 2. Add Promoted Data Discovered by Code 2.1
X_novel_list, Y_novel_list = [], []
promoted_files = glob.glob(f"{BASE_DIR}/promoted_class_*.pt")

if not promoted_files:
    print(f"\n⚠️ WARNING: No promoted classes found in '{BASE_DIR}'.")
    print("Ensure Code 2.1 ran completely and saved the tensors to disk.")
else:
    for pt_file in promoted_files:
        sem_label = int(os.path.basename(pt_file).replace("promoted_class_", "").replace(".pt", ""))
        X_novel = torch.load(pt_file)
        Y_novel = torch.full((len(X_novel),), sem_label, dtype=torch.long)
        X_novel_list.append(X_novel)
        Y_novel_list.append(Y_novel)
        print(f"   ➔ Loaded '{CIFAR10_LABELS[sem_label]}' (Class {sem_label}): {len(X_novel)} validated images.")

    X_train_joint = torch.cat([X_train_joint] + X_novel_list)
    Y_train_joint = torch.cat([Y_train_joint] + Y_novel_list)

print(f"📊 Final Joint Dataset Size: {len(X_train_joint)} images across {len(torch.unique(Y_train_joint))} classes.")

train_ds = TensorDataset(X_train_joint, Y_train_joint)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=locked_generator)

# Build Full 10-Class Test Loader
X_test, Y_test = [], []
for img, target in base_test:
    X_test.append(tf(img))
    Y_test.append(target)
X_test = torch.stack(X_test)
Y_test = torch.tensor(Y_test, dtype=torch.long)
test_ds = TensorDataset(X_test, Y_test)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

# ============================================================
# TRAINING & EVALUATION
# ============================================================
model = CNN(num_classes=10).to(DEVICE)
print(f"\n🎓 Training static Upper Bound model on exact CL data subset...")
train_supervised(model, train_loader)

model.eval()
correct_per_class = {i: 0 for i in range(10)}
total_per_class = {i: 0 for i in range(10)}

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits, _ = model(x)
        preds = logits.argmax(1)
        for i in range(len(y)):
            label = int(y[i])
            total_per_class[label] += 1
            if preds[i] == label:
                correct_per_class[label] += 1

# Calculate metrics
class_accuracies = {}
for i in range(10):
    class_accuracies[i] = correct_per_class[i] / max(total_per_class[i], 1)

# Only calculate KCA for classes the model actually saw
known_classes = torch.unique(Y_train_joint).tolist()
total_correct_known = sum([correct_per_class[c] for c in known_classes])
total_eval_known = sum([total_per_class[c] for c in known_classes])
kca = total_correct_known / max(total_eval_known, 1)

# ============================================================
# GRAND FINAL UPPER-BOUND TABLE
# ============================================================
print("\n" + "="*145)
print("🏆 ALGORITHM-SPECIFIC JOINT UPPER-BOUND SUMMARY (Zero Forgetting on Discovered Data)")
print("="*145)

columns = ["Method", "UB-KCA"] + [f"Cls{i}_{CIFAR10_LABELS[i][:3]}" for i in range(10)]
header_format = "{:<20} | {:<8} | " + " | ".join([f"{{:<8}}"]*10)

print(header_format.format(*columns))
print("-" * 145)

row_data = [METHOD, f"{kca:.3f}"] + [f"{class_accuracies[i]:.3f}" for i in range(10)]
print(header_format.format(*row_data))
print("="*145)

## introduction of 2-3 classes at a time (open world 3 waves 13 tasks each)

In [6]:
import os
# 🚨 THESE MUST BE SET BEFORE IMPORTING TORCH
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.manifold import TSNE
import torchvision.transforms as T
from torchvision.models import resnet18
import hdbscan
import copy
from torchvision import datasets as tv_datasets, transforms

# LabelBench Imports
from LabelBench.skeleton.dataset_skeleton import datasets as DATASET_REGISTRY
from LabelBench.skeleton.dataset_skeleton import register_dataset, LabelType, TransformDataset

# ============================================================
# BULLETPROOF REPRODUCIBILITY LOCK
# ============================================================
def set_seed(seed=42):
    """Forces strict deterministic behavior across different machines/Colab accounts."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # 🚨 STRICT LOCK: Crash if a non-deterministic operation is attempted
    torch.use_deterministic_algorithms(True, warn_only=False)

def get_locked_generator(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# ============================================================
# STABLE METRIC LEARNING LOSSES
# ============================================================
def soft_center_loss(z, labels, margin=0.15):
    loss = torch.tensor(0.0, device=z.device)
    for c in torch.unique(labels):
        mask = (labels == c)
        if mask.sum() > 1:
            centroid = z[mask].mean(dim=0, keepdim=True).detach()
            centroid = F.normalize(centroid, dim=1)
            dist = 1 - F.cosine_similarity(z[mask], centroid)
            loss += F.relu(dist - margin).mean()
    return loss

def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

def triplet_loss_batch(z, labels, margin=1.0):
    loss = torch.tensor(0.0, device=z.device)
    valid_triplets = 0
    for i in range(len(z)):
        anchor = z[i]
        pos_mask = (labels == labels[i])
        pos_mask[i] = False
        neg_mask = (labels != labels[i])
        if pos_mask.sum() > 0 and neg_mask.sum() > 0:
            pos = z[pos_mask][torch.randint(0, pos_mask.sum(), (1,))].squeeze(0)
            neg = z[neg_mask][torch.randint(0, neg_mask.sum(), (1,))].squeeze(0)
            loss += F.triplet_margin_loss(anchor.unsqueeze(0), pos.unsqueeze(0), neg.unsqueeze(0), margin=margin)
            valid_triplets += 1
    if valid_triplets > 0:
        loss /= valid_triplets
    return loss

# ============================================================
# DIRECTORIES & CONFIGURATION (STRICT HPTs ONLY)
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
global BASE_DIR

P = 1                              
TOPK_NOVELTY = 400              
MAX_NOVELTY_BUFFER = 2500        

# LOCKED STRICT HPT CONFIGURATION
ALPHA = 0.35              
BETA = 0.01                
DELTA = 15                 
MIN_CLUSTER_SIZE = 150
EPSILON = 0.00
METHOD = "margin_contrastive"

CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat",
    4: "deer", 5: "dog", 6: "frog", 7: "horse",
    8: "ship", 9: "truck"
}

# ============================================================
# DYNAMIC 40-TASK DATASET DEFINITION (MULTI-CLASS WAVE LOGIC)
# ============================================================
NUM_TASKS = 40   

class CIFARStream(Dataset):
    def __init__(self, base_ds, indices):
        self.base_ds = base_ds      
        self.indices = indices      

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.base_ds[self.indices[idx]]
        return x, y

def one_hot(y, n=10):
    return F.one_hot(torch.tensor(y), num_classes=n).float()

@register_dataset("splitcifar10", LabelType.MULTI_CLASS)
def get_splitcifar10(_):
    raise RuntimeError("Use splitcifar10_<id>")

# ------------------------------------------------------------
# Build task stream ONCE (Exhaust + Replay)
# ------------------------------------------------------------
base_train_global = tv_datasets.CIFAR10(root="./data", train=True, download=True)
targets = np.array(base_train_global.targets)

# Group indices by class
class_indices = {c: np.where(targets == c)[0] for c in range(10)}

rng = np.random.default_rng(42)
for c in range(10):
    rng.shuffle(class_indices[c])

stream_splits = {}
for t in range(1, NUM_TASKS):
    stream_splits[t] = []
    
    # 1. Add 100 Reminders for previously exhausted classes
    if 1 <= t <= 13:
        reminders = [0, 1]
    elif 14 <= t <= 26:
        reminders = [0, 1, 2, 3]
    elif 27 <= t <= 39:
        reminders = [0, 1, 2, 3, 4, 5, 6]
    else:
        reminders = []
        
    for c in reminders:
        stream_splits[t].extend(rng.choice(class_indices[c], 100, replace=False))
        
    # 2. Add New Wave active classes (exhausting ~5000 images over 13 tasks)
    if 1 <= t <= 13:
        stream_splits[t].extend(np.array_split(class_indices[2], 13)[t-1])
        stream_splits[t].extend(np.array_split(class_indices[3], 13)[t-1])
    elif 14 <= t <= 26:
        stream_splits[t].extend(np.array_split(class_indices[4], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[5], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[6], 13)[t-14])
    elif 27 <= t <= 39:
        stream_splits[t].extend(np.array_split(class_indices[7], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[8], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[9], 13)[t-27])
        
    rng.shuffle(stream_splits[t])

for split_id in range(NUM_TASKS):
    @register_dataset(f"splitcifar10_{split_id}", LabelType.MULTI_CLASS)
    def _make_split(data_dir, split_id=split_id):
        tf = transforms.Compose([transforms.ToTensor()])
        base_train = tv_datasets.CIFAR10(root=data_dir, train=True, download=True)
        base_test  = tv_datasets.CIFAR10(root=data_dir, train=False, download=True)
        
        # ----------------------------------------------------
        # OPEN WORLD LOGIC: 
        # Task 0 gets knowns. Everything else gets the stream.
        # ----------------------------------------------------
        if split_id == 0:
            indices = [i for i,(x,y) in enumerate(base_train) if y in [0,1]]
        else:
            indices = stream_splits[split_id]

        train_ds = CIFARStream(base_train, indices)

        train_ds = TransformDataset(
            train_ds,
            transform=tf,
            target_transform=lambda y: one_hot(y,10)
        )

        test_ds = TransformDataset(
            base_test,
            transform=tf,
            target_transform=lambda y: one_hot(y,10)
        )

        return train_ds, test_ds, test_ds, None, None, None, 10, [str(i) for i in range(10)]

# ============================================================
# MODEL DEFINITION
# ============================================================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = resnet18(weights='DEFAULT')
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes, bias=False)
        self.scale = 20.0  

    def expand_head(self, new_classes):
        old_w = self.classifier.weight.data.clone()
        old_n = old_w.shape[0]
        new_classifier = nn.Linear(512, new_classes, bias=False).to(old_w.device)
        new_classifier.weight.data[:old_n] = old_w
        self.classifier = new_classifier

    def forward(self, x, labels=None):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = self.embed(z)
        z = F.normalize(z, dim=1)
        W = F.normalize(self.classifier.weight, dim=1)
        cosine = torch.matmul(z, W.t())
        if labels is not None: 
            m = 0.2 
            theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
            target_logits = torch.cos(theta + m)
            one_hot = F.one_hot(labels, num_classes=W.size(0)).float()
            logits = cosine * (1 - one_hot) + target_logits * one_hot
        else:
            logits = cosine
        logits = logits * self.scale
        return logits, z

# ============================================================
# TRACKERS & MEMORY BUFFER
# ============================================================
class DebugCentroidTracker:
    def __init__(self):
        self.reference = {}
        self.history = defaultdict(list)

    def compute_centroids(self, model, memory):
        model.eval()
        centroids = {}
        with torch.no_grad():
            for cls, Xs in memory.items():
                if len(Xs) == 0: continue
                X = torch.stack([x for x, _ in Xs]).to(DEVICE)
                _, Z = model(X)
                mu = F.normalize(Z.mean(0), dim=0)
                centroids[cls] = mu.detach().cpu()
        return centroids

    def snapshot(self, model, memory):
        self.reference = self.compute_centroids(model, memory)

    def measure_drift(self, model, memory, epoch, tag=""):
        current = self.compute_centroids(model, memory)
        for cls in self.reference:
            if cls in current:
                drift = 1 - torch.dot(self.reference[cls], current[cls])
                self.history[(tag, cls)].append(drift.item())

class MemoryBuffer:
    def __init__(self, max_per_class=400):
        self.data = defaultdict(list)
        self.max_per_class = max_per_class
        self.aug = T.Compose([
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip()
        ])

    @torch.no_grad()
    def build_memory_herding(self, X_all, y_label, model):
        model.eval()
        logits_all, Z_all = [], []
        for i in range(0, len(X_all), 128):
            batch = X_all[i:i+128].to(DEVICE)
            logits, z = model(batch)
            logits_all.append(logits.cpu())
            Z_all.append(z.cpu())
        logits_all = torch.cat(logits_all)
        Z_all = torch.cat(Z_all)
        class_mean = F.normalize(Z_all.mean(0), dim=0)
        selected_idx = []
        features = Z_all.clone()

        for k in range(min(self.max_per_class, len(X_all))):
            if k > 0:
                S = Z_all[selected_idx].sum(0)
            else:
                S = torch.zeros_like(class_mean)
            target = (k + 1) * class_mean - S
            distances = torch.norm(features - target, dim=1)
            for idx in selected_idx:
                distances[idx] = float('inf')
            best = distances.argmin().item()
            selected_idx.append(best)

        self.data[int(y_label)] = []
        for idx in selected_idx:
            self.data[int(y_label)].append((X_all[idx].detach().cpu(), logits_all[idx].detach().cpu()))
    
    def get(self): return self.data

    def sample_balanced(self, batch_size, model):
        classes = list(self.data.keys())
        if not classes: return None, None, None
        samples_per_class = max(1, batch_size // len(classes))
        X_mem, Y_mem, L_mem = [], [], []
        current_dim = model.classifier.out_features
        for cls in classes:
            samples = self.data[cls]
            if len(samples) == 0: continue
            replace = len(samples) < samples_per_class
            idx = np.random.choice(len(samples), samples_per_class, replace=replace)
            for i in idx:
                x, logit = samples[i]
                if logit.shape[0] < current_dim:
                    padded = torch.zeros(current_dim)
                    padded[:logit.shape[0]] = logit
                    logit = padded
                X_mem.append(x)
                Y_mem.append(cls)
                L_mem.append(logit)
        if not X_mem: return None, None, None
        X_tensor = torch.stack(X_mem)
        X_tensor = self.aug(X_tensor) 
        return X_tensor, torch.tensor(Y_mem), torch.stack(L_mem)

class HypersphereNovelty:
    def __init__(self, q=0.90):
        self.q = q
        self.mu, self.r = {}, {}

    def update(self, memory, model):
        self.mu, self.r = {}, {}
        for k, X_tuples in memory.items():
            if len(X_tuples) == 0: continue
            X = torch.stack([x for x, _ in X_tuples]).to(DEVICE)
            with torch.no_grad(): 
                _, Z = model(X)
            mu = F.normalize(Z.mean(0), dim=0)
            d = 1 - torch.matmul(Z, mu)
            self.mu[k] = mu
            self.r[k] = torch.quantile(d, self.q)

    def score(self, z):
        if not self.mu: return torch.tensor(0.0)
        return min([(1 - torch.dot(z, self.mu[k].to(z.device))) - self.r[k].to(z.device) for k in self.mu])

# ============================================================
# TRAINING LOOPS
# ============================================================
def train_supervised(model, loader, test_loader, task_id, method="baseline_ce"):
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    
    for epoch in range(30):
        for x, y in loader:
            x_device, y_device = x.to(DEVICE), y.argmax(1).to(DEVICE)
            x_aug = aug(x_device)
            logits, z = model(x_aug, y_device)
            loss_ce = F.cross_entropy(logits, y_device)
            z_norm = F.normalize(z, dim=1)
            
            if method == "baseline_ce": loss = loss_ce
            elif method == "soft_center_loss": loss = loss_ce + 1.0 * soft_center_loss(z_norm, y_device, margin=0.15)
            elif method == "margin_contrastive": loss = loss_ce + 1.0 * margin_contrastive_loss(z_norm, y_device)
            elif method == "triplet": loss = loss_ce + 1.0 * triplet_loss_batch(z_norm, y_device, margin=1.0)
                
            opt.zero_grad()
            loss.backward()
            opt.step()

def finetune(model, memory, X_new, new_label, task_id, test_loader, locked_gen): 
    old_model = copy.deepcopy(model).eval()
    for p in old_model.parameters(): p.requires_grad = False 
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()
            
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    Y_new = torch.full((len(X_new),), new_label, dtype=torch.long)
    
    loader = DataLoader(TensorDataset(X_new, Y_new), batch_size=32, shuffle=True, generator=locked_gen)

    for p in model.parameters(): p.requires_grad = True
    opt = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

    for epoch in range(25):
        for xb, yb in loader:
            xb = aug(xb) 
            X_mem, Y_mem, L_mem = memory.sample_balanced(32, model)
            if X_mem is not None:
                xb = torch.cat([xb, X_mem], dim=0)
                yb = torch.cat([yb, Y_mem], dim=0)

            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits_margin, Z = model(xb, yb)
            loss_ce = F.cross_entropy(logits_margin, yb)
            
            if X_mem is not None:
                pure_logits, _ = model(xb) 
                logits_mem = pure_logits[-len(X_mem):]
                loss_der = F.mse_loss(logits_mem, L_mem.to(DEVICE))
            else:
                loss_der = torch.tensor(0.0, device=DEVICE)

            with torch.no_grad(): 
                logits_old, Z_old = old_model(xb)

            loss_feat = (1 - F.cosine_similarity(Z, Z_old)).mean()
            loss = loss_ce + 0.5 * loss_der + 1.0 * loss_feat
            
            opt.zero_grad()
            loss.backward()
            opt.step()
    model.eval()

def evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies):
    correct_per_class = defaultdict(int)
    total_per_class = defaultdict(int)
    model.eval()
    with torch.no_grad():
        for x_test, y_test in DataLoader(test_ds, batch_size=128):
            x_test = x_test.to(DEVICE)
            y_test = y_test.argmax(1).to(DEVICE)
            logits, _ = model(x_test)
            preds = logits.argmax(1)
            
            for i in range(len(y_test)):
                sem = int(y_test[i])
                if sem in semantic_to_internal:
                    internal_gt = semantic_to_internal[sem]
                    total_per_class[sem] += 1
                    if preds[i].item() == internal_gt:
                        correct_per_class[sem] += 1

    current_class_accs = {}
    for sem in semantic_to_internal.keys():
        current_class_accs[sem] = correct_per_class[sem] / max(total_per_class[sem], 1)
        
    class_accuracies[t] = current_class_accs
    total_correct = sum(correct_per_class.values())
    total_eval = sum(total_per_class.values())
    acc = total_correct / max(total_eval, 1)
    
    if t > 0:
        forgetting_list = []
        for sem in current_class_accs.keys():
            past_accs = [class_accuracies[k].get(sem, None) for k in range(t)]
            past_accs = [a for a in past_accs if a is not None]
            if past_accs:
                max_past = max(past_accs)
                forgetting = max_past - current_class_accs[sem]
                forgetting_list.append(forgetting)
        avg_forg = np.mean(forgetting_list) if forgetting_list else 0.0
    else:
        avg_forg = 0.0
        
    return acc, avg_forg, current_class_accs

# ============================================================
# GRAND SINGLE RUN: OPTIMIZED STRICT + MARGIN CONTRASTIVE
# ============================================================

print("\n" + "="*80)
print(f"🚀 RUNNING OPTIMIZED PIPELINE: [Strict] | METHOD [{METHOD.upper()}]")
print("="*80)

set_seed(42)
locked_generator = get_locked_generator(42)

BASE_DIR = f"debug_OptimizedStrict_{METHOD}"
os.makedirs(BASE_DIR, exist_ok=True)

task_kca, task_cpr = [], []
average_forgetting = [] 
class_accuracies = {}   
learned_classes_over_time = []
promoted_classes_log = []

TASKS = [f"splitcifar10_{i}" for i in range(NUM_TASKS)]

model = CNN(num_classes=2).to(DEVICE)

# 🚀 IMPROVEMENT: Increased Replay Memory max_per_class from 200 to 400
memory = MemoryBuffer(max_per_class=400) 
detector = HypersphereNovelty()
novelty_buffer = []

semantic_to_internal = {0: 0, 1: 1}
internal_to_semantic = {0: 0, 1: 1}
known_classes = 2

for t, task in enumerate(TASKS):
    _, dataset_fn = DATASET_REGISTRY[task]
    train_ds, test_ds, *_ = dataset_fn("./data")
    loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=locked_generator) 

    print(f"\n{'='*20} 🚀 TASK {t} {'='*20}")
    all_semantics = []
    for _, y in loader: all_semantics.extend(y.argmax(1).tolist())
    unique_semantics = list(set(all_semantics))
    class_names = [CIFAR10_LABELS[sem] for sem in unique_semantics]
    print(f"📦 [STREAM] Task {t} stream contains class(es): {class_names} (Label IDs: {unique_semantics})")
    print(f"📊 [STREAM] Total images in this chunk: {len(train_ds)}")

    if t == 0:
        print(f"🎓 [INIT] Running '{METHOD}' initialization on Task 0...")
        train_supervised(model, loader, DataLoader(test_ds, batch_size=128, shuffle=False), t, method=METHOD)
        for cls in [0, 1]:
            Xc = torch.cat([x[y.argmax(1) == cls] for x, y in loader])
            memory.build_memory_herding(Xc, cls, model)
        detector.update(memory.get(), model)
        learned_classes_over_time.append({CIFAR10_LABELS[0], CIFAR10_LABELS[1]})
        print(f"✅ [INIT COMPLETE] Known classes: {learned_classes_over_time[-1]}")
        
        acc, avg_forg, current_class_accs = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
        task_kca.append(acc)
        average_forgetting.append(avg_forg)
        print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")
        for sem, c_acc in current_class_accs.items():
            print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")
        continue

    model.eval()
    novelty_candidates = []
    novel, false_novel, total = 0, 0, 0
    
    with torch.no_grad():
        for x, y in loader:
            _, z = model(x.to(DEVICE))
            y_labels = y.argmax(1)
            scores = [detector.score(z[i]).item() for i in range(len(z))]
            thr = np.percentile(scores, 30)
            for i in range(len(z)):
                total += 1
                if scores[i] > thr:
                    novelty_candidates.append((scores[i], x[i].cpu(), y_labels[i].item()))
                    novel += 1
                    if y_labels[i].item() in semantic_to_internal:
                        false_novel += 1

    print(f"🔍 [NOVELTY] Flagged Novel: {novel}/{total} | False Novelty (Knowns flagged): {false_novel}")

    novelty_candidates.sort(reverse=True, key=lambda x: x[0])
    novelty_buffer.extend([(img, y) for _, img, y in novelty_candidates[:TOPK_NOVELTY]])
    novelty_buffer = novelty_buffer[-MAX_NOVELTY_BUFFER:]

    if t % P == 0 and len(novelty_buffer) >= 20: 
        print(f"\n🧠 [CLUSTERING] Running HDBSCAN on novelty buffer (Size: {len(novelty_buffer)})...")
        
        Z = []
        with torch.no_grad():
            for img, _ in novelty_buffer:
                _, z = model(img.unsqueeze(0).to(DEVICE))
                Z.append(z.squeeze().cpu().numpy())

        Z = np.stack(Z)
        labels = hdbscan.HDBSCAN(metric='euclidean', min_cluster_size=20, cluster_selection_epsilon=EPSILON).fit_predict(Z)
        
        new_buffer = []
        found, promoted = 0, 0

        for cid in sorted(set(labels)):
            idxs = np.where(labels == cid)[0]
            if cid == -1:
                print(f"  🗑️  [CLUSTER -1] NOISE: Found {len(idxs)} noise samples.")
                for i in idxs: new_buffer.append(novelty_buffer[i])
                continue

            found += 1
            Xc = torch.stack([novelty_buffer[i][0] for i in idxs])
            with torch.no_grad(): _, Zc = model(Xc.to(DEVICE))
            
            mu = F.normalize(Zc.mean(0), dim=0)
            n = len(idxs)
            S_intra = torch.mean(1 - torch.matmul(Zc, mu))
            
            if len(detector.mu) > 0:
                S_known = min([1 - torch.dot(mu, detector.mu[k].to(mu.device)) for k in detector.mu])
            else:
                S_known = torch.tensor(1.0)

            density = n / (S_intra.item() + 1e-6)
            margin = S_known - S_intra
            
            labels_true = [novelty_buffer[i][1] for i in idxs]
            sem_label, cnt = Counter(labels_true).most_common(1)[0]
            purity = cnt / len(labels_true) 
            
            if sem_label in semantic_to_internal:
                print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Dominant class '{CIFAR10_LABELS[sem_label]}' is already known.")
                continue 

            cond_intra = S_intra.item() <= ALPHA
            cond_density = density >= DELTA
            cond_known = S_known.item() >= BETA
            cond_margin = margin.item() > -0.10
            cond_size = n >= MIN_CLUSTER_SIZE

            print(f"\n  📊 [CLUSTER {cid} EVALUATION] Dominant: '{CIFAR10_LABELS[sem_label]}' (Purity: {purity:.2f})")
            print(f"     ➔ Size:    {n:3d}   (Req: >= {MIN_CLUSTER_SIZE:2d})   {'✅' if cond_size else '❌'}")
            print(f"     ➔ S_intra: {S_intra.item():.3f} (Req: <= {ALPHA:.2f}) {'✅' if cond_intra else '❌'}")
            print(f"     ➔ S_known: {S_known.item():.3f} (Req: >= {BETA:.2f}) {'✅' if cond_known else '❌'}")
            print(f"     ➔ Density: {density:.1f} (Req: >= {DELTA})   {'✅' if cond_density else '❌'}")
            print(f"     ➔ Margin:  {margin.item():.3f} (Req: > -0.10) {'✅' if cond_margin else '❌'}")

            if cond_intra and cond_density and cond_known and cond_margin and cond_size:
                promoted += 1
                new_label = known_classes
                semantic_to_internal[sem_label] = new_label
                internal_to_semantic[new_label] = sem_label
                known_classes += 1
                
                torch.save(Xc.cpu(), f"{BASE_DIR}/promoted_class_{sem_label}.pt")
                
                model.expand_head(known_classes)
                model.to(DEVICE)
                
                print(f"     🎉 [PROMOTION SUCCESS] -> Preparing to Finetune '{CIFAR10_LABELS[sem_label]}'...")
                finetune(model, memory, Xc, new_label, t, DataLoader(test_ds, batch_size=128, shuffle=False), locked_generator)
                print(f"     ✨ [LEARNED] -> Network successfully adapted to '{CIFAR10_LABELS[sem_label]}'")
                
                memory.build_memory_herding(Xc, new_label, model)
                detector.update(memory.get(), model)
                
                promoted_classes_log.append({"task": t, "semantic": CIFAR10_LABELS[sem_label]})
            else:
                print(f"     🛑 [PROMOTION FAILED] -> Conditions not met. Retaining samples in buffer.")
                for i in idxs: new_buffer.append(novelty_buffer[i])
                
        novelty_buffer = new_buffer
        task_cpr.append(promoted / max(found, 1))
    else:
        task_cpr.append(0.0)

    # FLUSH BUFFER AFTER EVERY 13 TASKS (END OF WAVE)
    if t > 0 and t % 13 == 0:
        novelty_buffer = [] 
        print(f"\n🧹 [BUFFER CLEAR] Task {t} marks the end of a multi-class wave! Flushed the novelty buffer for the next wave.")

    learned_classes_over_time.append(set(learned_classes_over_time[-1]) if len(learned_classes_over_time) > 0 else set())
    for p in promoted_classes_log:
        if p["task"] == t and p["semantic"] not in learned_classes_over_time[-1]:
            learned_classes_over_time[-1].add(p["semantic"])

    acc, avg_forg, current_class_accs = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
    task_kca.append(acc)
    average_forgetting.append(avg_forg)
    
    print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")
    for sem, c_acc in current_class_accs.items():
        print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")

# Store final results
final_accs = class_accuracies[NUM_TASKS - 1]
row_result = {
    "HPT": "Strict (Optimized)",
    "Method": METHOD,
    "Final_KCA": f"{task_kca[-1]:.3f}",
    "Avg_Forg": f"{average_forgetting[-1]:.3f}",
    "Classes_Learned": str(len(learned_classes_over_time[-1]))
}
for cls_id in range(10):
    row_result[f"Cls{cls_id}_{CIFAR10_LABELS[cls_id][:3]}"] = f"{final_accs.get(cls_id, 0.0):.2f}"

final_results_summary = [row_result]

# ============================================================
# CREATE & SAVE THE DESCRIPTIVE RESULT TABLE (PNG)
# ============================================================
print("\n" + "="*120)
print("📊 FINAL OPTIMIZED RUN SUMMARY (Multi-Class Wave Stream)")
print("="*120)

columns = ["HPT", "Method", "Final KCA", "Avg Forg", "Classes Learned"] + [f"Cls{i}" for i in range(10)]
cell_text = []

# Print to console
header_format = "{:<18} | {:<18} | {:<9} | {:<8} | {:<15} | " + " | ".join([f"{{:<4}}"]*10)
print(header_format.format(*columns))
print("-" * 140)

for res in final_results_summary:
    row = [res["HPT"], res["Method"], res["Final_KCA"], res["Avg_Forg"], res["Classes_Learned"]]
    row += [res[k] for k in list(res.keys())[5:]]
    cell_text.append(row)
    print(header_format.format(*row))

# Generate PNG Image of the Table
fig, ax = plt.subplots(figsize=(24, len(cell_text) * 0.5 + 2))
ax.axis('off')
ax.axis('tight')

table = ax.table(cellText=cell_text, colLabels=columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.8)

for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#4c72b0')
    else:
        if i % 2 == 0:
            cell.set_facecolor('#f2f2f2')

plt.title("Optimized Run Summary (Strict + Margin Contrastive)", fontsize=16, fontweight='bold', pad=20)
plt.savefig("final_optimized_summary_table.png", bbox_inches='tight', dpi=300)
plt.close()

print("\n✅ Script complete. Descriptive table saved to 'final_optimized_margincontrastive_stricthpt_summary_table.png'")


🚀 RUNNING OPTIMIZED PIPELINE: [Strict] | METHOD [MARGIN_CONTRASTIVE]

==================== 🚀 TASK 0 ====================
📦 [STREAM] Task 0 stream contains class(es): ['airplane', 'automobile'] (Label IDs: [0, 1])
📊 [STREAM] Total images in this chunk: 10000
🎓 [INIT] Running 'margin_contrastive' initialization on Task 0...
✅ [INIT COMPLETE] Known classes: {'automobile', 'airplane'}
📈 [METRIC] Known-Class Acc: 0.991 | Avg Forgetting: 0.000
   ➔ Class 'airplane' Acc: 0.984
   ➔ Class 'automobile' Acc: 0.997

==================== 🚀 TASK 1 ====================
📦 [STREAM] Task 1 stream contains class(es): ['airplane', 'automobile', 'bird', 'cat'] (Label IDs: [0, 1, 2, 3])
📊 [STREAM] Total images in this chunk: 970
🔍 [NOVELTY] Flagged Novel: 682/970 | False Novelty (Knowns flagged): 88

🧠 [CLUSTERING] Running HDBSCAN on novelty buffer (Size: 400)...
  🗑️  [CLUSTER -1] NOISE: Found 56 noise samples.

  📊 [CLUSTER 0 EVALUATION] Dominant: 'cat' (Purity: 0.56)
     ➔ Size:    261   (Req: >= 150)

### grid search on experiment A.1 for purity in open world (2-3 classes in 13 task waves) -> Phase 1

In [ ]:
import os
# 🚨 THESE MUST BE SET BEFORE IMPORTING TORCH
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from torch.utils.data import DataLoader, TensorDataset, Dataset
import hdbscan
import copy
from torchvision import datasets as tv_datasets, transforms
import torchvision.transforms as T  

# LabelBench Imports
from LabelBench.skeleton.dataset_skeleton import datasets as DATASET_REGISTRY
from LabelBench.skeleton.dataset_skeleton import register_dataset, LabelType, TransformDataset
from torchvision.models import resnet18

# ============================================================
# BULLETPROOF REPRODUCIBILITY LOCK
# ============================================================
def set_seed(seed=42):
    """Forces strict deterministic behavior across different machines/Colab accounts."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=False)

def get_locked_generator(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# ============================================================
# STABLE METRIC LEARNING LOSSES
# ============================================================
def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

# ============================================================
# DIRECTORIES & CONFIGURATION 
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
global BASE_DIR
BASE_DIR = f"debug_OptimizedStrict_margin_contrastive"
os.makedirs(BASE_DIR, exist_ok=True)

# BUFFER CONFIG
P = 1                              
TOPK_NOVELTY = 400              
MAX_NOVELTY_BUFFER = 2500        
STARTING_TTL = 3

# HPT CONFIG
ALPHA = 0.35              
BETA = 0.01                
DELTA = 15                 
EPSILON = 0.00
METHOD = "margin_contrastive"

CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat",
    4: "deer", 5: "dog", 6: "frog", 7: "horse",
    8: "ship", 9: "truck"
}

# ============================================================
# EXPERIMENT TRACK A: TRUE ABLATION (STRICTLY EARLIER BASELINE)
# ============================================================
EXPERIMENTS = {
    "Baseline (Earlier Code)": {
        "type": "baseline", 
        "min_cluster_size": 150, 
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    },
    "Centroid Distance (Top 50%)": {
        "type": "centroid", "ratio": 0.50,
        "min_cluster_size": 75,  # Lowered so the sliced core can pass the size check
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    },
    "Known Repulsion (< 0.50)": {
        "type": "repulsion", "thresh": 0.50,
        "min_cluster_size": 75, 
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    },
    "HDBSCAN Prob (>= 0.80)": {
        "type": "hdbscan_prob", "thresh": 0.80,
        "min_cluster_size": 75, 
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    },
    "Entropy / MSP (< 0.85)": {
        "type": "msp", "thresh": 0.85,
        "min_cluster_size": 75, 
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    },
    "k-NN Memory Overlap (k=5, max=2)": {
        "type": "knn", "k": 5, "overlap": 3,
        "min_cluster_size": 75, 
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    }
}

# ============================================================
# DYNAMIC 40-TASK DATASET DEFINITION (MULTI-CLASS WAVE LOGIC)
# ============================================================
NUM_TASKS = 40   

class CIFARStream(Dataset):
    def __init__(self, base_ds, indices):
        self.base_ds = base_ds      
        self.indices = indices      

    def __len__(self): return len(self.indices)
    def __getitem__(self, idx): return self.base_ds[self.indices[idx]]

def one_hot(y, n=10): return F.one_hot(torch.tensor(y), num_classes=n).float()

@register_dataset("splitcifar10", LabelType.MULTI_CLASS)
def get_splitcifar10(_): raise RuntimeError("Use splitcifar10_<id>")

base_train_global = tv_datasets.CIFAR10(root="./data", train=True, download=True)
targets = np.array(base_train_global.targets)
class_indices = {c: np.where(targets == c)[0] for c in range(10)}

rng = np.random.default_rng(42)
for c in range(10): rng.shuffle(class_indices[c])

stream_splits = {}
for t in range(1, NUM_TASKS):
    stream_splits[t] = []
    if 1 <= t <= 13: reminders = [0, 1]
    elif 14 <= t <= 26: reminders = [0, 1, 2, 3]
    elif 27 <= t <= 39: reminders = [0, 1, 2, 3, 4, 5, 6]
    else: reminders = []
        
    for c in reminders: stream_splits[t].extend(rng.choice(class_indices[c], 100, replace=False))
        
    if 1 <= t <= 13:
        stream_splits[t].extend(np.array_split(class_indices[2], 13)[t-1])
        stream_splits[t].extend(np.array_split(class_indices[3], 13)[t-1])
    elif 14 <= t <= 26:
        stream_splits[t].extend(np.array_split(class_indices[4], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[5], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[6], 13)[t-14])
    elif 27 <= t <= 39:
        stream_splits[t].extend(np.array_split(class_indices[7], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[8], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[9], 13)[t-27])
        
    rng.shuffle(stream_splits[t])

for split_id in range(NUM_TASKS):
    @register_dataset(f"splitcifar10_{split_id}", LabelType.MULTI_CLASS)
    def _make_split(data_dir, split_id=split_id):
        tf = transforms.Compose([transforms.ToTensor()])
        base_train = tv_datasets.CIFAR10(root=data_dir, train=True, download=True)
        base_test  = tv_datasets.CIFAR10(root=data_dir, train=False, download=True)
        
        if split_id == 0: indices = [i for i,(x,y) in enumerate(base_train) if y in [0,1]]
        else: indices = stream_splits[split_id]

        train_ds = TransformDataset(CIFARStream(base_train, indices), transform=tf, target_transform=lambda y: one_hot(y,10))
        test_ds = TransformDataset(base_test, transform=tf, target_transform=lambda y: one_hot(y,10))
        return train_ds, test_ds, test_ds, None, None, None, 10, [str(i) for i in range(10)]

# ============================================================
# MODEL & TRACKERS
# ============================================================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = resnet18(weights='DEFAULT')
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes, bias=False)
        self.scale = 20.0  

    def expand_head(self, new_classes, new_centroid=None):
        old_w = self.classifier.weight.data.clone()
        old_n = old_w.shape[0]
        new_classifier = nn.Linear(512, new_classes, bias=False).to(old_w.device)
        new_classifier.weight.data[:old_n] = old_w
        if new_centroid is not None:
            new_classifier.weight.data[old_n:] = new_centroid.to(old_w.device)
        self.classifier = new_classifier

    def forward(self, x, labels=None):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = self.embed(z)
        z = F.normalize(z, dim=1)
        W = F.normalize(self.classifier.weight, dim=1)
        cosine = torch.matmul(z, W.t())
        if labels is not None: 
            m = 0.2 
            theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
            target_logits = torch.cos(theta + m)
            one_hot = F.one_hot(labels, num_classes=W.size(0)).float()
            logits = cosine * (1 - one_hot) + target_logits * one_hot
        else:
            logits = cosine
        logits = logits * self.scale
        return logits, z

class MemoryBuffer:
    def __init__(self, max_per_class=400):
        self.data = defaultdict(list)
        self.max_per_class = max_per_class
        self.aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])

    @torch.no_grad()
    def build_memory_herding(self, X_all, y_label, model):
        model.eval()
        logits_all, Z_all = [], []
        for i in range(0, len(X_all), 128):
            batch = X_all[i:i+128].to(DEVICE)
            logits, z = model(batch)
            logits_all.append(logits.cpu())
            Z_all.append(z.cpu())
        logits_all = torch.cat(logits_all)
        Z_all = torch.cat(Z_all)
        class_mean = F.normalize(Z_all.mean(0), dim=0)
        selected_idx = []
        features = Z_all.clone()

        for k in range(min(self.max_per_class, len(X_all))):
            S = Z_all[selected_idx].sum(0) if k > 0 else torch.zeros_like(class_mean)
            target = (k + 1) * class_mean - S
            distances = torch.norm(features - target, dim=1)
            for idx in selected_idx: distances[idx] = float('inf')
            selected_idx.append(distances.argmin().item())

        self.data[int(y_label)] = []
        for idx in selected_idx:
            self.data[int(y_label)].append((X_all[idx].detach().cpu(), logits_all[idx].detach().cpu()))
            
    def get_all_features_and_labels(self, model):
        X_all, Y_all = [], []
        for c, data in self.data.items():
            for x, _ in data:
                X_all.append(x)
                Y_all.append(c)
        if not X_all: return None, None
        X_all = torch.stack(X_all).to(DEVICE)
        with torch.no_grad(): _, Z_all = model(X_all)
        return F.normalize(Z_all, dim=1), torch.tensor(Y_all).to(DEVICE)
    
    def get(self): return self.data

    def sample_balanced(self, batch_size, model):
        classes = list(self.data.keys())
        if not classes: return None, None, None
        samples_per_class = max(1, batch_size // len(classes))
        X_mem, Y_mem, L_mem = [], [], []
        current_dim = model.classifier.out_features
        for cls in classes:
            samples = self.data[cls]
            if len(samples) == 0: continue
            replace = len(samples) < samples_per_class
            idx = np.random.choice(len(samples), samples_per_class, replace=replace)
            for i in idx:
                x, logit = samples[i]
                if logit.shape[0] < current_dim:
                    padded = torch.zeros(current_dim)
                    padded[:logit.shape[0]] = logit
                    logit = padded
                X_mem.append(x)
                Y_mem.append(cls)
                L_mem.append(logit)
        if not X_mem: return None, None, None
        X_tensor = torch.stack(X_mem)
        return self.aug(X_tensor), torch.tensor(Y_mem), torch.stack(L_mem)

class HypersphereNovelty:
    def __init__(self, q=0.90):
        self.q = q
        self.mu, self.r = {}, {}

    def update(self, memory, model):
        self.mu, self.r = {}, {}
        for k, X_tuples in memory.items():
            if len(X_tuples) == 0: continue
            X = torch.stack([x for x, _ in X_tuples]).to(DEVICE)
            with torch.no_grad(): _, Z = model(X)
            mu = F.normalize(Z.mean(0), dim=0)
            d = 1 - torch.matmul(Z, mu)
            self.mu[k] = mu
            self.r[k] = torch.quantile(d, self.q)

    def score(self, z):
        if not self.mu: return torch.tensor(0.0)
        return min([(1 - torch.dot(z, self.mu[k].to(z.device))) - self.r[k].to(z.device) for k in self.mu])

# ============================================================
# TRAINING LOOPS
# ============================================================
def train_supervised(model, loader, test_loader, task_id):
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    
    for epoch in range(30):
        for x, y in loader:
            x_device, y_device = x.to(DEVICE), y.argmax(1).to(DEVICE)
            x_aug = aug(x_device)
            logits, z = model(x_aug, y_device)
            z_norm = F.normalize(z, dim=1)
            loss = F.cross_entropy(logits, y_device) + 1.0 * margin_contrastive_loss(z_norm, y_device)
            opt.zero_grad()
            loss.backward()
            opt.step()

def finetune(model, memory, X_new, new_label, task_id, locked_gen, exp_config): 
    old_model = copy.deepcopy(model).eval()
    for p in old_model.parameters(): p.requires_grad = False 
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()
            
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    Y_new = torch.full((len(X_new),), new_label, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_new, Y_new), batch_size=32, shuffle=True, generator=locked_gen)

    for p in model.parameters(): p.requires_grad = True
    opt = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

    # 🚀 Dynamically use experiment-specific hyperparameters
    epochs = exp_config.get("epochs", 25)
    feat_weight = exp_config.get("feat_weight", 1.0)

    for epoch in range(epochs):
        for xb, yb in loader:
            xb = aug(xb) 
            X_mem, Y_mem, L_mem = memory.sample_balanced(32, model)
            if X_mem is not None:
                xb = torch.cat([xb, X_mem], dim=0)
                yb = torch.cat([yb, Y_mem], dim=0)

            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits_margin, Z = model(xb, yb)
            loss_ce = F.cross_entropy(logits_margin, yb)
            
            if X_mem is not None:
                pure_logits, _ = model(xb) 
                logits_mem = pure_logits[-len(X_mem):]
                loss_der = F.mse_loss(logits_mem, L_mem.to(DEVICE))
            else:
                loss_der = torch.tensor(0.0, device=DEVICE)

            with torch.no_grad(): logits_old, Z_old = old_model(xb)
            loss_feat = (1 - F.cosine_similarity(Z, Z_old)).mean()
            
            loss = loss_ce + 0.5 * loss_der + feat_weight * loss_feat
            opt.zero_grad()
            loss.backward()
            opt.step()
    model.eval()

def evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies):
    correct_per_class = defaultdict(int)
    total_per_class = defaultdict(int)
    model.eval()
    with torch.no_grad():
        for x_test, y_test in DataLoader(test_ds, batch_size=128):
            x_test = x_test.to(DEVICE)
            y_test = y_test.argmax(1).to(DEVICE)
            logits, _ = model(x_test)
            preds = logits.argmax(1)
            for i in range(len(y_test)):
                sem = int(y_test[i])
                if sem in semantic_to_internal:
                    internal_gt = semantic_to_internal[sem]
                    total_per_class[sem] += 1
                    if preds[i].item() == internal_gt:
                        correct_per_class[sem] += 1

    current_class_accs = {}
    for sem in semantic_to_internal.keys():
        current_class_accs[sem] = correct_per_class[sem] / max(total_per_class[sem], 1)
        
    class_accuracies[t] = current_class_accs
    total_correct = sum(correct_per_class.values())
    total_eval = sum(total_per_class.values())
    acc = total_correct / max(total_eval, 1)
    
    if t > 0:
        forgetting_list = []
        for sem in current_class_accs.keys():
            past_accs = [class_accuracies[k].get(sem, None) for k in range(t)]
            past_accs = [a for a in past_accs if a is not None]
            if past_accs:
                max_past = max(past_accs)
                forgetting = max_past - current_class_accs[sem]
                forgetting_list.append(forgetting)
        avg_forg = np.mean(forgetting_list) if forgetting_list else 0.0
    else:
        avg_forg = 0.0
    return acc, avg_forg, current_class_accs


# ============================================================
# MASTER EXPERIMENT LOOP
# ============================================================
print("\n" + "="*100)
print(f"🚀 INITIATING MASTER ABLATION STUDY: TRACK A (CLUSTER PURIFICATION)")
print("="*100)

final_results_summary = []
TASKS = [f"splitcifar10_{i}" for i in range(NUM_TASKS)]

for exp_name, exp_config in EXPERIMENTS.items():
    print("\n\n" + "#"*100)
    print(f"⚙️  RUNNING EXPERIMENT: {exp_name}")
    print("#"*100)

    # 🚨 HARD RESET: Ensure Identical Starting Conditions
    set_seed(42)
    locked_generator = get_locked_generator(42)
    model = CNN(num_classes=2).to(DEVICE)
    memory = MemoryBuffer(max_per_class=400) 
    detector = HypersphereNovelty()
    
    novelty_buffer = []
    semantic_to_internal = {0: 0, 1: 1}
    internal_to_semantic = {0: 0, 1: 1}
    known_classes = 2
    
    task_kca, task_cpr, average_forgetting = [], [], []
    class_accuracies = {}   
    learned_classes_over_time = []
    promoted_classes_log = []
    
    # Trackers for Purity evaluation over the experiment
    exp_init_purities = []
    exp_core_purities = []

    for t, task in enumerate(TASKS):
        _, dataset_fn = DATASET_REGISTRY[task]
        train_ds, test_ds, *_ = dataset_fn("./data")
        loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=locked_generator) 

        print(f"\n{'-'*20} 🚀 TASK {t} {'-'*20}")
        
        # 🚀 RESTORED STREAM DEBUGGING
        all_semantics = []
        for _, y in loader: all_semantics.extend(y.argmax(1).tolist())
        unique_semantics = list(set(all_semantics))
        class_names = [CIFAR10_LABELS[sem] for sem in unique_semantics]
        print(f"📦 [STREAM] Task {t} stream contains class(es): {class_names} (Label IDs: {unique_semantics})")
        print(f"📊 [STREAM] Total images in this chunk: {len(train_ds)}")

        if t == 0:
            train_supervised(model, loader, DataLoader(test_ds, batch_size=128, shuffle=False), t)
            for cls in [0, 1]:
                Xc = torch.cat([x[y.argmax(1) == cls] for x, y in loader])
                memory.build_memory_herding(Xc, cls, model)
            detector.update(memory.get(), model)
            learned_classes_over_time.append({CIFAR10_LABELS[0], CIFAR10_LABELS[1]})
            acc, avg_forg, class_accuracies[t] = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
            task_kca.append(acc); average_forgetting.append(avg_forg)
            print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")
            for sem, c_acc in class_accuracies[t].items():
                print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")
            continue

        model.eval()
        novelty_candidates = []
        novel, false_novel, total = 0, 0, 0
        with torch.no_grad():
            for x, y in loader:
                _, z = model(x.to(DEVICE))
                y_labels = y.argmax(1)
                scores = [detector.score(z[i]).item() for i in range(len(z))]
                thr = np.percentile(scores, 30)
                for i in range(len(z)):
                    total += 1
                    if scores[i] > thr: 
                        novelty_candidates.append((scores[i], x[i].cpu(), y_labels[i].item()))
                        novel += 1
                        if y_labels[i].item() in semantic_to_internal: false_novel += 1

        print(f"🔍 [NOVELTY] Flagged Novel: {novel}/{total} | False Novelty: {false_novel}")

        novelty_candidates.sort(reverse=True, key=lambda x: x[0])
        novelty_buffer.extend([(img, y, STARTING_TTL) for _, img, y in novelty_candidates[:TOPK_NOVELTY]])
        novelty_buffer = novelty_buffer[-MAX_NOVELTY_BUFFER:]

        if t % P == 0 and len(novelty_buffer) >= 20: 
            print(f"\n🧠 [CLUSTERING] Running HDBSCAN on buffer (Size: {len(novelty_buffer)})...")
            Z = []
            with torch.no_grad():
                for img, _, _ in novelty_buffer:
                    _, z = model(img.unsqueeze(0).to(DEVICE))
                    Z.append(z.squeeze().cpu().numpy())

            Z = np.stack(Z)
            clusterer = hdbscan.HDBSCAN(metric='euclidean', min_cluster_size=20, cluster_selection_epsilon=EPSILON)
            labels = clusterer.fit_predict(Z)
            persistences = clusterer.cluster_persistence_
            max_persistence = np.max(persistences) if len(persistences) > 0 else 0.0
            
            new_buffer = []
            found, promoted = 0, 0

            # Dynamic TTL Config
            use_ttl = exp_config.get("use_ttl", False)

            for cid in sorted(set(labels)):
                idxs_full = np.where(labels == cid)[0].tolist()
                
                if cid == -1: 
                    print(f"  🗑️  [CLUSTER -1] NOISE: Found {len(idxs_full)} noise samples.")
                    for i in idxs_full: 
                        img, y, ttl = novelty_buffer[i]
                        if use_ttl:
                            if ttl - 1 > 0: new_buffer.append((img, y, ttl - 1))
                        else:
                            new_buffer.append((img, y, ttl))
                    continue

                found += 1
                Xc_full = torch.stack([novelty_buffer[i][0] for i in idxs_full])
                with torch.no_grad(): _, Zc_full = model(Xc_full.to(DEVICE))
                
                # =========================================================
                # 🧪 APPLY DYNAMIC PURIFICATION EXPERIMENT FILTER
                # =========================================================
                idxs = []
                if exp_config["type"] == "baseline":
                    idxs = idxs_full
                elif exp_config["type"] == "centroid":
                    mu_full = F.normalize(Zc_full.mean(0), dim=0)
                    sims_to_centroid = torch.matmul(Zc_full, mu_full)
                    keep_k = max(1, int(len(idxs_full) * exp_config["ratio"]))
                    _, topk_indices = torch.topk(sims_to_centroid, keep_k)
                    idxs = [idxs_full[idx.item()] for idx in topk_indices]
                elif exp_config["type"] == "repulsion":
                    if len(detector.mu) > 0:
                        known_mu = torch.stack(list(detector.mu.values())).to(DEVICE)
                        sims = torch.matmul(Zc_full, known_mu.T)
                        max_sims, _ = torch.max(sims, dim=1)
                        idxs = [idxs_full[i] for i in range(len(idxs_full)) if max_sims[i].item() < exp_config["thresh"]]
                    else:
                        idxs = idxs_full
                elif exp_config["type"] == "hdbscan_prob":
                    probs = clusterer.probabilities_[idxs_full]
                    idxs = [idxs_full[i] for i in range(len(idxs_full)) if probs[i] >= exp_config["thresh"]]
                elif exp_config["type"] == "msp":
                    with torch.no_grad():
                        logits, _ = model(Xc_full.to(DEVICE))
                        probs = F.softmax(logits, dim=1)
                        max_probs, _ = torch.max(probs, dim=1)
                    idxs = [idxs_full[i] for i in range(len(idxs_full)) if max_probs[i].item() < exp_config["thresh"]]
                elif exp_config["type"] == "knn":
                    Z_mem, Y_mem = memory.get_all_features_and_labels(model)
                    if Z_mem is not None:
                        sims = torch.matmul(Zc_full, Z_mem.T) 
                        _, topk_idx = torch.topk(sims, exp_config["k"], dim=1)
                        valid_indices = []
                        for i in range(len(idxs_full)):
                            neighbors = Y_mem[topk_idx[i]].tolist()
                            most_common_cnt = Counter(neighbors).most_common(1)[0][1]
                            if most_common_cnt < exp_config["overlap"]:
                                valid_indices.append(idxs_full[i])
                        idxs = valid_indices
                    else:
                        idxs = idxs_full

                labels_true_full = [novelty_buffer[i][1] for i in idxs_full]
                purity_full = Counter(labels_true_full).most_common(1)[0][1] / len(labels_true_full)
                
                # Apply buffer retention rules to items filtered out during purification
                filtered_out = [idx for idx in idxs_full if idx not in idxs]
                for i in filtered_out:
                    img, y, ttl = novelty_buffer[i]
                    if use_ttl:
                        if ttl - 1 > 0: new_buffer.append((img, y, ttl - 1))
                    else:
                        new_buffer.append((img, y, ttl))
                
                if len(idxs) == 0:
                    print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Completely eliminated by {exp_config['type']} filter.")
                    continue 
                    
                labels_true_core = [novelty_buffer[i][1] for i in idxs]
                sem_label, cnt_core = Counter(labels_true_core).most_common(1)[0]
                purity_core = cnt_core / len(labels_true_core)
                
                # =========================================================
                # 📊 RICH EVALUATION ON THE PURIFIED CORE
                # =========================================================
                Xc = torch.stack([novelty_buffer[i][0] for i in idxs])
                with torch.no_grad(): _, Zc = model(Xc.to(DEVICE))
                
                mu = F.normalize(Zc.mean(0), dim=0)
                n = len(idxs)
                S_intra = torch.mean(1 - torch.matmul(Zc, mu))
                S_known = min([1 - torch.dot(mu, detector.mu[k].to(mu.device)) for k in detector.mu]) if len(detector.mu)>0 else torch.tensor(1.0)
                density = n / (S_intra.item() + 1e-6)
                margin = S_known - S_intra
                
                if sem_label in semantic_to_internal:
                    print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Dominant class '{CIFAR10_LABELS[sem_label]}' already known.")
                    for i in idxs: 
                        img, y, ttl = novelty_buffer[i]
                        if use_ttl:
                            if ttl - 1 > 0: new_buffer.append((img, y, ttl - 1))
                        else:
                            new_buffer.append((img, y, ttl))
                    continue 

                req_min_size = exp_config.get("min_cluster_size", 150)
                
                if exp_config.get("use_persistence", False):
                    relative_persistence = persistences[cid] / (max_persistence + 1e-9)
                    is_highly_persistent = (relative_persistence >= 0.75)
                else:
                    relative_persistence = 0.0
                    is_highly_persistent = False

                cond_intra = S_intra.item() <= ALPHA
                cond_density = density >= DELTA
                cond_known = S_known.item() >= BETA
                cond_margin = margin.item() > -0.10
                cond_size = n >= req_min_size

                print(f"\n  📊 [CLUSTER {cid} EVALUATION] Dominant: '{CIFAR10_LABELS[sem_label]}' (Init Pur: {purity_full:.2f} -> Core Pur: {purity_core:.2f})")
                print(f"     ➔ Size:    {n:3d}   (Req: >= {req_min_size:2d})   {'✅' if cond_size else '❌'}")
                print(f"     ➔ S_intra: {S_intra.item():.3f} (Req: <= {ALPHA:.2f}) {'✅' if cond_intra else '❌'}")
                print(f"     ➔ S_known: {S_known.item():.3f} (Req: >= {BETA:.2f}) {'✅' if cond_known else '❌'}")
                print(f"     ➔ Density: {density:.1f} (Req: >= {DELTA})   {'✅' if cond_density else '❌'}")
                print(f"     ➔ Margin:  {margin.item():.3f} (Req: > -0.10) {'✅' if cond_margin else '❌'}")
                if exp_config.get("use_persistence", False):
                    print(f"     ➔ Rel Pers:{relative_persistence:.2f} (Req: >= 0.75)  {'✅' if is_highly_persistent else '❌'}")

                if cond_intra and cond_density and cond_size and cond_margin and (cond_known or is_highly_persistent):
                    if not cond_known and is_highly_persistent:
                        print(f"     🌟 [PERSISTENCE BYPASS] S_known failed, but highly stable cluster detected!")
                        
                    promoted += 1
                    new_label = known_classes
                    semantic_to_internal[sem_label] = new_label
                    internal_to_semantic[new_label] = sem_label
                    known_classes += 1
                    
                    exp_init_purities.append(purity_full)
                    exp_core_purities.append(purity_core)
                    
                    if exp_config.get("imprint", False):
                        model.expand_head(known_classes, new_centroid=mu)
                    else:
                        model.expand_head(known_classes)
                    model.to(DEVICE)
                    
                    print(f"     🎉 [PROMOTION SUCCESS] -> Preparing to Finetune '{CIFAR10_LABELS[sem_label]}'...")
                    finetune(model, memory, Xc, new_label, t, locked_generator, exp_config)
                    print(f"     ✨ [LEARNED] -> Network successfully adapted to '{CIFAR10_LABELS[sem_label]}'")
                    
                    memory.build_memory_herding(Xc, new_label, model)
                    detector.update(memory.get(), model)
                    promoted_classes_log.append({"task": t, "semantic": CIFAR10_LABELS[sem_label]})
                else:
                    print(f"     🛑 [PROMOTION FAILED] -> Conditions not met. Retaining samples in buffer.")
                    for i in idxs: 
                        img, y, ttl = novelty_buffer[i]
                        if use_ttl:
                            if ttl - 1 > 0: new_buffer.append((img, y, ttl - 1))
                        else:
                            new_buffer.append((img, y, ttl))
                    
            novelty_buffer = new_buffer
            task_cpr.append(promoted / max(found, 1))
        else:
            task_cpr.append(0.0)
            
        if t > 0 and t % 13 == 0: 
            novelty_buffer = [] 
            print(f"\n🧹 [BUFFER CLEAR] Task {t} marks the end of a multi-class wave! Flushed buffer.")

        learned_classes_over_time.append(set(learned_classes_over_time[-1]) if len(learned_classes_over_time) > 0 else set())
        for p in promoted_classes_log:
            if p["task"] == t and p["semantic"] not in learned_classes_over_time[-1]:
                learned_classes_over_time[-1].add(p["semantic"])

        acc, avg_forg, class_accuracies[t] = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
        task_kca.append(acc)
        average_forgetting.append(avg_forg)
        print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f} | Classes: {len(learned_classes_over_time[-1])}")
        
        # 🚀 RESTORED: Task-wise individual class accuracies
        for sem, c_acc in class_accuracies[t].items():
            print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")

    # ============================================================
    # STORE EXPERIMENT RESULT
    # ============================================================
    final_accs = class_accuracies[NUM_TASKS - 1]
    avg_init_pur = np.mean(exp_init_purities) if exp_init_purities else 0.0
    avg_core_pur = np.mean(exp_core_purities) if exp_core_purities else 0.0
    
    row_result = {
        "Experiment": exp_name,
        "Final KCA": f"{task_kca[-1]:.3f}",
        "Avg Forg": f"{average_forgetting[-1]:.3f}",
        "Classes": str(len(learned_classes_over_time[-1])),
        "Init Pur": f"{avg_init_pur:.2f}",
        "Core Pur": f"{avg_core_pur:.2f}"
    }
    
    # 🚀 Add dynamic class headers like C0(airplane)
    for cls_id in range(10): 
        row_result[f"C{cls_id}({CIFAR10_LABELS[cls_id][:3]})"] = f"{final_accs.get(cls_id, 0.0):.2f}"
        
    final_results_summary.append(row_result)

# ============================================================
# CREATE & SAVE THE MASTER ABLATION TABLE (PNG)
# ============================================================
print("\n" + "="*160)
print("📊 MASTER ABLATION SUMMARY: TRACK A (CLUSTER PURIFICATION)")
print("="*160)

columns = list(final_results_summary[0].keys())
cell_text = []

header_format = "{:<32} | {:<9} | {:<8} | {:<7} | {:<8} | {:<8} | " + " | ".join([f"{{:<10}}"]*10)
print(header_format.format(*columns))
print("-" * 175)

for res in final_results_summary:
    row = [res[col] for col in columns]
    cell_text.append(row)
    print(header_format.format(*row))

fig, ax = plt.subplots(figsize=(32, len(cell_text) * 0.6 + 2))
ax.axis('off')
ax.axis('tight')

table = ax.table(cellText=cell_text, colLabels=columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 2.0)

for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#2c3e50')
    else:
        if i % 2 == 0: cell.set_facecolor('#f8f9fa')

plt.title("Master Ablation Summary: Track A (Cluster Purification)", fontsize=18, fontweight='bold', pad=20)
plt.savefig("ablation_track_A_summary.png", bbox_inches='tight', dpi=300)
plt.close()

print("\n✅ Ablation complete. Master table saved to 'ablation_track_A_summary.png'")

## improving on above (EXP A.2) phase 2

In [ ]:
import os
# 🚨 THESE MUST BE SET BEFORE IMPORTING TORCH
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from torch.utils.data import DataLoader, TensorDataset, Dataset
import hdbscan
import copy
from torchvision import datasets as tv_datasets, transforms
import torchvision.transforms as T  

# LabelBench Imports
from LabelBench.skeleton.dataset_skeleton import datasets as DATASET_REGISTRY
from LabelBench.skeleton.dataset_skeleton import register_dataset, LabelType, TransformDataset
from torchvision.models import resnet18

# ============================================================
# BULLETPROOF REPRODUCIBILITY LOCK
# ============================================================
def set_seed(seed=42):
    """Forces strict deterministic behavior across different machines/Colab accounts."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=False)

def get_locked_generator(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# ============================================================
# STABLE METRIC LEARNING LOSSES
# ============================================================
def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

# ============================================================
# DIRECTORIES & CONFIGURATION 
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
global BASE_DIR
BASE_DIR = f"debug_OptimizedStrict_margin_contrastive"
os.makedirs(BASE_DIR, exist_ok=True)

# BUFFER CONFIG
P = 1                              
TOPK_NOVELTY = 400              
MAX_NOVELTY_BUFFER = 2500        
STARTING_TTL = 3

# HPT CONFIG
ALPHA = 0.35              
BETA = 0.01                
DELTA = 15                 
EPSILON = 0.00
METHOD = "margin_contrastive"

CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat",
    4: "deer", 5: "dog", 6: "frog", 7: "horse",
    8: "ship", 9: "truck"
}

# ============================================================
# EXPERIMENT TRACK A (PHASE 2): HYPERPARAMETER SWEEP
# ============================================================
EXPERIMENTS = {
    "Baseline (Control)": {
        "type": "baseline", 
        "min_cluster_size": 150, 
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    },
    "MSP Tight (< 0.85)": {
        "type": "msp", "thresh": 0.85,
        "min_cluster_size": 75, 
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    },
    "MSP Medium (< 0.95)": {
        "type": "msp", "thresh": 0.95,
        "min_cluster_size": 75, 
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    },
    "Centroid (Top 75%)": {
        "type": "centroid", "ratio": 0.75,
        "min_cluster_size": 100,  # Adjusted for 75% retention
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    },
    "Known Repulsion (< 0.55)": {
        "type": "repulsion", "thresh": 0.55, # Relaxed from 0.50
        "min_cluster_size": 75, 
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    },
    "HDBSCAN Prob (>= 0.40)": {
        "type": "hdbscan_prob", "thresh": 0.40, # Relaxed from 0.80
        "min_cluster_size": 75, 
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    },
    "k-NN (k=7, overlap=6)": {
        "type": "knn", "k": 7, "overlap": 6, # Relaxed logic
        "min_cluster_size": 75, 
        "use_ttl": False, "use_persistence": False, "epochs": 25, "feat_weight": 1.0, "imprint": False
    }
}

# ============================================================
# DYNAMIC 40-TASK DATASET DEFINITION (MULTI-CLASS WAVE LOGIC)
# ============================================================
NUM_TASKS = 40   

class CIFARStream(Dataset):
    def __init__(self, base_ds, indices):
        self.base_ds = base_ds      
        self.indices = indices      

    def __len__(self): return len(self.indices)
    def __getitem__(self, idx): return self.base_ds[self.indices[idx]]

def one_hot(y, n=10): return F.one_hot(torch.tensor(y), num_classes=n).float()

@register_dataset("splitcifar10", LabelType.MULTI_CLASS)
def get_splitcifar10(_): raise RuntimeError("Use splitcifar10_<id>")

base_train_global = tv_datasets.CIFAR10(root="./data", train=True, download=True)
targets = np.array(base_train_global.targets)
class_indices = {c: np.where(targets == c)[0] for c in range(10)}

rng = np.random.default_rng(42)
for c in range(10): rng.shuffle(class_indices[c])

stream_splits = {}
for t in range(1, NUM_TASKS):
    stream_splits[t] = []
    if 1 <= t <= 13: reminders = [0, 1]
    elif 14 <= t <= 26: reminders = [0, 1, 2, 3]
    elif 27 <= t <= 39: reminders = [0, 1, 2, 3, 4, 5, 6]
    else: reminders = []
        
    for c in reminders: stream_splits[t].extend(rng.choice(class_indices[c], 100, replace=False))
        
    if 1 <= t <= 13:
        stream_splits[t].extend(np.array_split(class_indices[2], 13)[t-1])
        stream_splits[t].extend(np.array_split(class_indices[3], 13)[t-1])
    elif 14 <= t <= 26:
        stream_splits[t].extend(np.array_split(class_indices[4], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[5], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[6], 13)[t-14])
    elif 27 <= t <= 39:
        stream_splits[t].extend(np.array_split(class_indices[7], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[8], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[9], 13)[t-27])
        
    rng.shuffle(stream_splits[t])

for split_id in range(NUM_TASKS):
    @register_dataset(f"splitcifar10_{split_id}", LabelType.MULTI_CLASS)
    def _make_split(data_dir, split_id=split_id):
        tf = transforms.Compose([transforms.ToTensor()])
        base_train = tv_datasets.CIFAR10(root=data_dir, train=True, download=True)
        base_test  = tv_datasets.CIFAR10(root=data_dir, train=False, download=True)
        
        if split_id == 0: indices = [i for i,(x,y) in enumerate(base_train) if y in [0,1]]
        else: indices = stream_splits[split_id]

        train_ds = TransformDataset(CIFARStream(base_train, indices), transform=tf, target_transform=lambda y: one_hot(y,10))
        test_ds = TransformDataset(base_test, transform=tf, target_transform=lambda y: one_hot(y,10))
        return train_ds, test_ds, test_ds, None, None, None, 10, [str(i) for i in range(10)]

# ============================================================
# MODEL & TRACKERS
# ============================================================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = resnet18(weights='DEFAULT')
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes, bias=False)
        self.scale = 20.0  

    def expand_head(self, new_classes, new_centroid=None):
        old_w = self.classifier.weight.data.clone()
        old_n = old_w.shape[0]
        new_classifier = nn.Linear(512, new_classes, bias=False).to(old_w.device)
        new_classifier.weight.data[:old_n] = old_w
        if new_centroid is not None:
            new_classifier.weight.data[old_n:] = new_centroid.to(old_w.device)
        self.classifier = new_classifier

    def forward(self, x, labels=None):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = self.embed(z)
        z = F.normalize(z, dim=1)
        W = F.normalize(self.classifier.weight, dim=1)
        cosine = torch.matmul(z, W.t())
        if labels is not None: 
            m = 0.2 
            theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
            target_logits = torch.cos(theta + m)
            one_hot = F.one_hot(labels, num_classes=W.size(0)).float()
            logits = cosine * (1 - one_hot) + target_logits * one_hot
        else:
            logits = cosine
        logits = logits * self.scale
        return logits, z

class MemoryBuffer:
    def __init__(self, max_per_class=400):
        self.data = defaultdict(list)
        self.max_per_class = max_per_class
        self.aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])

    @torch.no_grad()
    def build_memory_herding(self, X_all, y_label, model):
        model.eval()
        logits_all, Z_all = [], []
        for i in range(0, len(X_all), 128):
            batch = X_all[i:i+128].to(DEVICE)
            logits, z = model(batch)
            logits_all.append(logits.cpu())
            Z_all.append(z.cpu())
        logits_all = torch.cat(logits_all)
        Z_all = torch.cat(Z_all)
        class_mean = F.normalize(Z_all.mean(0), dim=0)
        selected_idx = []
        features = Z_all.clone()

        for k in range(min(self.max_per_class, len(X_all))):
            S = Z_all[selected_idx].sum(0) if k > 0 else torch.zeros_like(class_mean)
            target = (k + 1) * class_mean - S
            distances = torch.norm(features - target, dim=1)
            for idx in selected_idx: distances[idx] = float('inf')
            selected_idx.append(distances.argmin().item())

        self.data[int(y_label)] = []
        for idx in selected_idx:
            self.data[int(y_label)].append((X_all[idx].detach().cpu(), logits_all[idx].detach().cpu()))
            
    def get_all_features_and_labels(self, model):
        X_all, Y_all = [], []
        for c, data in self.data.items():
            for x, _ in data:
                X_all.append(x)
                Y_all.append(c)
        if not X_all: return None, None
        X_all = torch.stack(X_all).to(DEVICE)
        with torch.no_grad(): _, Z_all = model(X_all)
        return F.normalize(Z_all, dim=1), torch.tensor(Y_all).to(DEVICE)
    
    def get(self): return self.data

    def sample_balanced(self, batch_size, model):
        classes = list(self.data.keys())
        if not classes: return None, None, None
        samples_per_class = max(1, batch_size // len(classes))
        X_mem, Y_mem, L_mem = [], [], []
        current_dim = model.classifier.out_features
        for cls in classes:
            samples = self.data[cls]
            if len(samples) == 0: continue
            replace = len(samples) < samples_per_class
            idx = np.random.choice(len(samples), samples_per_class, replace=replace)
            for i in idx:
                x, logit = samples[i]
                if logit.shape[0] < current_dim:
                    padded = torch.zeros(current_dim)
                    padded[:logit.shape[0]] = logit
                    logit = padded
                X_mem.append(x)
                Y_mem.append(cls)
                L_mem.append(logit)
        if not X_mem: return None, None, None
        X_tensor = torch.stack(X_mem)
        return self.aug(X_tensor), torch.tensor(Y_mem), torch.stack(L_mem)

class HypersphereNovelty:
    def __init__(self, q=0.90):
        self.q = q
        self.mu, self.r = {}, {}

    def update(self, memory, model):
        self.mu, self.r = {}, {}
        for k, X_tuples in memory.items():
            if len(X_tuples) == 0: continue
            X = torch.stack([x for x, _ in X_tuples]).to(DEVICE)
            with torch.no_grad(): _, Z = model(X)
            mu = F.normalize(Z.mean(0), dim=0)
            d = 1 - torch.matmul(Z, mu)
            self.mu[k] = mu
            self.r[k] = torch.quantile(d, self.q)

    def score(self, z):
        if not self.mu: return torch.tensor(0.0)
        return min([(1 - torch.dot(z, self.mu[k].to(z.device))) - self.r[k].to(z.device) for k in self.mu])

# ============================================================
# TRAINING LOOPS
# ============================================================
def train_supervised(model, loader, test_loader, task_id):
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    
    for epoch in range(30):
        for x, y in loader:
            x_device, y_device = x.to(DEVICE), y.argmax(1).to(DEVICE)
            x_aug = aug(x_device)
            logits, z = model(x_aug, y_device)
            z_norm = F.normalize(z, dim=1)
            loss = F.cross_entropy(logits, y_device) + 1.0 * margin_contrastive_loss(z_norm, y_device)
            opt.zero_grad()
            loss.backward()
            opt.step()

def finetune(model, memory, X_new, new_label, task_id, locked_gen, exp_config): 
    old_model = copy.deepcopy(model).eval()
    for p in old_model.parameters(): p.requires_grad = False 
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()
            
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    Y_new = torch.full((len(X_new),), new_label, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_new, Y_new), batch_size=32, shuffle=True, generator=locked_gen)

    for p in model.parameters(): p.requires_grad = True
    opt = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

    # 🚀 Dynamically use experiment-specific hyperparameters
    epochs = exp_config.get("epochs", 25)
    feat_weight = exp_config.get("feat_weight", 1.0)

    for epoch in range(epochs):
        for xb, yb in loader:
            xb = aug(xb) 
            X_mem, Y_mem, L_mem = memory.sample_balanced(32, model)
            if X_mem is not None:
                xb = torch.cat([xb, X_mem], dim=0)
                yb = torch.cat([yb, Y_mem], dim=0)

            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits_margin, Z = model(xb, yb)
            loss_ce = F.cross_entropy(logits_margin, yb)
            
            if X_mem is not None:
                pure_logits, _ = model(xb) 
                logits_mem = pure_logits[-len(X_mem):]
                loss_der = F.mse_loss(logits_mem, L_mem.to(DEVICE))
            else:
                loss_der = torch.tensor(0.0, device=DEVICE)

            with torch.no_grad(): logits_old, Z_old = old_model(xb)
            loss_feat = (1 - F.cosine_similarity(Z, Z_old)).mean()
            
            loss = loss_ce + 0.5 * loss_der + feat_weight * loss_feat
            opt.zero_grad()
            loss.backward()
            opt.step()
    model.eval()

def evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies):
    correct_per_class = defaultdict(int)
    total_per_class = defaultdict(int)
    model.eval()
    with torch.no_grad():
        for x_test, y_test in DataLoader(test_ds, batch_size=128):
            x_test = x_test.to(DEVICE)
            y_test = y_test.argmax(1).to(DEVICE)
            logits, _ = model(x_test)
            preds = logits.argmax(1)
            for i in range(len(y_test)):
                sem = int(y_test[i])
                if sem in semantic_to_internal:
                    internal_gt = semantic_to_internal[sem]
                    total_per_class[sem] += 1
                    if preds[i].item() == internal_gt:
                        correct_per_class[sem] += 1

    current_class_accs = {}
    for sem in semantic_to_internal.keys():
        current_class_accs[sem] = correct_per_class[sem] / max(total_per_class[sem], 1)
        
    class_accuracies[t] = current_class_accs
    total_correct = sum(correct_per_class.values())
    total_eval = sum(total_per_class.values())
    acc = total_correct / max(total_eval, 1)
    
    if t > 0:
        forgetting_list = []
        for sem in current_class_accs.keys():
            past_accs = [class_accuracies[k].get(sem, None) for k in range(t)]
            past_accs = [a for a in past_accs if a is not None]
            if past_accs:
                max_past = max(past_accs)
                forgetting = max_past - current_class_accs[sem]
                forgetting_list.append(forgetting)
        avg_forg = np.mean(forgetting_list) if forgetting_list else 0.0
    else:
        avg_forg = 0.0
    return acc, avg_forg, current_class_accs


# ============================================================
# MASTER EXPERIMENT LOOP
# ============================================================
print("\n" + "="*100)
print(f"🚀 INITIATING MASTER ABLATION STUDY: TRACK A (PHASE 2)")
print("="*100)

final_results_summary = []
TASKS = [f"splitcifar10_{i}" for i in range(NUM_TASKS)]

for exp_name, exp_config in EXPERIMENTS.items():
    print("\n\n" + "#"*100)
    print(f"⚙️  RUNNING EXPERIMENT: {exp_name}")
    print("#"*100)

    # 🚨 HARD RESET: Ensure Identical Starting Conditions
    set_seed(42)
    locked_generator = get_locked_generator(42)
    model = CNN(num_classes=2).to(DEVICE)
    memory = MemoryBuffer(max_per_class=400) 
    detector = HypersphereNovelty()
    
    novelty_buffer = []
    semantic_to_internal = {0: 0, 1: 1}
    internal_to_semantic = {0: 0, 1: 1}
    known_classes = 2
    
    task_kca, task_cpr, average_forgetting = [], [], []
    class_accuracies = {}   
    learned_classes_over_time = []
    promoted_classes_log = []
    
    # Trackers for Purity evaluation over the experiment
    exp_init_purities = []
    exp_core_purities = []

    for t, task in enumerate(TASKS):
        _, dataset_fn = DATASET_REGISTRY[task]
        train_ds, test_ds, *_ = dataset_fn("./data")
        loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=locked_generator) 

        print(f"\n{'-'*20} 🚀 TASK {t} {'-'*20}")
        
        # 🚀 RESTORED STREAM DEBUGGING
        all_semantics = []
        for _, y in loader: all_semantics.extend(y.argmax(1).tolist())
        unique_semantics = list(set(all_semantics))
        class_names = [CIFAR10_LABELS[sem] for sem in unique_semantics]
        print(f"📦 [STREAM] Task {t} stream contains class(es): {class_names} (Label IDs: {unique_semantics})")
        print(f"📊 [STREAM] Total images in this chunk: {len(train_ds)}")

        if t == 0:
            train_supervised(model, loader, DataLoader(test_ds, batch_size=128, shuffle=False), t)
            for cls in [0, 1]:
                Xc = torch.cat([x[y.argmax(1) == cls] for x, y in loader])
                memory.build_memory_herding(Xc, cls, model)
            detector.update(memory.get(), model)
            learned_classes_over_time.append({CIFAR10_LABELS[0], CIFAR10_LABELS[1]})
            acc, avg_forg, class_accuracies[t] = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
            task_kca.append(acc); average_forgetting.append(avg_forg)
            print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")
            for sem, c_acc in class_accuracies[t].items():
                print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")
            continue

        model.eval()
        novelty_candidates = []
        novel, false_novel, total = 0, 0, 0
        with torch.no_grad():
            for x, y in loader:
                _, z = model(x.to(DEVICE))
                y_labels = y.argmax(1)
                scores = [detector.score(z[i]).item() for i in range(len(z))]
                thr = np.percentile(scores, 30)
                for i in range(len(z)):
                    total += 1
                    if scores[i] > thr: 
                        novelty_candidates.append((scores[i], x[i].cpu(), y_labels[i].item()))
                        novel += 1
                        if y_labels[i].item() in semantic_to_internal: false_novel += 1

        print(f"🔍 [NOVELTY] Flagged Novel: {novel}/{total} | False Novelty: {false_novel}")

        novelty_candidates.sort(reverse=True, key=lambda x: x[0])
        novelty_buffer.extend([(img, y, STARTING_TTL) for _, img, y in novelty_candidates[:TOPK_NOVELTY]])
        novelty_buffer = novelty_buffer[-MAX_NOVELTY_BUFFER:]

        if t % P == 0 and len(novelty_buffer) >= 20: 
            print(f"\n🧠 [CLUSTERING] Running HDBSCAN on buffer (Size: {len(novelty_buffer)})...")
            Z = []
            with torch.no_grad():
                for img, _, _ in novelty_buffer:
                    _, z = model(img.unsqueeze(0).to(DEVICE))
                    Z.append(z.squeeze().cpu().numpy())

            Z = np.stack(Z)
            clusterer = hdbscan.HDBSCAN(metric='euclidean', min_cluster_size=20, cluster_selection_epsilon=EPSILON)
            labels = clusterer.fit_predict(Z)
            persistences = clusterer.cluster_persistence_
            max_persistence = np.max(persistences) if len(persistences) > 0 else 0.0
            
            new_buffer = []
            found, promoted = 0, 0

            # Dynamic TTL Config
            use_ttl = exp_config.get("use_ttl", False)

            for cid in sorted(set(labels)):
                idxs_full = np.where(labels == cid)[0].tolist()
                
                if cid == -1: 
                    print(f"  🗑️  [CLUSTER -1] NOISE: Found {len(idxs_full)} noise samples.")
                    for i in idxs_full: 
                        img, y, ttl = novelty_buffer[i]
                        if use_ttl:
                            if ttl - 1 > 0: new_buffer.append((img, y, ttl - 1))
                        else:
                            new_buffer.append((img, y, ttl))
                    continue

                found += 1
                Xc_full = torch.stack([novelty_buffer[i][0] for i in idxs_full])
                with torch.no_grad(): _, Zc_full = model(Xc_full.to(DEVICE))
                
                # =========================================================
                # 🧪 APPLY DYNAMIC PURIFICATION EXPERIMENT FILTER
                # =========================================================
                idxs = []
                if exp_config["type"] == "baseline":
                    idxs = idxs_full
                elif exp_config["type"] == "centroid":
                    mu_full = F.normalize(Zc_full.mean(0), dim=0)
                    sims_to_centroid = torch.matmul(Zc_full, mu_full)
                    keep_k = max(1, int(len(idxs_full) * exp_config["ratio"]))
                    _, topk_indices = torch.topk(sims_to_centroid, keep_k)
                    idxs = [idxs_full[idx.item()] for idx in topk_indices]
                elif exp_config["type"] == "repulsion":
                    if len(detector.mu) > 0:
                        known_mu = torch.stack(list(detector.mu.values())).to(DEVICE)
                        sims = torch.matmul(Zc_full, known_mu.T)
                        max_sims, _ = torch.max(sims, dim=1)
                        idxs = [idxs_full[i] for i in range(len(idxs_full)) if max_sims[i].item() < exp_config["thresh"]]
                    else:
                        idxs = idxs_full
                elif exp_config["type"] == "hdbscan_prob":
                    probs = clusterer.probabilities_[idxs_full]
                    idxs = [idxs_full[i] for i in range(len(idxs_full)) if probs[i] >= exp_config["thresh"]]
                elif exp_config["type"] == "msp":
                    with torch.no_grad():
                        logits, _ = model(Xc_full.to(DEVICE))
                        probs = F.softmax(logits, dim=1)
                        max_probs, _ = torch.max(probs, dim=1)
                    idxs = [idxs_full[i] for i in range(len(idxs_full)) if max_probs[i].item() < exp_config["thresh"]]
                elif exp_config["type"] == "knn":
                    Z_mem, Y_mem = memory.get_all_features_and_labels(model)
                    if Z_mem is not None:
                        sims = torch.matmul(Zc_full, Z_mem.T) 
                        _, topk_idx = torch.topk(sims, exp_config["k"], dim=1)
                        valid_indices = []
                        for i in range(len(idxs_full)):
                            neighbors = Y_mem[topk_idx[i]].tolist()
                            most_common_cnt = Counter(neighbors).most_common(1)[0][1]
                            if most_common_cnt < exp_config["overlap"]:
                                valid_indices.append(idxs_full[i])
                        idxs = valid_indices
                    else:
                        idxs = idxs_full

                labels_true_full = [novelty_buffer[i][1] for i in idxs_full]
                purity_full = Counter(labels_true_full).most_common(1)[0][1] / len(labels_true_full)
                
                # Apply buffer retention rules to items filtered out during purification
                filtered_out = [idx for idx in idxs_full if idx not in idxs]
                for i in filtered_out:
                    img, y, ttl = novelty_buffer[i]
                    if use_ttl:
                        if ttl - 1 > 0: new_buffer.append((img, y, ttl - 1))
                    else:
                        new_buffer.append((img, y, ttl))
                
                if len(idxs) == 0:
                    print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Completely eliminated by {exp_config['type']} filter.")
                    continue 
                    
                labels_true_core = [novelty_buffer[i][1] for i in idxs]
                sem_label, cnt_core = Counter(labels_true_core).most_common(1)[0]
                purity_core = cnt_core / len(labels_true_core)
                
                # =========================================================
                # 📊 RICH EVALUATION ON THE PURIFIED CORE
                # =========================================================
                Xc = torch.stack([novelty_buffer[i][0] for i in idxs])
                with torch.no_grad(): _, Zc = model(Xc.to(DEVICE))
                
                mu = F.normalize(Zc.mean(0), dim=0)
                n = len(idxs)
                S_intra = torch.mean(1 - torch.matmul(Zc, mu))
                S_known = min([1 - torch.dot(mu, detector.mu[k].to(mu.device)) for k in detector.mu]) if len(detector.mu)>0 else torch.tensor(1.0)
                density = n / (S_intra.item() + 1e-6)
                margin = S_known - S_intra
                
                if sem_label in semantic_to_internal:
                    print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Dominant class '{CIFAR10_LABELS[sem_label]}' already known.")
                    for i in idxs: 
                        img, y, ttl = novelty_buffer[i]
                        if use_ttl:
                            if ttl - 1 > 0: new_buffer.append((img, y, ttl - 1))
                        else:
                            new_buffer.append((img, y, ttl))
                    continue 

                req_min_size = exp_config.get("min_cluster_size", 150)
                
                if exp_config.get("use_persistence", False):
                    relative_persistence = persistences[cid] / (max_persistence + 1e-9)
                    is_highly_persistent = (relative_persistence >= 0.75)
                else:
                    relative_persistence = 0.0
                    is_highly_persistent = False

                cond_intra = S_intra.item() <= ALPHA
                cond_density = density >= DELTA
                cond_known = S_known.item() >= BETA
                cond_margin = margin.item() > -0.10
                cond_size = n >= req_min_size

                print(f"\n  📊 [CLUSTER {cid} EVALUATION] Dominant: '{CIFAR10_LABELS[sem_label]}' (Init Pur: {purity_full:.2f} -> Core Pur: {purity_core:.2f})")
                print(f"     ➔ Size:    {n:3d}   (Req: >= {req_min_size:2d})   {'✅' if cond_size else '❌'}")
                print(f"     ➔ S_intra: {S_intra.item():.3f} (Req: <= {ALPHA:.2f}) {'✅' if cond_intra else '❌'}")
                print(f"     ➔ S_known: {S_known.item():.3f} (Req: >= {BETA:.2f}) {'✅' if cond_known else '❌'}")
                print(f"     ➔ Density: {density:.1f} (Req: >= {DELTA})   {'✅' if cond_density else '❌'}")
                print(f"     ➔ Margin:  {margin.item():.3f} (Req: > -0.10) {'✅' if cond_margin else '❌'}")
                if exp_config.get("use_persistence", False):
                    print(f"     ➔ Rel Pers:{relative_persistence:.2f} (Req: >= 0.75)  {'✅' if is_highly_persistent else '❌'}")

                if cond_intra and cond_density and cond_size and cond_margin and (cond_known or is_highly_persistent):
                    if not cond_known and is_highly_persistent:
                        print(f"     🌟 [PERSISTENCE BYPASS] S_known failed, but highly stable cluster detected!")
                        
                    promoted += 1
                    new_label = known_classes
                    semantic_to_internal[sem_label] = new_label
                    internal_to_semantic[new_label] = sem_label
                    known_classes += 1
                    
                    exp_init_purities.append(purity_full)
                    exp_core_purities.append(purity_core)
                    
                    if exp_config.get("imprint", False):
                        model.expand_head(known_classes, new_centroid=mu)
                    else:
                        model.expand_head(known_classes)
                    model.to(DEVICE)
                    
                    print(f"     🎉 [PROMOTION SUCCESS] -> Preparing to Finetune '{CIFAR10_LABELS[sem_label]}'...")
                    finetune(model, memory, Xc, new_label, t, locked_generator, exp_config)
                    print(f"     ✨ [LEARNED] -> Network successfully adapted to '{CIFAR10_LABELS[sem_label]}'")
                    
                    memory.build_memory_herding(Xc, new_label, model)
                    detector.update(memory.get(), model)
                    promoted_classes_log.append({"task": t, "semantic": CIFAR10_LABELS[sem_label]})
                else:
                    print(f"     🛑 [PROMOTION FAILED] -> Conditions not met. Retaining samples in buffer.")
                    for i in idxs: 
                        img, y, ttl = novelty_buffer[i]
                        if use_ttl:
                            if ttl - 1 > 0: new_buffer.append((img, y, ttl - 1))
                        else:
                            new_buffer.append((img, y, ttl))
                    
            novelty_buffer = new_buffer
            task_cpr.append(promoted / max(found, 1))
        else:
            task_cpr.append(0.0)
            
        if t > 0 and t % 13 == 0: 
            novelty_buffer = [] 
            print(f"\n🧹 [BUFFER CLEAR] Task {t} marks the end of a multi-class wave! Flushed buffer.")

        learned_classes_over_time.append(set(learned_classes_over_time[-1]) if len(learned_classes_over_time) > 0 else set())
        for p in promoted_classes_log:
            if p["task"] == t and p["semantic"] not in learned_classes_over_time[-1]:
                learned_classes_over_time[-1].add(p["semantic"])

        acc, avg_forg, class_accuracies[t] = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
        task_kca.append(acc)
        average_forgetting.append(avg_forg)
        print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f} | Classes: {len(learned_classes_over_time[-1])}")
        
        # 🚀 RESTORED: Task-wise individual class accuracies
        for sem, c_acc in class_accuracies[t].items():
            print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")

    # ============================================================
    # STORE EXPERIMENT RESULT
    # ============================================================
    final_accs = class_accuracies[NUM_TASKS - 1]
    avg_init_pur = np.mean(exp_init_purities) if exp_init_purities else 0.0
    avg_core_pur = np.mean(exp_core_purities) if exp_core_purities else 0.0
    
    row_result = {
        "Experiment": exp_name,
        "Final KCA": f"{task_kca[-1]:.3f}",
        "Avg Forg": f"{average_forgetting[-1]:.3f}",
        "Classes": str(len(learned_classes_over_time[-1])),
        "Init Pur": f"{avg_init_pur:.2f}",
        "Core Pur": f"{avg_core_pur:.2f}"
    }
    
    # 🚀 Add dynamic class headers like C0(airplane)
    for cls_id in range(10): 
        row_result[f"C{cls_id}({CIFAR10_LABELS[cls_id][:3]})"] = f"{final_accs.get(cls_id, 0.0):.2f}"
        
    final_results_summary.append(row_result)

# ============================================================
# CREATE & SAVE THE MASTER ABLATION TABLE (PNG)
# ============================================================
print("\n" + "="*160)
print("📊 MASTER ABLATION SUMMARY: TRACK A (PHASE 2)")
print("="*160)

columns = list(final_results_summary[0].keys())
cell_text = []

header_format = "{:<32} | {:<9} | {:<8} | {:<7} | {:<8} | {:<8} | " + " | ".join([f"{{:<10}}"]*10)
print(header_format.format(*columns))
print("-" * 175)

for res in final_results_summary:
    row = [res[col] for col in columns]
    cell_text.append(row)
    print(header_format.format(*row))

fig, ax = plt.subplots(figsize=(32, len(cell_text) * 0.6 + 2))
ax.axis('off')
ax.axis('tight')

table = ax.table(cellText=cell_text, colLabels=columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 2.0)

for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#2c3e50')
    else:
        if i % 2 == 0: cell.set_facecolor('#f8f9fa')

plt.title("Master Ablation Summary: Track A (Phase 2)", fontsize=18, fontweight='bold', pad=20)
plt.savefig("ablation_track_A_phase2_summary.png", bbox_inches='tight', dpi=300)
plt.close()

print("\n✅ Ablation complete. Master table saved to 'ablation_track_A_phase2_summary.png'")

### track B.1 clustering algo checking 

In [ ]:
import os
# 🚨 THESE MUST BE SET BEFORE IMPORTING TORCH
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import random
import traceback
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from torch.utils.data import DataLoader, TensorDataset, Dataset
import hdbscan
import copy
from torchvision import datasets as tv_datasets, transforms
import torchvision.transforms as T  

# Track B Specific Imports
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_distances
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components

# LabelBench Imports
from LabelBench.skeleton.dataset_skeleton import datasets as DATASET_REGISTRY
from LabelBench.skeleton.dataset_skeleton import register_dataset, LabelType, TransformDataset
from torchvision.models import resnet18

# ============================================================
# BULLETPROOF REPRODUCIBILITY LOCK
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=False)

def get_locked_generator(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# ============================================================
# STABLE METRIC LEARNING LOSSES
# ============================================================
def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

# ============================================================
# DIRECTORIES & CONFIGURATION 
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
global BASE_DIR
BASE_DIR = f"debug_TrackB_Clustering"
os.makedirs(BASE_DIR, exist_ok=True)

# BUFFER CONFIG
P = 1                              
TOPK_NOVELTY = 400              
MAX_NOVELTY_BUFFER = 2500        

# HPT CONFIG (STRICT GATES)
ALPHA = 0.35              
BETA = 0.01                
DELTA = 15                 

CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat",
    4: "deer", 5: "dog", 6: "frog", 7: "horse",
    8: "ship", 9: "truck"
}

# ============================================================
# EXPERIMENT TRACK B: THE COSINE CLUSTERING SWEEP
# ============================================================
EXPERIMENTS = {
    "Baseline (HDBSCAN Euclidean)": {
        "type": "hdbscan_euclidean", "min_cluster_size": 150
    },
    "HDBSCAN Cosine": {
        "type": "hdbscan_cosine", "min_cluster_size": 150
    },
    "Spherical K-Means (Dynamic)": {
        "type": "kmeans_cosine", "min_cluster_size": 150
    },
    "FINCH 1-NN Graph (Cosine)": {
        "type": "finch_proxy", "min_cluster_size": 150
    },
    "SNN-DBSCAN (Shared NN)": {
        "type": "snn_dbscan", "min_cluster_size": 150
    },
    "Agglomerative (Cosine + Avg)": {
        "type": "agglomerative", "min_cluster_size": 150
    }
}

# ============================================================
# DYNAMIC 40-TASK DATASET DEFINITION (MULTI-CLASS WAVE LOGIC)
# ============================================================
NUM_TASKS = 40   

class CIFARStream(Dataset):
    def __init__(self, base_ds, indices):
        self.base_ds = base_ds      
        self.indices = indices      

    def __len__(self): return len(self.indices)
    def __getitem__(self, idx): return self.base_ds[self.indices[idx]]

def one_hot(y, n=10): return F.one_hot(torch.tensor(y), num_classes=n).float()

@register_dataset("splitcifar10", LabelType.MULTI_CLASS)
def get_splitcifar10(_): raise RuntimeError("Use splitcifar10_<id>")

base_train_global = tv_datasets.CIFAR10(root="./data", train=True, download=True)
targets = np.array(base_train_global.targets)
class_indices = {c: np.where(targets == c)[0] for c in range(10)}

rng = np.random.default_rng(42)
for c in range(10): rng.shuffle(class_indices[c])

stream_splits = {}
for t in range(1, NUM_TASKS):
    stream_splits[t] = []
    if 1 <= t <= 13: reminders = [0, 1]
    elif 14 <= t <= 26: reminders = [0, 1, 2, 3]
    elif 27 <= t <= 39: reminders = [0, 1, 2, 3, 4, 5, 6]
    else: reminders = []
        
    for c in reminders: stream_splits[t].extend(rng.choice(class_indices[c], 100, replace=False))
        
    if 1 <= t <= 13:
        stream_splits[t].extend(np.array_split(class_indices[2], 13)[t-1])
        stream_splits[t].extend(np.array_split(class_indices[3], 13)[t-1])
    elif 14 <= t <= 26:
        stream_splits[t].extend(np.array_split(class_indices[4], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[5], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[6], 13)[t-14])
    elif 27 <= t <= 39:
        stream_splits[t].extend(np.array_split(class_indices[7], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[8], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[9], 13)[t-27])
        
    rng.shuffle(stream_splits[t])

for split_id in range(NUM_TASKS):
    @register_dataset(f"splitcifar10_{split_id}", LabelType.MULTI_CLASS)
    def _make_split(data_dir, split_id=split_id):
        tf = transforms.Compose([transforms.ToTensor()])
        base_train = tv_datasets.CIFAR10(root=data_dir, train=True, download=True)
        base_test  = tv_datasets.CIFAR10(root=data_dir, train=False, download=True)
        
        if split_id == 0: indices = [i for i,(x,y) in enumerate(base_train) if y in [0,1]]
        else: indices = stream_splits[split_id]

        train_ds = TransformDataset(CIFARStream(base_train, indices), transform=tf, target_transform=lambda y: one_hot(y,10))
        test_ds = TransformDataset(base_test, transform=tf, target_transform=lambda y: one_hot(y,10))
        return train_ds, test_ds, test_ds, None, None, None, 10, [str(i) for i in range(10)]

# ============================================================
# MODEL & TRACKERS
# ============================================================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = resnet18(weights='DEFAULT')
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes, bias=False)
        self.scale = 20.0  

    def expand_head(self, new_classes, new_centroid=None):
        old_w = self.classifier.weight.data.clone()
        old_n = old_w.shape[0]
        new_classifier = nn.Linear(512, new_classes, bias=False).to(old_w.device)
        new_classifier.weight.data[:old_n] = old_w
        if new_centroid is not None:
            new_classifier.weight.data[old_n:] = new_centroid.to(old_w.device)
        self.classifier = new_classifier

    def forward(self, x, labels=None):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = self.embed(z)
        z = F.normalize(z, dim=1)
        W = F.normalize(self.classifier.weight, dim=1)
        cosine = torch.matmul(z, W.t())
        if labels is not None: 
            m = 0.2 
            theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
            target_logits = torch.cos(theta + m)
            one_hot = F.one_hot(labels, num_classes=W.size(0)).float()
            logits = cosine * (1 - one_hot) + target_logits * one_hot
        else:
            logits = cosine
        logits = logits * self.scale
        return logits, z

class MemoryBuffer:
    def __init__(self, max_per_class=400):
        self.data = defaultdict(list)
        self.max_per_class = max_per_class
        self.aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])

    @torch.no_grad()
    def build_memory_herding(self, X_all, y_label, model):
        model.eval()
        logits_all, Z_all = [], []
        for i in range(0, len(X_all), 128):
            batch = X_all[i:i+128].to(DEVICE)
            logits, z = model(batch)
            logits_all.append(logits.cpu())
            Z_all.append(z.cpu())
        logits_all = torch.cat(logits_all)
        Z_all = torch.cat(Z_all)
        class_mean = F.normalize(Z_all.mean(0), dim=0)
        selected_idx = []
        features = Z_all.clone()

        for k in range(min(self.max_per_class, len(X_all))):
            S = Z_all[selected_idx].sum(0) if k > 0 else torch.zeros_like(class_mean)
            target = (k + 1) * class_mean - S
            distances = torch.norm(features - target, dim=1)
            for idx in selected_idx: distances[idx] = float('inf')
            selected_idx.append(distances.argmin().item())

        self.data[int(y_label)] = []
        for idx in selected_idx:
            self.data[int(y_label)].append((X_all[idx].detach().cpu(), logits_all[idx].detach().cpu()))
    
    def get(self): return self.data

    def sample_balanced(self, batch_size, model):
        classes = list(self.data.keys())
        if not classes: return None, None, None
        samples_per_class = max(1, batch_size // len(classes))
        X_mem, Y_mem, L_mem = [], [], []
        current_dim = model.classifier.out_features
        for cls in classes:
            samples = self.data[cls]
            if len(samples) == 0: continue
            replace = len(samples) < samples_per_class
            idx = np.random.choice(len(samples), samples_per_class, replace=replace)
            for i in idx:
                x, logit = samples[i]
                if logit.shape[0] < current_dim:
                    padded = torch.zeros(current_dim)
                    padded[:logit.shape[0]] = logit
                    logit = padded
                X_mem.append(x)
                Y_mem.append(cls)
                L_mem.append(logit)
        if not X_mem: return None, None, None
        X_tensor = torch.stack(X_mem)
        return self.aug(X_tensor), torch.tensor(Y_mem), torch.stack(L_mem)

class HypersphereNovelty:
    def __init__(self, q=0.90):
        self.q = q
        self.mu, self.r = {}, {}

    def update(self, memory, model):
        self.mu, self.r = {}, {}
        for k, X_tuples in memory.items():
            if len(X_tuples) == 0: continue
            X = torch.stack([x for x, _ in X_tuples]).to(DEVICE)
            with torch.no_grad(): _, Z = model(X)
            mu = F.normalize(Z.mean(0), dim=0)
            d = 1 - torch.matmul(Z, mu)
            self.mu[k] = mu
            self.r[k] = torch.quantile(d, self.q)

    def score(self, z):
        if not self.mu: return torch.tensor(0.0)
        return min([(1 - torch.dot(z, self.mu[k].to(z.device))) - self.r[k].to(z.device) for k in self.mu])

# ============================================================
# TRAINING LOOPS
# ============================================================
def train_supervised(model, loader, test_loader, task_id):
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    for epoch in range(30):
        for x, y in loader:
            x_device, y_device = x.to(DEVICE), y.argmax(1).to(DEVICE)
            x_aug = aug(x_device)
            logits, z = model(x_aug, y_device)
            z_norm = F.normalize(z, dim=1)
            loss = F.cross_entropy(logits, y_device) + 1.0 * margin_contrastive_loss(z_norm, y_device)
            opt.zero_grad()
            loss.backward()
            opt.step()

def finetune(model, memory, X_new, new_label, task_id, locked_gen): 
    # STRICT BASELINE RULES: 25 Epochs, No Imprinting, Feat Weight 1.0
    old_model = copy.deepcopy(model).eval()
    for p in old_model.parameters(): p.requires_grad = False 
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()
            
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    Y_new = torch.full((len(X_new),), new_label, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_new, Y_new), batch_size=32, shuffle=True, generator=locked_gen)

    for p in model.parameters(): p.requires_grad = True
    opt = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

    for epoch in range(25):
        for xb, yb in loader:
            xb = aug(xb) 
            X_mem, Y_mem, L_mem = memory.sample_balanced(32, model)
            if X_mem is not None:
                xb = torch.cat([xb, X_mem], dim=0)
                yb = torch.cat([yb, Y_mem], dim=0)

            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits_margin, Z = model(xb, yb)
            loss_ce = F.cross_entropy(logits_margin, yb)
            
            if X_mem is not None:
                pure_logits, _ = model(xb) 
                logits_mem = pure_logits[-len(X_mem):]
                loss_der = F.mse_loss(logits_mem, L_mem.to(DEVICE))
            else:
                loss_der = torch.tensor(0.0, device=DEVICE)

            with torch.no_grad(): logits_old, Z_old = old_model(xb)
            loss_feat = (1 - F.cosine_similarity(Z, Z_old)).mean()
            
            loss = loss_ce + 0.5 * loss_der + 1.0 * loss_feat
            opt.zero_grad()
            loss.backward()
            opt.step()
    model.eval()

def evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies):
    correct_per_class = defaultdict(int)
    total_per_class = defaultdict(int)
    model.eval()
    with torch.no_grad():
        for x_test, y_test in DataLoader(test_ds, batch_size=128):
            x_test = x_test.to(DEVICE)
            y_test = y_test.argmax(1).to(DEVICE)
            logits, _ = model(x_test)
            preds = logits.argmax(1)
            for i in range(len(y_test)):
                sem = int(y_test[i])
                if sem in semantic_to_internal:
                    internal_gt = semantic_to_internal[sem]
                    total_per_class[sem] += 1
                    if preds[i].item() == internal_gt:
                        correct_per_class[sem] += 1

    current_class_accs = {}
    for sem in semantic_to_internal.keys():
        current_class_accs[sem] = correct_per_class[sem] / max(total_per_class[sem], 1)
        
    class_accuracies[t] = current_class_accs
    total_correct = sum(correct_per_class.values())
    total_eval = sum(total_per_class.values())
    acc = total_correct / max(total_eval, 1)
    
    if t > 0:
        forgetting_list = []
        for sem in current_class_accs.keys():
            past_accs = [class_accuracies[k].get(sem, None) for k in range(t)]
            past_accs = [a for a in past_accs if a is not None]
            if past_accs:
                max_past = max(past_accs)
                forgetting = max_past - current_class_accs[sem]
                forgetting_list.append(forgetting)
        avg_forg = np.mean(forgetting_list) if forgetting_list else 0.0
    else:
        avg_forg = 0.0
    return acc, avg_forg, current_class_accs

# ============================================================
# MASTER EXPERIMENT LOOP: TRACK B (WITH RESUME CHECKPOINTING)
# ============================================================
print("\n" + "="*100)
print(f"🚀 INITIATING MASTER ABLATION STUDY: TRACK B (WITH CHECKPOINTING)")
print("="*100)

CHECKPOINT_FILE = os.path.join(BASE_DIR, "track_b_checkpoint.pth")
TASKS = [f"splitcifar10_{i}" for i in range(NUM_TASKS)]

# Default Starting State
start_exp_idx = 0
start_task_idx = 0
final_results_summary = []

# ==========================================
# 🔄 CHECKPOINT RESUME LOGIC
# ==========================================
if os.path.exists(CHECKPOINT_FILE):
    print(f"🔄 Found existing checkpoint at '{CHECKPOINT_FILE}'. Loading state...")
    checkpoint = torch.load(CHECKPOINT_FILE)
    
    start_exp_idx = checkpoint['exp_idx']
    start_task_idx = checkpoint['task_idx']
    final_results_summary = checkpoint['final_results_summary']
    
    print(f"⏩ Fast-forwarding to Experiment {start_exp_idx+1}/{len(EXPERIMENTS)} | Task {start_task_idx}/{NUM_TASKS}")
else:
    print("🆕 No checkpoint found. Starting fresh from Experiment 1, Task 0.")

experiment_items = list(EXPERIMENTS.items())

for exp_idx in range(start_exp_idx, len(experiment_items)):
    exp_name, exp_config = experiment_items[exp_idx]
    
    print("\n\n" + "#"*100)
    print(f"⚙️  RUNNING EXPERIMENT: {exp_name}")
    print("#"*100)

    # 🚨 HARD RESET FOR NEW EXPERIMENT
    set_seed(42)
    locked_generator = get_locked_generator(42)
    model = CNN(num_classes=2).to(DEVICE)
    memory = MemoryBuffer(max_per_class=400) 
    detector = HypersphereNovelty()
    
    novelty_buffer = []
    semantic_to_internal = {0: 0, 1: 1}
    internal_to_semantic = {0: 0, 1: 1}
    known_classes = 2
    
    task_kca, task_cpr, average_forgetting = [], [], []
    class_accuracies = {}   
    learned_classes_over_time = []
    promoted_classes_log = []
    exp_init_purities = [] 

    # 🔄 IF RESUMING MID-EXPERIMENT, OVERRIDE THE RESET WITH SAVED STATE
    if exp_idx == start_exp_idx and start_task_idx > 0:
        model.load_state_dict(checkpoint['model_state'])
        model.classifier = checkpoint['classifier_module'] 
        memory.data = checkpoint['memory_data']
        detector.mu = checkpoint['detector_mu']
        detector.r = checkpoint['detector_r']
        
        novelty_buffer = checkpoint['novelty_buffer']
        semantic_to_internal = checkpoint['semantic_to_internal']
        internal_to_semantic = checkpoint['internal_to_semantic']
        known_classes = checkpoint['known_classes']
        
        task_kca = checkpoint['task_kca']
        task_cpr = checkpoint['task_cpr']
        average_forgetting = checkpoint['average_forgetting']
        class_accuracies = checkpoint['class_accuracies']
        learned_classes_over_time = checkpoint['learned_classes_over_time']
        promoted_classes_log = checkpoint['promoted_classes_log']
        exp_init_purities = checkpoint['exp_init_purities']
        
        # Advance the random generator state to maintain determinism
        for _ in range(start_task_idx): torch.rand(1, generator=locked_generator)

    # Begin Task Loop
    for t in range(start_task_idx if exp_idx == start_exp_idx else 0, NUM_TASKS):
        task = TASKS[t]
        _, dataset_fn = DATASET_REGISTRY[task]
        train_ds, test_ds, *_ = dataset_fn("./data")
        loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=locked_generator) 

        print(f"\n{'-'*20} 🚀 TASK {t} {'-'*20}")
        all_semantics = []
        for _, y in loader: all_semantics.extend(y.argmax(1).tolist())
        unique_semantics = list(set(all_semantics))
        class_names = [CIFAR10_LABELS[sem] for sem in unique_semantics]
        print(f"📦 [STREAM] Task {t} stream contains class(es): {class_names} (Label IDs: {unique_semantics})")
        print(f"📊 [STREAM] Total images in this chunk: {len(train_ds)}")

        if t == 0:
            train_supervised(model, loader, DataLoader(test_ds, batch_size=128, shuffle=False), t)
            for cls in [0, 1]:
                Xc = torch.cat([x[y.argmax(1) == cls] for x, y in loader])
                memory.build_memory_herding(Xc, cls, model)
            detector.update(memory.get(), model)
            learned_classes_over_time.append({CIFAR10_LABELS[0], CIFAR10_LABELS[1]})
            acc, avg_forg, class_accuracies[t] = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
            task_kca.append(acc); average_forgetting.append(avg_forg)
            print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")
            for sem, c_acc in class_accuracies[t].items():
                print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")
        else:
            model.eval()
            novelty_candidates = []
            novel, false_novel, total = 0, 0, 0
            with torch.no_grad():
                for x, y in loader:
                    _, z = model(x.to(DEVICE))
                    y_labels = y.argmax(1)
                    scores = [detector.score(z[i]).item() for i in range(len(z))]
                    thr = np.percentile(scores, 30)
                    for i in range(len(z)):
                        total += 1
                        if scores[i] > thr: 
                            novelty_candidates.append((scores[i], x[i].cpu(), y_labels[i].item()))
                            novel += 1
                            if y_labels[i].item() in semantic_to_internal: false_novel += 1

            print(f"🔍 [NOVELTY] Flagged Novel: {novel}/{total} | False Novelty: {false_novel}")

            novelty_candidates.sort(reverse=True, key=lambda x: x[0])
            novelty_buffer.extend([(img, y) for _, img, y in novelty_candidates[:TOPK_NOVELTY]])
            novelty_buffer = novelty_buffer[-MAX_NOVELTY_BUFFER:]

            if t % P == 0 and len(novelty_buffer) >= 20: 
                Z = []
                with torch.no_grad():
                    for img, _ in novelty_buffer:
                        _, z = model(img.unsqueeze(0).to(DEVICE))
                        Z.append(z.squeeze().cpu().numpy())

                Z = np.stack(Z)
                exp_type = exp_config["type"]
                print(f"\n🧠 [CLUSTERING] Running {exp_type.upper()} on buffer (Size: {len(novelty_buffer)})...")
                
                labels = np.zeros(len(Z))
                
                # =========================================================
                # 🚀 TRACK B: DYNAMIC CLUSTERING ENGINE SWAP (WITH SAFETY NET)
                # =========================================================
                try:
                    if exp_type == "hdbscan_euclidean":
                        clusterer = hdbscan.HDBSCAN(metric='euclidean', min_cluster_size=20)
                        labels = clusterer.fit_predict(Z)
                        
                    elif exp_type == "hdbscan_cosine":
                        dist_matrix = cosine_distances(Z).astype(np.float64) 
                        clusterer = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=20)
                        labels = clusterer.fit_predict(dist_matrix)
                        
                    elif exp_type == "kmeans_cosine":
                        best_k, best_score, best_labels = 2, -1, np.zeros(len(Z))
                        for k in range(2, min(11, len(Z) // 20 + 2)):
                            lbls = KMeans(n_clusters=k, random_state=42, n_init='auto').fit_predict(Z)
                            if len(set(lbls)) > 1:
                                score = silhouette_score(Z, lbls, metric='cosine')
                                if score > best_score:
                                    best_score, best_k, best_labels = score, k, lbls
                        labels = best_labels
                        print(f"  🤖 Dynamic K-Means settled on k={best_k} (Silhouette: {best_score:.3f})")
                        
                    elif exp_type == "finch_proxy":
                        nn_finder = NearestNeighbors(n_neighbors=2, metric='cosine').fit(Z)
                        _, indices = nn_finder.kneighbors(Z)
                        adj = np.zeros((len(Z), len(Z)))
                        for i in range(len(Z)):
                            adj[i, indices[i, 1]] = 1
                            adj[indices[i, 1], i] = 1
                        _, labels = connected_components(adj, directed=False)
                        
                    elif exp_type == "snn_dbscan":
                        nn_finder = NearestNeighbors(n_neighbors=15, metric='cosine').fit(Z)
                        _, indices = nn_finder.kneighbors(Z)
                        dist_matrix = np.zeros((len(Z), len(Z)))
                        for i in range(len(Z)):
                            for j in range(i+1, len(Z)):
                                shared = len(np.intersect1d(indices[i], indices[j]))
                                dist = 1.0 - (shared / 15.0)
                                dist_matrix[i, j] = dist
                                dist_matrix[j, i] = dist
                        labels = DBSCAN(eps=0.6, min_samples=10, metric='precomputed').fit_predict(dist_matrix)
                        
                    elif exp_type == "agglomerative":
                        clusterer = AgglomerativeClustering(n_clusters=None, distance_threshold=0.35, metric='cosine', linkage='average')
                        labels = clusterer.fit_predict(Z)
                        
                except Exception as e:
                    print(f"  ❌ [ERROR] Clustering engine crashed: {str(e)}")
                    print("  ⚠️ Safely degrading buffer to Global Noise to prevent loop crash.")
                    labels = np.full(len(Z), -1)
                # =========================================================
                
                new_buffer = []
                found, promoted = 0, 0

                for cid in sorted(set(labels)):
                    idxs = np.where(labels == cid)[0].tolist()
                    
                    if cid == -1: 
                        print(f"  🗑️  [CLUSTER -1] NOISE: Found {len(idxs)} noise samples.")
                        for i in idxs: 
                            new_buffer.append(novelty_buffer[i])
                        continue

                    found += 1
                    labels_true = [novelty_buffer[i][1] for i in idxs]
                    sem_label, cnt = Counter(labels_true).most_common(1)[0]
                    purity = cnt / len(labels_true)
                    
                    Xc = torch.stack([novelty_buffer[i][0] for i in idxs])
                    with torch.no_grad(): _, Zc = model(Xc.to(DEVICE))
                    
                    mu = F.normalize(Zc.mean(0), dim=0)
                    n = len(idxs)
                    S_intra = torch.mean(1 - torch.matmul(Zc, mu))
                    S_known = min([1 - torch.dot(mu, detector.mu[k].to(mu.device)) for k in detector.mu]) if len(detector.mu)>0 else torch.tensor(1.0)
                    density = n / (S_intra.item() + 1e-6)
                    margin = S_known - S_intra
                    
                    if sem_label in semantic_to_internal:
                        print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Dominant class '{CIFAR10_LABELS[sem_label]}' already known.")
                        for i in idxs: 
                            new_buffer.append(novelty_buffer[i])
                        continue 

                    req_min_size = exp_config.get("min_cluster_size", 150)

                    cond_intra = S_intra.item() <= ALPHA
                    cond_density = density >= DELTA
                    cond_known = S_known.item() >= BETA
                    cond_margin = margin.item() > -0.10
                    cond_size = n >= req_min_size

                    print(f"\n  📊 [CLUSTER {cid} EVALUATION] Dominant: '{CIFAR10_LABELS[sem_label]}' (Raw Purity: {purity:.2f})")
                    print(f"     ➔ Size:    {n:3d}   (Req: >= {req_min_size:2d})   {'✅' if cond_size else '❌'}")
                    print(f"     ➔ S_intra: {S_intra.item():.3f} (Req: <= {ALPHA:.2f}) {'✅' if cond_intra else '❌'}")
                    print(f"     ➔ S_known: {S_known.item():.3f} (Req: >= {BETA:.2f}) {'✅' if cond_known else '❌'}")
                    print(f"     ➔ Density: {density:.1f} (Req: >= {DELTA})   {'✅' if cond_density else '❌'}")
                    print(f"     ➔ Margin:  {margin.item():.3f} (Req: > -0.10) {'✅' if cond_margin else '❌'}")

                    if cond_intra and cond_density and cond_size and cond_margin and cond_known:
                        promoted += 1
                        new_label = known_classes
                        semantic_to_internal[sem_label] = new_label
                        internal_to_semantic[new_label] = sem_label
                        known_classes += 1
                        
                        exp_init_purities.append(purity)
                        
                        model.expand_head(known_classes) 
                        model.to(DEVICE)
                        
                        print(f"     🎉 [PROMOTION SUCCESS] -> Preparing to Finetune '{CIFAR10_LABELS[sem_label]}'...")
                        finetune(model, memory, Xc, new_label, t, locked_generator)
                        print(f"     ✨ [LEARNED] -> Network successfully adapted to '{CIFAR10_LABELS[sem_label]}'")
                        
                        memory.build_memory_herding(Xc, new_label, model)
                        detector.update(memory.get(), model)
                        promoted_classes_log.append({"task": t, "semantic": CIFAR10_LABELS[sem_label]})
                    else:
                        print(f"     🛑 [PROMOTION FAILED] -> Conditions not met. Retaining samples in buffer.")
                        for i in idxs: 
                            new_buffer.append(novelty_buffer[i])
                        
                novelty_buffer = new_buffer
                task_cpr.append(promoted / max(found, 1))
            else:
                task_cpr.append(0.0)
                
            if t > 0 and t % 13 == 0: 
                novelty_buffer = [] 
                print(f"\n🧹 [BUFFER CLEAR] Task {t} marks the end of a multi-class wave! Flushed buffer.")

            learned_classes_over_time.append(set(learned_classes_over_time[-1]) if len(learned_classes_over_time) > 0 else set())
            for p in promoted_classes_log:
                if p["task"] == t and p["semantic"] not in learned_classes_over_time[-1]:
                    learned_classes_over_time[-1].add(p["semantic"])

            acc, avg_forg, class_accuracies[t] = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
            task_kca.append(acc)
            average_forgetting.append(avg_forg)
            print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f} | Classes: {len(learned_classes_over_time[-1])}")
            
            for sem, c_acc in class_accuracies[t].items():
                print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")

        # ==========================================
        # 💾 SAVE CHECKPOINT AT THE END OF EVERY TASK
        # ==========================================
        next_task = t + 1
        next_exp = exp_idx
        if next_task >= NUM_TASKS:
            next_task = 0
            next_exp = exp_idx + 1

        torch.save({
            'exp_idx': next_exp,
            'task_idx': next_task,
            'model_state': model.state_dict(),
            'classifier_module': model.classifier,
            'memory_data': memory.data,
            'detector_mu': detector.mu,
            'detector_r': detector.r,
            'novelty_buffer': novelty_buffer,
            'semantic_to_internal': semantic_to_internal,
            'internal_to_semantic': internal_to_semantic,
            'known_classes': known_classes,
            'task_kca': task_kca,
            'task_cpr': task_cpr,
            'average_forgetting': average_forgetting,
            'class_accuracies': class_accuracies,
            'learned_classes_over_time': learned_classes_over_time,
            'promoted_classes_log': promoted_classes_log,
            'exp_init_purities': exp_init_purities,
            'final_results_summary': final_results_summary
        }, CHECKPOINT_FILE)
        print(f"💾 Checkpoint saved for Experiment {exp_idx+1}/{len(EXPERIMENTS)}, Task {t}/{NUM_TASKS-1}.")

    # ============================================================
    # STORE EXPERIMENT RESULT
    # ============================================================
    final_accs = class_accuracies[NUM_TASKS - 1]
    avg_raw_pur = np.mean(exp_init_purities) if exp_init_purities else 0.0
    
    row_result = {
        "Experiment": exp_name,
        "Final KCA": f"{task_kca[-1]:.3f}",
        "Avg Forg": f"{average_forgetting[-1]:.3f}",
        "Classes": str(len(learned_classes_over_time[-1])),
        "Raw Pur": f"{avg_raw_pur:.2f}"
    }
    
    for cls_id in range(10): 
        row_result[f"C{cls_id}({CIFAR10_LABELS[cls_id][:3]})"] = f"{final_accs.get(cls_id, 0.0):.2f}"
        
    final_results_summary.append(row_result)
    
    # Reset start_task_idx for the next experiment
    start_task_idx = 0

# ============================================================
# CREATE & SAVE THE MASTER ABLATION TABLE (PNG)
# ============================================================
print("\n" + "="*160)
print("📊 MASTER ABLATION SUMMARY: TRACK B (COSINE CLUSTERING)")
print("="*160)

columns = list(final_results_summary[0].keys())
cell_text = []

header_format = "{:<32} | {:<9} | {:<8} | {:<7} | {:<8} | " + " | ".join([f"{{:<10}}"]*10)
print(header_format.format(*columns))
print("-" * 175)

for res in final_results_summary:
    row = [res[col] for col in columns]
    cell_text.append(row)
    print(header_format.format(*row))

fig, ax = plt.subplots(figsize=(32, len(cell_text) * 0.6 + 2))
ax.axis('off')
ax.axis('tight')

table = ax.table(cellText=cell_text, colLabels=columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 2.0)

for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#27ae60') 
    else:
        if i % 2 == 0: cell.set_facecolor('#f8f9fa')

plt.title("Master Ablation Summary: Track B (Cosine Clustering)", fontsize=18, fontweight='bold', pad=20)
plt.savefig("ablation_track_B_summary.png", bbox_inches='tight', dpi=300)
plt.close()

print("\n✅ Ablation complete. Master table saved to 'ablation_track_B_summary.png'")

### experiment c (Open space losses - forcing the neural network to make room for the unclustered buffer images)

In [ ]:
import os
# 🚨 THESE MUST BE SET BEFORE IMPORTING TORCH
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import random
import traceback
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from torch.utils.data import DataLoader, TensorDataset, Dataset
import hdbscan
import copy
from torchvision import datasets as tv_datasets, transforms
import torchvision.transforms as T  

# LabelBench Imports
from LabelBench.skeleton.dataset_skeleton import datasets as DATASET_REGISTRY
from LabelBench.skeleton.dataset_skeleton import register_dataset, LabelType, TransformDataset
from torchvision.models import resnet18

# ============================================================
# BULLETPROOF REPRODUCIBILITY LOCK
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=False)

def get_locked_generator(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# ============================================================
# STABLE METRIC LEARNING LOSSES
# ============================================================
def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

# ============================================================
# DIRECTORIES & CONFIGURATION 
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
global BASE_DIR
BASE_DIR = f"debug_TrackC_OpenSpace"
os.makedirs(BASE_DIR, exist_ok=True)

# BUFFER CONFIG
P = 1                              
TOPK_NOVELTY = 400              
MAX_NOVELTY_BUFFER = 2500        
STARTING_TTL = 3

# HPT CONFIG
ALPHA = 0.35              
BETA = 0.01                
DELTA = 15                 
EPSILON = 0.00
METHOD = "margin_contrastive"

CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat",
    4: "deer", 5: "dog", 6: "frog", 7: "horse",
    8: "ship", 9: "truck"
}

# ============================================================
# EXPERIMENT TRACK C: OPEN-SPACE LOSS SWEEP
# ============================================================
EXPERIMENTS = {
    "Baseline (No Space Carving)": {
        "loss_type": "baseline"
    },
    "Background Repulsion Loss": {
        "loss_type": "repulsion", "repulsion_margin": 0.10, "loss_weight": 1.0
    },
    "Virtual Placeholder (K+1)": {
        "loss_type": "virtual_class", "loss_weight": 0.5
    },
    "Entropic Open-Space Loss": {
        "loss_type": "entropy", "loss_weight": 0.5
    }
}

# ============================================================
# DYNAMIC 40-TASK DATASET DEFINITION (MULTI-CLASS WAVE LOGIC)
# ============================================================
NUM_TASKS = 40   

class CIFARStream(Dataset):
    def __init__(self, base_ds, indices):
        self.base_ds = base_ds      
        self.indices = indices      

    def __len__(self): return len(self.indices)
    def __getitem__(self, idx): return self.base_ds[self.indices[idx]]

def one_hot(y, n=10): return F.one_hot(torch.tensor(y), num_classes=n).float()

@register_dataset("splitcifar10", LabelType.MULTI_CLASS)
def get_splitcifar10(_): raise RuntimeError("Use splitcifar10_<id>")

base_train_global = tv_datasets.CIFAR10(root="./data", train=True, download=True)
targets = np.array(base_train_global.targets)
class_indices = {c: np.where(targets == c)[0] for c in range(10)}

rng = np.random.default_rng(42)
for c in range(10): rng.shuffle(class_indices[c])

stream_splits = {}
for t in range(1, NUM_TASKS):
    stream_splits[t] = []
    if 1 <= t <= 13: reminders = [0, 1]
    elif 14 <= t <= 26: reminders = [0, 1, 2, 3]
    elif 27 <= t <= 39: reminders = [0, 1, 2, 3, 4, 5, 6]
    else: reminders = []
        
    for c in reminders: stream_splits[t].extend(rng.choice(class_indices[c], 100, replace=False))
        
    if 1 <= t <= 13:
        stream_splits[t].extend(np.array_split(class_indices[2], 13)[t-1])
        stream_splits[t].extend(np.array_split(class_indices[3], 13)[t-1])
    elif 14 <= t <= 26:
        stream_splits[t].extend(np.array_split(class_indices[4], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[5], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[6], 13)[t-14])
    elif 27 <= t <= 39:
        stream_splits[t].extend(np.array_split(class_indices[7], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[8], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[9], 13)[t-27])
        
    rng.shuffle(stream_splits[t])

for split_id in range(NUM_TASKS):
    @register_dataset(f"splitcifar10_{split_id}", LabelType.MULTI_CLASS)
    def _make_split(data_dir, split_id=split_id):
        tf = transforms.Compose([transforms.ToTensor()])
        base_train = tv_datasets.CIFAR10(root=data_dir, train=True, download=True)
        base_test  = tv_datasets.CIFAR10(root=data_dir, train=False, download=True)
        
        if split_id == 0: indices = [i for i,(x,y) in enumerate(base_train) if y in [0,1]]
        else: indices = stream_splits[split_id]

        train_ds = TransformDataset(CIFARStream(base_train, indices), transform=tf, target_transform=lambda y: one_hot(y,10))
        test_ds = TransformDataset(base_test, transform=tf, target_transform=lambda y: one_hot(y,10))
        return train_ds, test_ds, test_ds, None, None, None, 10, [str(i) for i in range(10)]

# ============================================================
# MODEL & TRACKERS
# ============================================================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = resnet18(weights='DEFAULT')
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes, bias=False)
        self.scale = 20.0  

    def expand_head(self, new_classes, new_centroid=None):
        old_w = self.classifier.weight.data.clone()
        old_n = old_w.shape[0]
        new_classifier = nn.Linear(512, new_classes, bias=False).to(old_w.device)
        new_classifier.weight.data[:old_n] = old_w
        if new_centroid is not None:
            new_classifier.weight.data[old_n:] = new_centroid.to(old_w.device)
        self.classifier = new_classifier

    def forward(self, x, labels=None):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = self.embed(z)
        z = F.normalize(z, dim=1)
        W = F.normalize(self.classifier.weight, dim=1)
        cosine = torch.matmul(z, W.t())
        if labels is not None: 
            m = 0.2 
            theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
            target_logits = torch.cos(theta + m)
            one_hot = F.one_hot(labels, num_classes=W.size(0)).float()
            logits = cosine * (1 - one_hot) + target_logits * one_hot
        else:
            logits = cosine
        logits = logits * self.scale
        return logits, z

class MemoryBuffer:
    def __init__(self, max_per_class=400):
        self.data = defaultdict(list)
        self.max_per_class = max_per_class
        self.aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])

    @torch.no_grad()
    def build_memory_herding(self, X_all, y_label, model):
        model.eval()
        logits_all, Z_all = [], []
        for i in range(0, len(X_all), 128):
            batch = X_all[i:i+128].to(DEVICE)
            logits, z = model(batch)
            logits_all.append(logits.cpu())
            Z_all.append(z.cpu())
        logits_all = torch.cat(logits_all)
        Z_all = torch.cat(Z_all)
        class_mean = F.normalize(Z_all.mean(0), dim=0)
        selected_idx = []
        features = Z_all.clone()

        for k in range(min(self.max_per_class, len(X_all))):
            S = Z_all[selected_idx].sum(0) if k > 0 else torch.zeros_like(class_mean)
            target = (k + 1) * class_mean - S
            distances = torch.norm(features - target, dim=1)
            for idx in selected_idx: distances[idx] = float('inf')
            selected_idx.append(distances.argmin().item())

        self.data[int(y_label)] = []
        for idx in selected_idx:
            self.data[int(y_label)].append((X_all[idx].detach().cpu(), logits_all[idx].detach().cpu()))
            
    def get(self): return self.data

    def sample_balanced(self, batch_size, model):
        classes = list(self.data.keys())
        if not classes: return None, None, None
        samples_per_class = max(1, batch_size // len(classes))
        X_mem, Y_mem, L_mem = [], [], []
        current_dim = model.classifier.out_features
        for cls in classes:
            samples = self.data[cls]
            if len(samples) == 0: continue
            replace = len(samples) < samples_per_class
            idx = np.random.choice(len(samples), samples_per_class, replace=replace)
            for i in idx:
                x, logit = samples[i]
                if logit.shape[0] < current_dim:
                    padded = torch.zeros(current_dim)
                    padded[:logit.shape[0]] = logit
                    logit = padded
                X_mem.append(x)
                Y_mem.append(cls)
                L_mem.append(logit)
        if not X_mem: return None, None, None
        X_tensor = torch.stack(X_mem)
        return self.aug(X_tensor), torch.tensor(Y_mem), torch.stack(L_mem)

class HypersphereNovelty:
    def __init__(self, q=0.90):
        self.q = q
        self.mu, self.r = {}, {}

    def update(self, memory, model):
        self.mu, self.r = {}, {}
        for k, X_tuples in memory.items():
            if len(X_tuples) == 0: continue
            X = torch.stack([x for x, _ in X_tuples]).to(DEVICE)
            with torch.no_grad(): _, Z = model(X)
            mu = F.normalize(Z.mean(0), dim=0)
            d = 1 - torch.matmul(Z, mu)
            self.mu[k] = mu
            self.r[k] = torch.quantile(d, self.q)

    def score(self, z):
        if not self.mu: return torch.tensor(0.0)
        return min([(1 - torch.dot(z, self.mu[k].to(z.device))) - self.r[k].to(z.device) for k in self.mu])

# ============================================================
# TRAINING LOOPS
# ============================================================
def train_supervised(model, loader, test_loader, task_id):
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    
    for epoch in range(30):
        for x, y in loader:
            x_device, y_device = x.to(DEVICE), y.argmax(1).to(DEVICE)
            x_aug = aug(x_device)
            logits, z = model(x_aug, y_device)
            z_norm = F.normalize(z, dim=1)
            loss = F.cross_entropy(logits, y_device) + 1.0 * margin_contrastive_loss(z_norm, y_device)
            opt.zero_grad()
            loss.backward()
            opt.step()

def finetune_track_c(model, memory, X_new, new_label, novelty_buffer, task_id, locked_gen, exp_config): 
    old_model = copy.deepcopy(model).eval()
    for p in old_model.parameters(): p.requires_grad = False 
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()
            
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    Y_new = torch.full((len(X_new),), new_label, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_new, Y_new), batch_size=32, shuffle=True, generator=locked_gen)

    # Prepare background noise for Track C loss computation
    X_bg = None
    if exp_config["loss_type"] != "baseline" and len(novelty_buffer) > 0:
        # Sample random noise from the buffer that wasn't promoted
        bg_imgs = [img for img, _ in novelty_buffer]
        if len(bg_imgs) > 0:
            bg_tensor = torch.stack(bg_imgs)
            bg_idx = torch.randperm(len(bg_tensor))[:128] # Keep batch size reasonable
            X_bg = bg_tensor[bg_idx].to(DEVICE)

    for p in model.parameters(): p.requires_grad = True
    opt = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

    for epoch in range(25):
        for xb, yb in loader:
            xb = aug(xb) 
            X_mem, Y_mem, L_mem = memory.sample_balanced(32, model)
            if X_mem is not None:
                xb = torch.cat([xb, X_mem], dim=0)
                yb = torch.cat([yb, Y_mem], dim=0)

            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits_margin, Z = model(xb, yb)
            loss_ce = F.cross_entropy(logits_margin, yb)
            
            if X_mem is not None:
                pure_logits, _ = model(xb) 
                logits_mem = pure_logits[-len(X_mem):]
                loss_der = F.mse_loss(logits_mem, L_mem.to(DEVICE))
            else:
                loss_der = torch.tensor(0.0, device=DEVICE)

            with torch.no_grad(): logits_old, Z_old = old_model(xb)
            loss_feat = (1 - F.cosine_similarity(Z, Z_old)).mean()
            
            loss = loss_ce + 0.5 * loss_der + 1.0 * loss_feat
            
            # =========================================================
            # 🚀 TRACK C: OPEN-SPACE LOSS COMPUTATION
            # =========================================================
            loss_open = torch.tensor(0.0, device=DEVICE)
            
            if X_bg is not None:
                bg_logits, bg_Z = model(aug(X_bg))
                
                if exp_config["loss_type"] == "repulsion":
                    # Push noise away from all known class centroids
                    W = F.normalize(model.classifier.weight, dim=1)
                    sims = torch.matmul(bg_Z, W.t())
                    margin = exp_config.get("repulsion_margin", 0.1)
                    loss_open = F.relu(sims - margin).mean()
                    
                elif exp_config["loss_type"] == "virtual_class":
                    # Treat noise as a temporary unified "unknown" class
                    virtual_logits = torch.zeros((bg_logits.shape[0], bg_logits.shape[1] + 1), device=DEVICE)
                    virtual_logits[:, :-1] = bg_logits
                    # Simple heuristic: max logit of the background minus a margin
                    virtual_logits[:, -1] = bg_logits.max(dim=1)[0] + 1.0 
                    y_virtual = torch.full((bg_logits.shape[0],), bg_logits.shape[1], dtype=torch.long, device=DEVICE)
                    loss_open = F.cross_entropy(virtual_logits, y_virtual)
                    
                elif exp_config["loss_type"] == "entropy":
                    # Force network to output uniform probabilities for noise
                    probs = F.softmax(bg_logits, dim=1)
                    uniform_target = torch.ones_like(probs) / probs.shape[1]
                    # KL Divergence between prediction and uniform distribution
                    loss_open = F.kl_div(torch.log(probs + 1e-7), uniform_target, reduction='batchmean')
                    
            # Add the open space penalty
            total_loss = loss + (exp_config.get("loss_weight", 0.0) * loss_open)
            # =========================================================

            opt.zero_grad()
            total_loss.backward()
            opt.step()
    model.eval()

def evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies):
    correct_per_class = defaultdict(int)
    total_per_class = defaultdict(int)
    model.eval()
    with torch.no_grad():
        for x_test, y_test in DataLoader(test_ds, batch_size=128):
            x_test = x_test.to(DEVICE)
            y_test = y_test.argmax(1).to(DEVICE)
            logits, _ = model(x_test)
            preds = logits.argmax(1)
            for i in range(len(y_test)):
                sem = int(y_test[i])
                if sem in semantic_to_internal:
                    internal_gt = semantic_to_internal[sem]
                    total_per_class[sem] += 1
                    if preds[i].item() == internal_gt:
                        correct_per_class[sem] += 1

    current_class_accs = {}
    for sem in semantic_to_internal.keys():
        current_class_accs[sem] = correct_per_class[sem] / max(total_per_class[sem], 1)
        
    class_accuracies[t] = current_class_accs
    total_correct = sum(correct_per_class.values())
    total_eval = sum(total_per_class.values())
    acc = total_correct / max(total_eval, 1)
    
    if t > 0:
        forgetting_list = []
        for sem in current_class_accs.keys():
            past_accs = [class_accuracies[k].get(sem, None) for k in range(t)]
            past_accs = [a for a in past_accs if a is not None]
            if past_accs:
                max_past = max(past_accs)
                forgetting = max_past - current_class_accs[sem]
                forgetting_list.append(forgetting)
        avg_forg = np.mean(forgetting_list) if forgetting_list else 0.0
    else:
        avg_forg = 0.0
    return acc, avg_forg, current_class_accs

# ============================================================
# MASTER EXPERIMENT LOOP: TRACK C
# ============================================================
print("\n" + "="*100)
print(f"🚀 INITIATING MASTER ABLATION STUDY: TRACK C (OPEN-SPACE LOSS)")
print("="*100)

CHECKPOINT_FILE = os.path.join(BASE_DIR, "track_c_checkpoint.pth")
TASKS = [f"splitcifar10_{i}" for i in range(NUM_TASKS)]

start_exp_idx = 0
start_task_idx = 0
final_results_summary = []

if os.path.exists(CHECKPOINT_FILE):
    print(f"🔄 Found existing checkpoint at '{CHECKPOINT_FILE}'. Loading state...")
    try:
        checkpoint = torch.load(CHECKPOINT_FILE)
        start_exp_idx = checkpoint['exp_idx']
        start_task_idx = checkpoint['task_idx']
        final_results_summary = checkpoint['final_results_summary']
        print(f"⏩ Fast-forwarding to Experiment {start_exp_idx+1}/{len(EXPERIMENTS)} | Task {start_task_idx}/{NUM_TASKS}")
    except Exception as e:
        print(f"⚠️ Checkpoint corrupted. Starting fresh. Error: {e}")

experiment_items = list(EXPERIMENTS.items())

for exp_idx in range(start_exp_idx, len(experiment_items)):
    exp_name, exp_config = experiment_items[exp_idx]
    
    print("\n\n" + "#"*100)
    print(f"⚙️  RUNNING EXPERIMENT: {exp_name}")
    print("#"*100)

    # 🚨 HARD RESET
    set_seed(42)
    locked_generator = get_locked_generator(42)
    model = CNN(num_classes=2).to(DEVICE)
    memory = MemoryBuffer(max_per_class=400) 
    detector = HypersphereNovelty()
    
    novelty_buffer = []
    semantic_to_internal = {0: 0, 1: 1}
    internal_to_semantic = {0: 0, 1: 1}
    known_classes = 2
    
    task_kca, task_cpr, average_forgetting = [], [], []
    class_accuracies = {}   
    learned_classes_over_time = []
    promoted_classes_log = []
    exp_init_purities = []

    if exp_idx == start_exp_idx and start_task_idx > 0:
        model.load_state_dict(checkpoint['model_state'])
        model.classifier = checkpoint['classifier_module'] 
        memory.data = checkpoint['memory_data']
        detector.mu = checkpoint['detector_mu']
        detector.r = checkpoint['detector_r']
        
        novelty_buffer = checkpoint['novelty_buffer']
        semantic_to_internal = checkpoint['semantic_to_internal']
        internal_to_semantic = checkpoint['internal_to_semantic']
        known_classes = checkpoint['known_classes']
        
        task_kca = checkpoint['task_kca']
        task_cpr = checkpoint['task_cpr']
        average_forgetting = checkpoint['average_forgetting']
        class_accuracies = checkpoint['class_accuracies']
        learned_classes_over_time = checkpoint['learned_classes_over_time']
        promoted_classes_log = checkpoint['promoted_classes_log']
        exp_init_purities = checkpoint['exp_init_purities']
        
        for _ in range(start_task_idx): torch.rand(1, generator=locked_generator)

    for t in range(start_task_idx if exp_idx == start_exp_idx else 0, NUM_TASKS):
        task = TASKS[t]
        _, dataset_fn = DATASET_REGISTRY[task]
        train_ds, test_ds, *_ = dataset_fn("./data")
        loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=locked_generator) 

        print(f"\n{'-'*20} 🚀 TASK {t} {'-'*20}")
        
        all_semantics = []
        for _, y in loader: all_semantics.extend(y.argmax(1).tolist())
        unique_semantics = list(set(all_semantics))
        class_names = [CIFAR10_LABELS[sem] for sem in unique_semantics]
        print(f"📦 [STREAM] Task {t} stream contains class(es): {class_names} (Label IDs: {unique_semantics})")
        print(f"📊 [STREAM] Total images in this chunk: {len(train_ds)}")

        if t == 0:
            train_supervised(model, loader, DataLoader(test_ds, batch_size=128, shuffle=False), t)
            for cls in [0, 1]:
                Xc = torch.cat([x[y.argmax(1) == cls] for x, y in loader])
                memory.build_memory_herding(Xc, cls, model)
            detector.update(memory.get(), model)
            learned_classes_over_time.append({CIFAR10_LABELS[0], CIFAR10_LABELS[1]})
            acc, avg_forg, class_accuracies[t] = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
            task_kca.append(acc); average_forgetting.append(avg_forg)
            print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")
            for sem, c_acc in class_accuracies[t].items():
                print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")
            continue

        model.eval()
        novelty_candidates = []
        novel, false_novel, total = 0, 0, 0
        with torch.no_grad():
            for x, y in loader:
                _, z = model(x.to(DEVICE))
                y_labels = y.argmax(1)
                scores = [detector.score(z[i]).item() for i in range(len(z))]
                thr = np.percentile(scores, 30)
                for i in range(len(z)):
                    total += 1
                    if scores[i] > thr: 
                        novelty_candidates.append((scores[i], x[i].cpu(), y_labels[i].item()))
                        novel += 1
                        if y_labels[i].item() in semantic_to_internal: false_novel += 1

        print(f"🔍 [NOVELTY] Flagged Novel: {novel}/{total} | False Novelty: {false_novel}")

        novelty_candidates.sort(reverse=True, key=lambda x: x[0])
        # Note: TTL is entirely disabled for Track C
        novelty_buffer.extend([(img, y) for _, img, y in novelty_candidates[:TOPK_NOVELTY]])
        novelty_buffer = novelty_buffer[-MAX_NOVELTY_BUFFER:]

        if t % P == 0 and len(novelty_buffer) >= 20: 
            Z = []
            with torch.no_grad():
                for img, _ in novelty_buffer:
                    _, z = model(img.unsqueeze(0).to(DEVICE))
                    Z.append(z.squeeze().cpu().numpy())

            Z = np.stack(Z)
            print(f"\n🧠 [CLUSTERING] Running HDBSCAN Euclidean on buffer (Size: {len(novelty_buffer)})...")
            
            try:
                clusterer = hdbscan.HDBSCAN(metric='euclidean', min_cluster_size=20)
                labels = clusterer.fit_predict(Z)
            except Exception as e:
                print(f"  ❌ [ERROR] Clustering engine crashed: {str(e)}")
                labels = np.full(len(Z), -1)
            
            new_buffer = []
            found, promoted = 0, 0

            for cid in sorted(set(labels)):
                idxs = np.where(labels == cid)[0].tolist()
                
                if cid == -1: 
                    print(f"  🗑️  [CLUSTER -1] NOISE: Found {len(idxs)} noise samples.")
                    for i in idxs: new_buffer.append(novelty_buffer[i])
                    continue

                found += 1
                labels_true = [novelty_buffer[i][1] for i in idxs]
                sem_label, cnt = Counter(labels_true).most_common(1)[0]
                purity = cnt / len(labels_true)
                
                Xc = torch.stack([novelty_buffer[i][0] for i in idxs])
                with torch.no_grad(): _, Zc = model(Xc.to(DEVICE))
                
                mu = F.normalize(Zc.mean(0), dim=0)
                n = len(idxs)
                S_intra = torch.mean(1 - torch.matmul(Zc, mu))
                S_known = min([1 - torch.dot(mu, detector.mu[k].to(mu.device)) for k in detector.mu]) if len(detector.mu)>0 else torch.tensor(1.0)
                density = n / (S_intra.item() + 1e-6)
                margin = S_known - S_intra
                
                if sem_label in semantic_to_internal:
                    print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Dominant class '{CIFAR10_LABELS[sem_label]}' already known.")
                    for i in idxs: new_buffer.append(novelty_buffer[i])
                    continue 

                cond_intra = S_intra.item() <= ALPHA
                cond_density = density >= DELTA
                cond_known = S_known.item() >= BETA
                cond_margin = margin.item() > -0.10
                cond_size = n >= 150 # Baseline Size

                print(f"\n  📊 [CLUSTER {cid} EVALUATION] Dominant: '{CIFAR10_LABELS[sem_label]}' (Raw Purity: {purity:.2f})")
                print(f"     ➔ Size:    {n:3d}   (Req: >= 150)   {'✅' if cond_size else '❌'}")
                print(f"     ➔ S_intra: {S_intra.item():.3f} (Req: <= {ALPHA:.2f}) {'✅' if cond_intra else '❌'}")
                print(f"     ➔ S_known: {S_known.item():.3f} (Req: >= {BETA:.2f}) {'✅' if cond_known else '❌'}")
                print(f"     ➔ Density: {density:.1f} (Req: >= {DELTA})   {'✅' if cond_density else '❌'}")
                print(f"     ➔ Margin:  {margin.item():.3f} (Req: > -0.10) {'✅' if cond_margin else '❌'}")

                if cond_intra and cond_density and cond_size and cond_margin and cond_known:
                    promoted += 1
                    new_label = known_classes
                    semantic_to_internal[sem_label] = new_label
                    internal_to_semantic[new_label] = sem_label
                    known_classes += 1
                    
                    exp_init_purities.append(purity)
                    
                    model.expand_head(known_classes) 
                    model.to(DEVICE)
                    
                    print(f"     🎉 [PROMOTION SUCCESS] -> Preparing to Finetune '{CIFAR10_LABELS[sem_label]}'...")
                    
                    # Pass the novelty buffer to finetune for open-space loss computation
                    finetune_track_c(model, memory, Xc, new_label, novelty_buffer, t, locked_generator, exp_config)
                    
                    print(f"     ✨ [LEARNED] -> Network successfully adapted to '{CIFAR10_LABELS[sem_label]}'")
                    
                    memory.build_memory_herding(Xc, new_label, model)
                    detector.update(memory.get(), model)
                    promoted_classes_log.append({"task": t, "semantic": CIFAR10_LABELS[sem_label]})
                else:
                    print(f"     🛑 [PROMOTION FAILED] -> Conditions not met. Retaining samples in buffer.")
                    for i in idxs: new_buffer.append(novelty_buffer[i])
                    
            novelty_buffer = new_buffer
            task_cpr.append(promoted / max(found, 1))
        else:
            task_cpr.append(0.0)
            
        if t > 0 and t % 13 == 0: 
            novelty_buffer = [] 
            print(f"\n🧹 [BUFFER CLEAR] Task {t} marks the end of a multi-class wave! Flushed buffer.")

        learned_classes_over_time.append(set(learned_classes_over_time[-1]) if len(learned_classes_over_time) > 0 else set())
        for p in promoted_classes_log:
            if p["task"] == t and p["semantic"] not in learned_classes_over_time[-1]:
                learned_classes_over_time[-1].add(p["semantic"])

        acc, avg_forg, class_accuracies[t] = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
        task_kca.append(acc)
        average_forgetting.append(avg_forg)
        print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f} | Classes: {len(learned_classes_over_time[-1])}")
        
        for sem, c_acc in class_accuracies[t].items():
            print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")

        # ==========================================
        # 💾 SAVE CHECKPOINT
        # ==========================================
        next_task = t + 1
        next_exp = exp_idx
        if next_task >= NUM_TASKS:
            next_task = 0
            next_exp = exp_idx + 1

        torch.save({
            'exp_idx': next_exp,
            'task_idx': next_task,
            'model_state': model.state_dict(),
            'classifier_module': model.classifier,
            'memory_data': memory.data,
            'detector_mu': detector.mu,
            'detector_r': detector.r,
            'novelty_buffer': novelty_buffer,
            'semantic_to_internal': semantic_to_internal,
            'internal_to_semantic': internal_to_semantic,
            'known_classes': known_classes,
            'task_kca': task_kca,
            'task_cpr': task_cpr,
            'average_forgetting': average_forgetting,
            'class_accuracies': class_accuracies,
            'learned_classes_over_time': learned_classes_over_time,
            'promoted_classes_log': promoted_classes_log,
            'exp_init_purities': exp_init_purities,
            'final_results_summary': final_results_summary
        }, CHECKPOINT_FILE)

    # ============================================================
    # STORE EXPERIMENT RESULT
    # ============================================================
    final_accs = class_accuracies[NUM_TASKS - 1]
    avg_raw_pur = np.mean(exp_init_purities) if exp_init_purities else 0.0
    
    row_result = {
        "Experiment": exp_name,
        "Final KCA": f"{task_kca[-1]:.3f}",
        "Avg Forg": f"{average_forgetting[-1]:.3f}",
        "Classes": str(len(learned_classes_over_time[-1])),
        "Raw Pur": f"{avg_raw_pur:.2f}"
    }
    
    for cls_id in range(10): 
        row_result[f"C{cls_id}({CIFAR10_LABELS[cls_id][:3]})"] = f"{final_accs.get(cls_id, 0.0):.2f}"
        
    final_results_summary.append(row_result)
    start_task_idx = 0

# ============================================================
# CREATE & SAVE THE MASTER ABLATION TABLE (PNG)
# ============================================================
print("\n" + "="*160)
print("📊 MASTER ABLATION SUMMARY: TRACK C (OPEN-SPACE LOSS)")
print("="*160)

columns = list(final_results_summary[0].keys())
cell_text = []

header_format = "{:<32} | {:<9} | {:<8} | {:<7} | {:<8} | " + " | ".join([f"{{:<10}}"]*10)
print(header_format.format(*columns))
print("-" * 175)

for res in final_results_summary:
    row = [res[col] for col in columns]
    cell_text.append(row)
    print(header_format.format(*row))

fig, ax = plt.subplots(figsize=(32, len(cell_text) * 0.6 + 2))
ax.axis('off')
ax.axis('tight')

table = ax.table(cellText=cell_text, colLabels=columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 2.0)

for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#8e44ad') 
    else:
        if i % 2 == 0: cell.set_facecolor('#f8f9fa')

plt.title("Master Ablation Summary: Track C (Open-Space Loss)", fontsize=18, fontweight='bold', pad=20)
plt.savefig("ablation_track_C_summary.png", bbox_inches='tight', dpi=300)
plt.close()

print("\n✅ Ablation complete. Master table saved to 'ablation_track_C_summary.png'")

### c.2 advance open space losses trying

In [ ]:
import os
# 🚨 THESE MUST BE SET BEFORE IMPORTING TORCH
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import random
import traceback
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from torch.utils.data import DataLoader, TensorDataset, Dataset
import hdbscan
import copy
from torchvision import datasets as tv_datasets, transforms
import torchvision.transforms as T  

# LabelBench Imports
from LabelBench.skeleton.dataset_skeleton import datasets as DATASET_REGISTRY
from LabelBench.skeleton.dataset_skeleton import register_dataset, LabelType, TransformDataset
from torchvision.models import resnet18

# ============================================================
# BULLETPROOF REPRODUCIBILITY LOCK
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=False)

def get_locked_generator(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# ============================================================
# STABLE METRIC LEARNING LOSSES
# ============================================================
def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

# ============================================================
# DIRECTORIES & CONFIGURATION 
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
global BASE_DIR
BASE_DIR = f"debug_TrackC_Advanced"
os.makedirs(BASE_DIR, exist_ok=True)

# BUFFER CONFIG
P = 1                              
TOPK_NOVELTY = 400              
MAX_NOVELTY_BUFFER = 2500        
STARTING_TTL = 3

# HPT CONFIG
ALPHA = 0.35              
BETA = 0.01                
DELTA = 15                 
EPSILON = 0.00
METHOD = "margin_contrastive"

CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat",
    4: "deer", 5: "dog", 6: "frog", 7: "horse",
    8: "ship", 9: "truck"
}

# ============================================================
# EXPERIMENT TRACK C: ADVANCED SOTA LOSS SWEEP
# ============================================================
EXPERIMENTS = {
    "Baseline (No Space Carving)": {
        "loss_type": "baseline"
    },
    "Objectosphere (Mag Collapse)": {
        "loss_type": "objectosphere", "loss_weight": 0.05
    },
    "EDL (Dirichlet Evidence)": {
        "loss_type": "edl_evidence", "loss_weight": 0.1
    },
    "Proxy Reciprocal Points (RPL)": {
        "loss_type": "rpl_proxy", "loss_weight": 0.5
    }
}

# ============================================================
# DYNAMIC 40-TASK DATASET DEFINITION (MULTI-CLASS WAVE LOGIC)
# ============================================================
NUM_TASKS = 40   

class CIFARStream(Dataset):
    def __init__(self, base_ds, indices):
        self.base_ds = base_ds      
        self.indices = indices      

    def __len__(self): return len(self.indices)
    def __getitem__(self, idx): return self.base_ds[self.indices[idx]]

def one_hot(y, n=10): return F.one_hot(torch.tensor(y), num_classes=n).float()

@register_dataset("splitcifar10", LabelType.MULTI_CLASS)
def get_splitcifar10(_): raise RuntimeError("Use splitcifar10_<id>")

base_train_global = tv_datasets.CIFAR10(root="./data", train=True, download=True)
targets = np.array(base_train_global.targets)
class_indices = {c: np.where(targets == c)[0] for c in range(10)}

rng = np.random.default_rng(42)
for c in range(10): rng.shuffle(class_indices[c])

stream_splits = {}
for t in range(1, NUM_TASKS):
    stream_splits[t] = []
    if 1 <= t <= 13: reminders = [0, 1]
    elif 14 <= t <= 26: reminders = [0, 1, 2, 3]
    elif 27 <= t <= 39: reminders = [0, 1, 2, 3, 4, 5, 6]
    else: reminders = []
        
    for c in reminders: stream_splits[t].extend(rng.choice(class_indices[c], 100, replace=False))
        
    if 1 <= t <= 13:
        stream_splits[t].extend(np.array_split(class_indices[2], 13)[t-1])
        stream_splits[t].extend(np.array_split(class_indices[3], 13)[t-1])
    elif 14 <= t <= 26:
        stream_splits[t].extend(np.array_split(class_indices[4], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[5], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[6], 13)[t-14])
    elif 27 <= t <= 39:
        stream_splits[t].extend(np.array_split(class_indices[7], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[8], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[9], 13)[t-27])
        
    rng.shuffle(stream_splits[t])

for split_id in range(NUM_TASKS):
    @register_dataset(f"splitcifar10_{split_id}", LabelType.MULTI_CLASS)
    def _make_split(data_dir, split_id=split_id):
        tf = transforms.Compose([transforms.ToTensor()])
        base_train = tv_datasets.CIFAR10(root=data_dir, train=True, download=True)
        base_test  = tv_datasets.CIFAR10(root=data_dir, train=False, download=True)
        
        if split_id == 0: indices = [i for i,(x,y) in enumerate(base_train) if y in [0,1]]
        else: indices = stream_splits[split_id]

        train_ds = TransformDataset(CIFARStream(base_train, indices), transform=tf, target_transform=lambda y: one_hot(y,10))
        test_ds = TransformDataset(base_test, transform=tf, target_transform=lambda y: one_hot(y,10))
        return train_ds, test_ds, test_ds, None, None, None, 10, [str(i) for i in range(10)]

# ============================================================
# MODEL & TRACKERS
# ============================================================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = resnet18(weights='DEFAULT')
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes, bias=False)
        self.scale = 20.0  

    def expand_head(self, new_classes, new_centroid=None):
        old_w = self.classifier.weight.data.clone()
        old_n = old_w.shape[0]
        new_classifier = nn.Linear(512, new_classes, bias=False).to(old_w.device)
        new_classifier.weight.data[:old_n] = old_w
        if new_centroid is not None:
            new_classifier.weight.data[old_n:] = new_centroid.to(old_w.device)
        self.classifier = new_classifier

    def forward(self, x, labels=None):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = self.embed(z)
        z = F.normalize(z, dim=1)
        W = F.normalize(self.classifier.weight, dim=1)
        cosine = torch.matmul(z, W.t())
        if labels is not None: 
            m = 0.2 
            theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
            target_logits = torch.cos(theta + m)
            one_hot = F.one_hot(labels, num_classes=W.size(0)).float()
            logits = cosine * (1 - one_hot) + target_logits * one_hot
        else:
            logits = cosine
        logits = logits * self.scale
        return logits, z

class MemoryBuffer:
    def __init__(self, max_per_class=400):
        self.data = defaultdict(list)
        self.max_per_class = max_per_class
        self.aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])

    @torch.no_grad()
    def build_memory_herding(self, X_all, y_label, model):
        model.eval()
        logits_all, Z_all = [], []
        for i in range(0, len(X_all), 128):
            batch = X_all[i:i+128].to(DEVICE)
            logits, z = model(batch)
            logits_all.append(logits.cpu())
            Z_all.append(z.cpu())
        logits_all = torch.cat(logits_all)
        Z_all = torch.cat(Z_all)
        class_mean = F.normalize(Z_all.mean(0), dim=0)
        selected_idx = []
        features = Z_all.clone()

        for k in range(min(self.max_per_class, len(X_all))):
            S = Z_all[selected_idx].sum(0) if k > 0 else torch.zeros_like(class_mean)
            target = (k + 1) * class_mean - S
            distances = torch.norm(features - target, dim=1)
            for idx in selected_idx: distances[idx] = float('inf')
            selected_idx.append(distances.argmin().item())

        self.data[int(y_label)] = []
        for idx in selected_idx:
            self.data[int(y_label)].append((X_all[idx].detach().cpu(), logits_all[idx].detach().cpu()))
            
    def get(self): return self.data

    def sample_balanced(self, batch_size, model):
        classes = list(self.data.keys())
        if not classes: return None, None, None
        samples_per_class = max(1, batch_size // len(classes))
        X_mem, Y_mem, L_mem = [], [], []
        current_dim = model.classifier.out_features
        for cls in classes:
            samples = self.data[cls]
            if len(samples) == 0: continue
            replace = len(samples) < samples_per_class
            idx = np.random.choice(len(samples), samples_per_class, replace=replace)
            for i in idx:
                x, logit = samples[i]
                if logit.shape[0] < current_dim:
                    padded = torch.zeros(current_dim)
                    padded[:logit.shape[0]] = logit
                    logit = padded
                X_mem.append(x)
                Y_mem.append(cls)
                L_mem.append(logit)
        if not X_mem: return None, None, None
        X_tensor = torch.stack(X_mem)
        return self.aug(X_tensor), torch.tensor(Y_mem), torch.stack(L_mem)

class HypersphereNovelty:
    def __init__(self, q=0.90):
        self.q = q
        self.mu, self.r = {}, {}

    def update(self, memory, model):
        self.mu, self.r = {}, {}
        for k, X_tuples in memory.items():
            if len(X_tuples) == 0: continue
            X = torch.stack([x for x, _ in X_tuples]).to(DEVICE)
            with torch.no_grad(): _, Z = model(X)
            mu = F.normalize(Z.mean(0), dim=0)
            d = 1 - torch.matmul(Z, mu)
            self.mu[k] = mu
            self.r[k] = torch.quantile(d, self.q)

    def score(self, z):
        if not self.mu: return torch.tensor(0.0)
        return min([(1 - torch.dot(z, self.mu[k].to(z.device))) - self.r[k].to(z.device) for k in self.mu])

# ============================================================
# TRAINING LOOPS
# ============================================================
def train_supervised(model, loader, test_loader, task_id):
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    
    for epoch in range(30):
        for x, y in loader:
            x_device, y_device = x.to(DEVICE), y.argmax(1).to(DEVICE)
            x_aug = aug(x_device)
            logits, z = model(x_aug, y_device)
            z_norm = F.normalize(z, dim=1)
            loss = F.cross_entropy(logits, y_device) + 1.0 * margin_contrastive_loss(z_norm, y_device)
            opt.zero_grad()
            loss.backward()
            opt.step()

def finetune_track_c(model, memory, X_new, new_label, novelty_buffer, task_id, locked_gen, exp_config): 
    old_model = copy.deepcopy(model).eval()
    for p in old_model.parameters(): p.requires_grad = False 
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()
            
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    Y_new = torch.full((len(X_new),), new_label, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_new, Y_new), batch_size=32, shuffle=True, generator=locked_gen)

    # Prepare background noise for Advanced Track C loss computation
    X_bg = None
    if exp_config["loss_type"] != "baseline" and len(novelty_buffer) > 0:
        bg_imgs = [img for img, _ in novelty_buffer]
        if len(bg_imgs) > 0:
            bg_tensor = torch.stack(bg_imgs)
            bg_idx = torch.randperm(len(bg_tensor))[:128] # Keep batch size reasonable
            X_bg = bg_tensor[bg_idx].to(DEVICE)

    for p in model.parameters(): p.requires_grad = True
    opt = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

    for epoch in range(25):
        for xb, yb in loader:
            xb = aug(xb) 
            X_mem, Y_mem, L_mem = memory.sample_balanced(32, model)
            if X_mem is not None:
                xb = torch.cat([xb, X_mem], dim=0)
                yb = torch.cat([yb, Y_mem], dim=0)

            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits_margin, Z = model(xb, yb)
            loss_ce = F.cross_entropy(logits_margin, yb)
            
            if X_mem is not None:
                pure_logits, _ = model(xb) 
                logits_mem = pure_logits[-len(X_mem):]
                loss_der = F.mse_loss(logits_mem, L_mem.to(DEVICE))
            else:
                loss_der = torch.tensor(0.0, device=DEVICE)

            with torch.no_grad(): logits_old, Z_old = old_model(xb)
            loss_feat = (1 - F.cosine_similarity(Z, Z_old)).mean()
            
            loss = loss_ce + 0.5 * loss_der + 1.0 * loss_feat
            
            # =========================================================
            # 🚀 TRACK C (ADVANCED): SOTA OPEN-SPACE LOSS COMPUTATION
            # =========================================================
            loss_open = torch.tensor(0.0, device=DEVICE)
            
            if X_bg is not None:
                bg_logits, bg_Z = model(aug(X_bg))
                
                if exp_config["loss_type"] == "objectosphere":
                    # Recompute unnormalized features to penalize magnitude directly
                    with torch.enable_grad():
                        z_unnorm = model.embed(model.encoder(aug(X_bg)).view(X_bg.size(0), -1))
                    # L2 Norm penalty forces features to collapse to the origin (0,0,0...)
                    loss_open = torch.mean(torch.norm(z_unnorm, p=2, dim=1)**2)
                    
                elif exp_config["loss_type"] == "edl_evidence":
                    # Evidential Deep Learning (Dirichlet Prior)
                    # Minimize total evidence (ReLU of logits) for unknown data
                    evidence = F.relu(bg_logits)
                    loss_open = torch.mean(torch.sum(evidence, dim=1))
                    
                elif exp_config["loss_type"] == "rpl_proxy":
                    # Proxy Reciprocal Point Learning (RPL)
                    # Find nearest known class, then push to the exact negative (opposite side of sphere)
                    W = F.normalize(model.classifier.weight, dim=1)
                    sims = torch.matmul(bg_Z, W.t())
                    nearest_classes = torch.argmax(sims, dim=1)
                    reciprocal_targets = -W[nearest_classes] # The anti-class
                    # Minimize cosine distance to the anti-class
                    loss_open = torch.mean(1 - F.cosine_similarity(bg_Z, reciprocal_targets))
                    
            # Add the open space penalty
            total_loss = loss + (exp_config.get("loss_weight", 0.0) * loss_open)
            # =========================================================

            opt.zero_grad()
            total_loss.backward()
            opt.step()
    model.eval()

def evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies):
    correct_per_class = defaultdict(int)
    total_per_class = defaultdict(int)
    model.eval()
    with torch.no_grad():
        for x_test, y_test in DataLoader(test_ds, batch_size=128):
            x_test = x_test.to(DEVICE)
            y_test = y_test.argmax(1).to(DEVICE)
            logits, _ = model(x_test)
            preds = logits.argmax(1)
            for i in range(len(y_test)):
                sem = int(y_test[i])
                if sem in semantic_to_internal:
                    internal_gt = semantic_to_internal[sem]
                    total_per_class[sem] += 1
                    if preds[i].item() == internal_gt:
                        correct_per_class[sem] += 1

    current_class_accs = {}
    for sem in semantic_to_internal.keys():
        current_class_accs[sem] = correct_per_class[sem] / max(total_per_class[sem], 1)
        
    class_accuracies[t] = current_class_accs
    total_correct = sum(correct_per_class.values())
    total_eval = sum(total_per_class.values())
    acc = total_correct / max(total_eval, 1)
    
    if t > 0:
        forgetting_list = []
        for sem in current_class_accs.keys():
            past_accs = [class_accuracies[k].get(sem, None) for k in range(t)]
            past_accs = [a for a in past_accs if a is not None]
            if past_accs:
                max_past = max(past_accs)
                forgetting = max_past - current_class_accs[sem]
                forgetting_list.append(forgetting)
        avg_forg = np.mean(forgetting_list) if forgetting_list else 0.0
    else:
        avg_forg = 0.0
    return acc, avg_forg, current_class_accs

# ============================================================
# MASTER EXPERIMENT LOOP: TRACK C (ADVANCED)
# ============================================================
print("\n" + "="*100)
print(f"🚀 INITIATING MASTER ABLATION STUDY: TRACK C (ADVANCED SOTA LOSS)")
print("="*100)

CHECKPOINT_FILE = os.path.join(BASE_DIR, "track_c_advanced_checkpoint.pth")
TASKS = [f"splitcifar10_{i}" for i in range(NUM_TASKS)]

start_exp_idx = 0
start_task_idx = 0
final_results_summary = []

if os.path.exists(CHECKPOINT_FILE):
    print(f"🔄 Found existing checkpoint at '{CHECKPOINT_FILE}'. Loading state...")
    try:
        checkpoint = torch.load(CHECKPOINT_FILE)
        start_exp_idx = checkpoint['exp_idx']
        start_task_idx = checkpoint['task_idx']
        final_results_summary = checkpoint['final_results_summary']
        print(f"⏩ Fast-forwarding to Experiment {start_exp_idx+1}/{len(EXPERIMENTS)} | Task {start_task_idx}/{NUM_TASKS}")
    except Exception as e:
        print(f"⚠️ Checkpoint corrupted. Starting fresh. Error: {e}")

experiment_items = list(EXPERIMENTS.items())

for exp_idx in range(start_exp_idx, len(experiment_items)):
    exp_name, exp_config = experiment_items[exp_idx]
    
    print("\n\n" + "#"*100)
    print(f"⚙️  RUNNING EXPERIMENT: {exp_name}")
    print("#"*100)

    # 🚨 HARD RESET
    set_seed(42)
    locked_generator = get_locked_generator(42)
    model = CNN(num_classes=2).to(DEVICE)
    memory = MemoryBuffer(max_per_class=400) 
    detector = HypersphereNovelty()
    
    novelty_buffer = []
    semantic_to_internal = {0: 0, 1: 1}
    internal_to_semantic = {0: 0, 1: 1}
    known_classes = 2
    
    task_kca, task_cpr, average_forgetting = [], [], []
    class_accuracies = {}   
    learned_classes_over_time = []
    promoted_classes_log = []
    exp_init_purities = []

    if exp_idx == start_exp_idx and start_task_idx > 0:
        model.load_state_dict(checkpoint['model_state'])
        model.classifier = checkpoint['classifier_module'] 
        memory.data = checkpoint['memory_data']
        detector.mu = checkpoint['detector_mu']
        detector.r = checkpoint['detector_r']
        
        novelty_buffer = checkpoint['novelty_buffer']
        semantic_to_internal = checkpoint['semantic_to_internal']
        internal_to_semantic = checkpoint['internal_to_semantic']
        known_classes = checkpoint['known_classes']
        
        task_kca = checkpoint['task_kca']
        task_cpr = checkpoint['task_cpr']
        average_forgetting = checkpoint['average_forgetting']
        class_accuracies = checkpoint['class_accuracies']
        learned_classes_over_time = checkpoint['learned_classes_over_time']
        promoted_classes_log = checkpoint['promoted_classes_log']
        exp_init_purities = checkpoint['exp_init_purities']
        
        for _ in range(start_task_idx): torch.rand(1, generator=locked_generator)

    for t in range(start_task_idx if exp_idx == start_exp_idx else 0, NUM_TASKS):
        task = TASKS[t]
        _, dataset_fn = DATASET_REGISTRY[task]
        train_ds, test_ds, *_ = dataset_fn("./data")
        loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=locked_generator) 

        print(f"\n{'-'*20} 🚀 TASK {t} {'-'*20}")
        
        all_semantics = []
        for _, y in loader: all_semantics.extend(y.argmax(1).tolist())
        unique_semantics = list(set(all_semantics))
        class_names = [CIFAR10_LABELS[sem] for sem in unique_semantics]
        print(f"📦 [STREAM] Task {t} stream contains class(es): {class_names} (Label IDs: {unique_semantics})")
        print(f"📊 [STREAM] Total images in this chunk: {len(train_ds)}")

        if t == 0:
            train_supervised(model, loader, DataLoader(test_ds, batch_size=128, shuffle=False), t)
            for cls in [0, 1]:
                Xc = torch.cat([x[y.argmax(1) == cls] for x, y in loader])
                memory.build_memory_herding(Xc, cls, model)
            detector.update(memory.get(), model)
            learned_classes_over_time.append({CIFAR10_LABELS[0], CIFAR10_LABELS[1]})
            acc, avg_forg, class_accuracies[t] = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
            task_kca.append(acc); average_forgetting.append(avg_forg)
            print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")
            for sem, c_acc in class_accuracies[t].items():
                print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")
            continue

        model.eval()
        novelty_candidates = []
        novel, false_novel, total = 0, 0, 0
        with torch.no_grad():
            for x, y in loader:
                _, z = model(x.to(DEVICE))
                y_labels = y.argmax(1)
                scores = [detector.score(z[i]).item() for i in range(len(z))]
                thr = np.percentile(scores, 30)
                for i in range(len(z)):
                    total += 1
                    if scores[i] > thr: 
                        novelty_candidates.append((scores[i], x[i].cpu(), y_labels[i].item()))
                        novel += 1
                        if y_labels[i].item() in semantic_to_internal: false_novel += 1

        print(f"🔍 [NOVELTY] Flagged Novel: {novel}/{total} | False Novelty: {false_novel}")

        novelty_candidates.sort(reverse=True, key=lambda x: x[0])
        # Note: TTL is entirely disabled for Track C
        novelty_buffer.extend([(img, y) for _, img, y in novelty_candidates[:TOPK_NOVELTY]])
        novelty_buffer = novelty_buffer[-MAX_NOVELTY_BUFFER:]

        if t % P == 0 and len(novelty_buffer) >= 20: 
            Z = []
            with torch.no_grad():
                for img, _ in novelty_buffer:
                    _, z = model(img.unsqueeze(0).to(DEVICE))
                    Z.append(z.squeeze().cpu().numpy())

            Z = np.stack(Z)
            print(f"\n🧠 [CLUSTERING] Running HDBSCAN Euclidean on buffer (Size: {len(novelty_buffer)})...")
            
            try:
                clusterer = hdbscan.HDBSCAN(metric='euclidean', min_cluster_size=20)
                labels = clusterer.fit_predict(Z)
            except Exception as e:
                print(f"  ❌ [ERROR] Clustering engine crashed: {str(e)}")
                labels = np.full(len(Z), -1)
            
            new_buffer = []
            found, promoted = 0, 0

            for cid in sorted(set(labels)):
                idxs = np.where(labels == cid)[0].tolist()
                
                if cid == -1: 
                    print(f"  🗑️  [CLUSTER -1] NOISE: Found {len(idxs)} noise samples.")
                    for i in idxs: new_buffer.append(novelty_buffer[i])
                    continue

                found += 1
                labels_true = [novelty_buffer[i][1] for i in idxs]
                sem_label, cnt = Counter(labels_true).most_common(1)[0]
                purity = cnt / len(labels_true)
                
                Xc = torch.stack([novelty_buffer[i][0] for i in idxs])
                with torch.no_grad(): _, Zc = model(Xc.to(DEVICE))
                
                mu = F.normalize(Zc.mean(0), dim=0)
                n = len(idxs)
                S_intra = torch.mean(1 - torch.matmul(Zc, mu))
                S_known = min([1 - torch.dot(mu, detector.mu[k].to(mu.device)) for k in detector.mu]) if len(detector.mu)>0 else torch.tensor(1.0)
                density = n / (S_intra.item() + 1e-6)
                margin = S_known - S_intra
                
                if sem_label in semantic_to_internal:
                    print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Dominant class '{CIFAR10_LABELS[sem_label]}' already known.")
                    for i in idxs: new_buffer.append(novelty_buffer[i])
                    continue 

                cond_intra = S_intra.item() <= ALPHA
                cond_density = density >= DELTA
                cond_known = S_known.item() >= BETA
                cond_margin = margin.item() > -0.10
                cond_size = n >= 150 # Baseline Size

                print(f"\n  📊 [CLUSTER {cid} EVALUATION] Dominant: '{CIFAR10_LABELS[sem_label]}' (Raw Purity: {purity:.2f})")
                print(f"     ➔ Size:    {n:3d}   (Req: >= 150)   {'✅' if cond_size else '❌'}")
                print(f"     ➔ S_intra: {S_intra.item():.3f} (Req: <= {ALPHA:.2f}) {'✅' if cond_intra else '❌'}")
                print(f"     ➔ S_known: {S_known.item():.3f} (Req: >= {BETA:.2f}) {'✅' if cond_known else '❌'}")
                print(f"     ➔ Density: {density:.1f} (Req: >= {DELTA})   {'✅' if cond_density else '❌'}")
                print(f"     ➔ Margin:  {margin.item():.3f} (Req: > -0.10) {'✅' if cond_margin else '❌'}")

                if cond_intra and cond_density and cond_size and cond_margin and cond_known:
                    promoted += 1
                    new_label = known_classes
                    semantic_to_internal[sem_label] = new_label
                    internal_to_semantic[new_label] = sem_label
                    known_classes += 1
                    
                    exp_init_purities.append(purity)
                    
                    model.expand_head(known_classes) 
                    model.to(DEVICE)
                    
                    print(f"     🎉 [PROMOTION SUCCESS] -> Preparing to Finetune '{CIFAR10_LABELS[sem_label]}'...")
                    
                    # Pass the novelty buffer to finetune for open-space loss computation
                    finetune_track_c(model, memory, Xc, new_label, novelty_buffer, t, locked_generator, exp_config)
                    
                    print(f"     ✨ [LEARNED] -> Network successfully adapted to '{CIFAR10_LABELS[sem_label]}'")
                    
                    memory.build_memory_herding(Xc, new_label, model)
                    detector.update(memory.get(), model)
                    promoted_classes_log.append({"task": t, "semantic": CIFAR10_LABELS[sem_label]})
                else:
                    print(f"     🛑 [PROMOTION FAILED] -> Conditions not met. Retaining samples in buffer.")
                    for i in idxs: new_buffer.append(novelty_buffer[i])
                    
            novelty_buffer = new_buffer
            task_cpr.append(promoted / max(found, 1))
        else:
            task_cpr.append(0.0)
            
        if t > 0 and t % 13 == 0: 
            novelty_buffer = [] 
            print(f"\n🧹 [BUFFER CLEAR] Task {t} marks the end of a multi-class wave! Flushed buffer.")

        learned_classes_over_time.append(set(learned_classes_over_time[-1]) if len(learned_classes_over_time) > 0 else set())
        for p in promoted_classes_log:
            if p["task"] == t and p["semantic"] not in learned_classes_over_time[-1]:
                learned_classes_over_time[-1].add(p["semantic"])

        acc, avg_forg, class_accuracies[t] = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
        task_kca.append(acc)
        average_forgetting.append(avg_forg)
        print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f} | Classes: {len(learned_classes_over_time[-1])}")
        
        for sem, c_acc in class_accuracies[t].items():
            print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")

        # ==========================================
        # 💾 SAVE CHECKPOINT
        # ==========================================
        next_task = t + 1
        next_exp = exp_idx
        if next_task >= NUM_TASKS:
            next_task = 0
            next_exp = exp_idx + 1

        torch.save({
            'exp_idx': next_exp,
            'task_idx': next_task,
            'model_state': model.state_dict(),
            'classifier_module': model.classifier,
            'memory_data': memory.data,
            'detector_mu': detector.mu,
            'detector_r': detector.r,
            'novelty_buffer': novelty_buffer,
            'semantic_to_internal': semantic_to_internal,
            'internal_to_semantic': internal_to_semantic,
            'known_classes': known_classes,
            'task_kca': task_kca,
            'task_cpr': task_cpr,
            'average_forgetting': average_forgetting,
            'class_accuracies': class_accuracies,
            'learned_classes_over_time': learned_classes_over_time,
            'promoted_classes_log': promoted_classes_log,
            'exp_init_purities': exp_init_purities,
            'final_results_summary': final_results_summary
        }, CHECKPOINT_FILE)

    # ============================================================
    # STORE EXPERIMENT RESULT
    # ============================================================
    final_accs = class_accuracies[NUM_TASKS - 1]
    avg_raw_pur = np.mean(exp_init_purities) if exp_init_purities else 0.0
    
    row_result = {
        "Experiment": exp_name,
        "Final KCA": f"{task_kca[-1]:.3f}",
        "Avg Forg": f"{average_forgetting[-1]:.3f}",
        "Classes": str(len(learned_classes_over_time[-1])),
        "Raw Pur": f"{avg_raw_pur:.2f}"
    }
    
    for cls_id in range(10): 
        row_result[f"C{cls_id}({CIFAR10_LABELS[cls_id][:3]})"] = f"{final_accs.get(cls_id, 0.0):.2f}"
        
    final_results_summary.append(row_result)
    start_task_idx = 0

# ============================================================
# CREATE & SAVE THE MASTER ABLATION TABLE (PNG)
# ============================================================
print("\n" + "="*160)
print("📊 MASTER ABLATION SUMMARY: TRACK C (ADVANCED SOTA LOSS)")
print("="*160)

columns = list(final_results_summary[0].keys())
cell_text = []

header_format = "{:<32} | {:<9} | {:<8} | {:<7} | {:<8} | " + " | ".join([f"{{:<10}}"]*10)
print(header_format.format(*columns))
print("-" * 175)

for res in final_results_summary:
    row = [res[col] for col in columns]
    cell_text.append(row)
    print(header_format.format(*row))

fig, ax = plt.subplots(figsize=(32, len(cell_text) * 0.6 + 2))
ax.axis('off')
ax.axis('tight')

table = ax.table(cellText=cell_text, colLabels=columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 2.0)

for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#d35400') # Dark purple/orange for Advanced C 
    else:
        if i % 2 == 0: cell.set_facecolor('#f8f9fa')

plt.title("Master Ablation Summary: Track C (Advanced SOTA Loss)", fontsize=18, fontweight='bold', pad=20)
plt.savefig("ablation_track_C_advanced_summary.png", bbox_inches='tight', dpi=300)
plt.close()

print("\n✅ Ablation complete. Master table saved to 'ablation_track_C_advanced_summary.png'")

### combining the three experiments (mainly B, C and Lwf loss)

In [ ]:
import os
# 🚨 THESE MUST BE SET BEFORE IMPORTING TORCH
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import random
import traceback
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from torch.utils.data import DataLoader, TensorDataset, Dataset
import hdbscan
import copy
from torchvision import datasets as tv_datasets, transforms
import torchvision.transforms as T  

# Track B Specific Imports
from sklearn.cluster import AgglomerativeClustering

# LabelBench Imports
from LabelBench.skeleton.dataset_skeleton import datasets as DATASET_REGISTRY
from LabelBench.skeleton.dataset_skeleton import register_dataset, LabelType, TransformDataset
from torchvision.models import resnet18

# ============================================================
# BULLETPROOF REPRODUCIBILITY LOCK
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=False)

def get_locked_generator(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# ============================================================
# STABLE METRIC LEARNING LOSSES
# ============================================================
def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

# ============================================================
# DIRECTORIES & CONFIGURATION 
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
global BASE_DIR
BASE_DIR = f"debug_Final_Voltron_Pipeline"
os.makedirs(BASE_DIR, exist_ok=True)

# BUFFER CONFIG
P = 1                              
TOPK_NOVELTY = 400              
MAX_NOVELTY_BUFFER = 2500        

# HPT CONFIG
ALPHA = 0.35              
BETA = 0.01                
DELTA = 15                 

CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat",
    4: "deer", 5: "dog", 6: "frog", 7: "horse",
    8: "ship", 9: "truck"
}

# ============================================================
# FINAL EXPERIMENT SWEEP: VOLTRON ONLY
# ============================================================
EXPERIMENTS = {
    "The Voltron Pipeline (Agglo + Entropy + LwF + Hard Lock)": {
        "pipeline": "voltron"
    }
}

# ============================================================
# DYNAMIC 40-TASK DATASET DEFINITION (MULTI-CLASS WAVE LOGIC)
# ============================================================
NUM_TASKS = 40   

class CIFARStream(Dataset):
    def __init__(self, base_ds, indices):
        self.base_ds = base_ds      
        self.indices = indices      

    def __len__(self): return len(self.indices)
    def __getitem__(self, idx): return self.base_ds[self.indices[idx]]

def one_hot(y, n=10): return F.one_hot(torch.tensor(y), num_classes=n).float()

@register_dataset("splitcifar10", LabelType.MULTI_CLASS)
def get_splitcifar10(_): raise RuntimeError("Use splitcifar10_<id>")

base_train_global = tv_datasets.CIFAR10(root="./data", train=True, download=True)
targets = np.array(base_train_global.targets)
class_indices = {c: np.where(targets == c)[0] for c in range(10)}

rng = np.random.default_rng(42)
for c in range(10): rng.shuffle(class_indices[c])

stream_splits = {}
for t in range(1, NUM_TASKS):
    stream_splits[t] = []
    if 1 <= t <= 13: reminders = [0, 1]
    elif 14 <= t <= 26: reminders = [0, 1, 2, 3]
    elif 27 <= t <= 39: reminders = [0, 1, 2, 3, 4, 5, 6]
    else: reminders = []
        
    for c in reminders: stream_splits[t].extend(rng.choice(class_indices[c], 100, replace=False))
        
    if 1 <= t <= 13:
        stream_splits[t].extend(np.array_split(class_indices[2], 13)[t-1])
        stream_splits[t].extend(np.array_split(class_indices[3], 13)[t-1])
    elif 14 <= t <= 26:
        stream_splits[t].extend(np.array_split(class_indices[4], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[5], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[6], 13)[t-14])
    elif 27 <= t <= 39:
        stream_splits[t].extend(np.array_split(class_indices[7], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[8], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[9], 13)[t-27])
        
    rng.shuffle(stream_splits[t])

for split_id in range(NUM_TASKS):
    @register_dataset(f"splitcifar10_{split_id}", LabelType.MULTI_CLASS)
    def _make_split(data_dir, split_id=split_id):
        tf = transforms.Compose([transforms.ToTensor()])
        base_train = tv_datasets.CIFAR10(root=data_dir, train=True, download=True)
        base_test  = tv_datasets.CIFAR10(root=data_dir, train=False, download=True)
        
        if split_id == 0: indices = [i for i,(x,y) in enumerate(base_train) if y in [0,1]]
        else: indices = stream_splits[split_id]

        train_ds = TransformDataset(CIFARStream(base_train, indices), transform=tf, target_transform=lambda y: one_hot(y,10))
        test_ds = TransformDataset(base_test, transform=tf, target_transform=lambda y: one_hot(y,10))
        return train_ds, test_ds, test_ds, None, None, None, 10, [str(i) for i in range(10)]

# ============================================================
# MODEL & TRACKERS
# ============================================================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = resnet18(weights='DEFAULT')
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes, bias=False)
        self.scale = 20.0  

    def expand_head(self, new_classes, new_centroid=None):
        old_w = self.classifier.weight.data.clone()
        old_n = old_w.shape[0]
        new_classifier = nn.Linear(512, new_classes, bias=False).to(old_w.device)
        new_classifier.weight.data[:old_n] = old_w
        if new_centroid is not None:
            new_classifier.weight.data[old_n:] = new_centroid.to(old_w.device)
        self.classifier = new_classifier

    def forward(self, x, labels=None):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = self.embed(z)
        z = F.normalize(z, dim=1)
        W = F.normalize(self.classifier.weight, dim=1)
        cosine = torch.matmul(z, W.t())
        if labels is not None: 
            m = 0.2 
            theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
            target_logits = torch.cos(theta + m)
            one_hot = F.one_hot(labels, num_classes=W.size(0)).float()
            logits = cosine * (1 - one_hot) + target_logits * one_hot
        else:
            logits = cosine
        logits = logits * self.scale
        return logits, z

class MemoryBuffer:
    def __init__(self, max_per_class=400):
        self.data = defaultdict(list)
        self.max_per_class = max_per_class
        self.aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])

    @torch.no_grad()
    def build_memory_herding(self, X_all, y_label, model):
        model.eval()
        logits_all, Z_all = [], []
        for i in range(0, len(X_all), 128):
            batch = X_all[i:i+128].to(DEVICE)
            logits, z = model(batch)
            logits_all.append(logits.cpu())
            Z_all.append(z.cpu())
        logits_all = torch.cat(logits_all)
        Z_all = torch.cat(Z_all)
        class_mean = F.normalize(Z_all.mean(0), dim=0)
        selected_idx = []
        features = Z_all.clone()

        for k in range(min(self.max_per_class, len(X_all))):
            S = Z_all[selected_idx].sum(0) if k > 0 else torch.zeros_like(class_mean)
            target = (k + 1) * class_mean - S
            distances = torch.norm(features - target, dim=1)
            for idx in selected_idx: distances[idx] = float('inf')
            selected_idx.append(distances.argmin().item())

        self.data[int(y_label)] = []
        for idx in selected_idx:
            self.data[int(y_label)].append((X_all[idx].detach().cpu(), logits_all[idx].detach().cpu()))
            
    def get(self): return self.data

    def sample_balanced(self, batch_size, model):
        classes = list(self.data.keys())
        if not classes: return None, None, None
        samples_per_class = max(1, batch_size // len(classes))
        X_mem, Y_mem, L_mem = [], [], []
        current_dim = model.classifier.out_features
        for cls in classes:
            samples = self.data[cls]
            if len(samples) == 0: continue
            replace = len(samples) < samples_per_class
            idx = np.random.choice(len(samples), samples_per_class, replace=replace)
            for i in idx:
                x, logit = samples[i]
                if logit.shape[0] < current_dim:
                    padded = torch.zeros(current_dim)
                    padded[:logit.shape[0]] = logit
                    logit = padded
                X_mem.append(x)
                Y_mem.append(cls)
                L_mem.append(logit)
        if not X_mem: return None, None, None
        X_tensor = torch.stack(X_mem)
        return self.aug(X_tensor), torch.tensor(Y_mem), torch.stack(L_mem)

class HypersphereNovelty:
    def __init__(self, q=0.90):
        self.q = q
        self.mu, self.r = {}, {}

    def update(self, memory, model):
        self.mu, self.r = {}, {}
        for k, X_tuples in memory.items():
            if len(X_tuples) == 0: continue
            X = torch.stack([x for x, _ in X_tuples]).to(DEVICE)
            with torch.no_grad(): _, Z = model(X)
            mu = F.normalize(Z.mean(0), dim=0)
            d = 1 - torch.matmul(Z, mu)
            self.mu[k] = mu
            self.r[k] = torch.quantile(d, self.q)

    def score(self, z):
        if not self.mu: return torch.tensor(0.0)
        return min([(1 - torch.dot(z, self.mu[k].to(z.device))) - self.r[k].to(z.device) for k in self.mu])

# ============================================================
# TRAINING LOOPS
# ============================================================
def train_supervised(model, loader, test_loader, task_id):
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    
    for epoch in range(30):
        for x, y in loader:
            x_device, y_device = x.to(DEVICE), y.argmax(1).to(DEVICE)
            x_aug = aug(x_device)
            logits, z = model(x_aug, y_device)
            z_norm = F.normalize(z, dim=1)
            loss = F.cross_entropy(logits, y_device) + 1.0 * margin_contrastive_loss(z_norm, y_device)
            opt.zero_grad()
            loss.backward()
            opt.step()

# 🛠️ FIXED: Added bg_buffer parameter to prevent friendly fire
def finetune_master(model, memory, X_new, new_label, bg_buffer, task_id, locked_gen, exp_config): 
    old_model = copy.deepcopy(model).eval()
    for p in old_model.parameters(): p.requires_grad = False 
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()
            
    aug_func = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    Y_new = torch.full((len(X_new),), new_label, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_new, Y_new), batch_size=32, shuffle=True, generator=locked_gen)

    # 🚀 VOLTRON: Prepare ONLY true background noise for Open-Space Carving
    X_bg = None
    if exp_config["pipeline"] == "voltron" and len(bg_buffer) > 0:
        bg_imgs = [img for img, _ in bg_buffer]
        if len(bg_imgs) > 0:
            bg_tensor = torch.stack(bg_imgs)
            bg_idx = torch.randperm(len(bg_tensor))[:128]
            X_bg = bg_tensor[bg_idx].to(DEVICE)

    for p in model.parameters(): p.requires_grad = True
    opt = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

    for epoch in range(25):
        for xb, yb in loader:
            xb = aug_func(xb) 
            X_mem, Y_mem, L_mem = memory.sample_balanced(32, model)
            if X_mem is not None:
                xb = torch.cat([xb, X_mem], dim=0)
                yb = torch.cat([yb, Y_mem], dim=0)

            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits_margin, Z = model(xb, yb)
            loss_ce = F.cross_entropy(logits_margin, yb)

            if exp_config["pipeline"] == "voltron":
                # 1. The Classifier Shield (LwF) - 🛠️ FIXED: Distill ONLY the old classes
                loss_lwf = torch.tensor(0.0, device=DEVICE)
                if X_mem is not None:
                    with torch.no_grad():
                        old_logits_mem, _ = old_model(X_mem.to(DEVICE))
                    pure_logits, _ = model(xb)
                    new_logits_mem = pure_logits[-len(X_mem):]
                    
                    # Slice both logits up to the new_label index to ignore the randomly initialized new class
                    new_logits_sliced = new_logits_mem[:, :new_label]
                    old_logits_sliced = old_logits_mem[:, :new_label]
                    
                    temp = 2.0
                    loss_lwf = nn.KLDivLoss(reduction='batchmean')(
                        F.log_softmax(new_logits_sliced / temp, dim=1),
                        F.softmax(old_logits_sliced / temp, dim=1)
                    ) * (temp * temp)
                    
                # 2. The Foundation Anchor (Feature Consistency)
                with torch.no_grad(): logits_old, Z_old = old_model(xb)
                loss_feat = (1 - F.cosine_similarity(Z, Z_old)).mean()

                # 3. The Space Carver (Entropic Loss)
                loss_open = torch.tensor(0.0, device=DEVICE)
                if X_bg is not None:
                    bg_logits, _ = model(aug_func(X_bg))
                    probs = F.softmax(bg_logits, dim=1)
                    uniform_target = torch.ones_like(probs) / probs.shape[1]
                    loss_open = F.kl_div(torch.log(probs + 1e-7), uniform_target, reduction='batchmean')
                
                # Voltron Synthesis
                total_loss = loss_ce + 2.0 * loss_lwf + 1.0 * loss_feat + 0.5 * loss_open

            opt.zero_grad()
            total_loss.backward()
            opt.step()
            
            # 🛡️ THE HARD WEIGHT LOCK - 🛠️ FIXED: Lock ONLY the old classes
            if exp_config["pipeline"] == "voltron":
                old_w = old_model.classifier.weight.data
                model.classifier.weight.data[:new_label] = old_w[:new_label]

    model.eval()

def evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies):
    correct_per_class = defaultdict(int)
    total_per_class = defaultdict(int)
    model.eval()
    with torch.no_grad():
        for x_test, y_test in DataLoader(test_ds, batch_size=128):
            x_test = x_test.to(DEVICE)
            y_test = y_test.argmax(1).to(DEVICE)
            logits, _ = model(x_test)
            preds = logits.argmax(1)
            for i in range(len(y_test)):
                sem = int(y_test[i])
                if sem in semantic_to_internal:
                    internal_gt = semantic_to_internal[sem]
                    total_per_class[sem] += 1
                    if preds[i].item() == internal_gt:
                        correct_per_class[sem] += 1

    current_class_accs = {}
    for sem in semantic_to_internal.keys():
        current_class_accs[sem] = correct_per_class[sem] / max(total_per_class[sem], 1)
        
    class_accuracies[t] = current_class_accs
    total_correct = sum(correct_per_class.values())
    total_eval = sum(total_per_class.values())
    acc = total_correct / max(total_eval, 1)
    
    if t > 0:
        forgetting_list = []
        for sem in current_class_accs.keys():
            past_accs = [class_accuracies[k].get(sem, None) for k in range(t)]
            past_accs = [a for a in past_accs if a is not None]
            if past_accs:
                max_past = max(past_accs)
                forgetting = max_past - current_class_accs[sem]
                forgetting_list.append(forgetting)
        avg_forg = np.mean(forgetting_list) if forgetting_list else 0.0
    else:
        avg_forg = 0.0
    return acc, avg_forg, current_class_accs

# ============================================================
# MASTER EXPERIMENT LOOP: FINAL INTEGRATION
# ============================================================
print("\n" + "="*100)
print(f"🚀 INITIATING MASTER ABLATION STUDY: FINAL VOLTRON PIPELINE")
print("="*100)

# Renamed checkpoint file to prevent loading the broken run
CHECKPOINT_FILE = os.path.join(BASE_DIR, "voltron_fixed_checkpoint.pth")
TASKS = [f"splitcifar10_{i}" for i in range(NUM_TASKS)]

start_exp_idx = 0
start_task_idx = 0
final_results_summary = []

if os.path.exists(CHECKPOINT_FILE):
    print(f"🔄 Found existing checkpoint at '{CHECKPOINT_FILE}'. Loading state...")
    try:
        checkpoint = torch.load(CHECKPOINT_FILE)
        start_exp_idx = checkpoint['exp_idx']
        start_task_idx = checkpoint['task_idx']
        final_results_summary = checkpoint['final_results_summary']
        print(f"⏩ Fast-forwarding to Experiment {start_exp_idx+1}/{len(EXPERIMENTS)} | Task {start_task_idx}/{NUM_TASKS}")
    except Exception as e:
        print(f"⚠️ Checkpoint corrupted. Starting fresh. Error: {e}")

experiment_items = list(EXPERIMENTS.items())

for exp_idx in range(start_exp_idx, len(experiment_items)):
    exp_name, exp_config = experiment_items[exp_idx]
    
    print("\n\n" + "#"*100)
    print(f"⚙️  RUNNING EXPERIMENT: {exp_name}")
    print("#"*100)

    # 🚨 HARD RESET
    set_seed(42)
    locked_generator = get_locked_generator(42)
    model = CNN(num_classes=2).to(DEVICE)
    memory = MemoryBuffer(max_per_class=400) 
    detector = HypersphereNovelty()
    
    novelty_buffer = []
    semantic_to_internal = {0: 0, 1: 1}
    internal_to_semantic = {0: 0, 1: 1}
    known_classes = 2
    
    task_kca, task_cpr, average_forgetting = [], [], []
    class_accuracies = {}   
    learned_classes_over_time = []
    promoted_classes_log = []
    exp_init_purities = []

    if exp_idx == start_exp_idx and start_task_idx > 0:
        model.load_state_dict(checkpoint['model_state'])
        model.classifier = checkpoint['classifier_module'] 
        memory.data = checkpoint['memory_data']
        detector.mu = checkpoint['detector_mu']
        detector.r = checkpoint['detector_r']
        
        novelty_buffer = checkpoint['novelty_buffer']
        semantic_to_internal = checkpoint['semantic_to_internal']
        internal_to_semantic = checkpoint['internal_to_semantic']
        known_classes = checkpoint['known_classes']
        
        task_kca = checkpoint['task_kca']
        task_cpr = checkpoint['task_cpr']
        average_forgetting = checkpoint['average_forgetting']
        class_accuracies = checkpoint['class_accuracies']
        learned_classes_over_time = checkpoint['learned_classes_over_time']
        promoted_classes_log = checkpoint['promoted_classes_log']
        exp_init_purities = checkpoint['exp_init_purities']
        
        for _ in range(start_task_idx): torch.rand(1, generator=locked_generator)

    for t in range(start_task_idx if exp_idx == start_exp_idx else 0, NUM_TASKS):
        task = TASKS[t]
        _, dataset_fn = DATASET_REGISTRY[task]
        train_ds, test_ds, *_ = dataset_fn("./data")
        loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=locked_generator) 

        print(f"\n{'-'*20} 🚀 TASK {t} {'-'*20}")
        
        all_semantics = []
        for _, y in loader: all_semantics.extend(y.argmax(1).tolist())
        unique_semantics = list(set(all_semantics))
        class_names = [CIFAR10_LABELS[sem] for sem in unique_semantics]
        print(f"📦 [STREAM] Task {t} stream contains class(es): {class_names} (Label IDs: {unique_semantics})")
        print(f"📊 [STREAM] Total images in this chunk: {len(train_ds)}")

        if t == 0:
            train_supervised(model, loader, DataLoader(test_ds, batch_size=128, shuffle=False), t)
            for cls in [0, 1]:
                Xc = torch.cat([x[y.argmax(1) == cls] for x, y in loader])
                memory.build_memory_herding(Xc, cls, model)
            detector.update(memory.get(), model)
            learned_classes_over_time.append({CIFAR10_LABELS[0], CIFAR10_LABELS[1]})
            acc, avg_forg, class_accuracies[t] = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
            task_kca.append(acc); average_forgetting.append(avg_forg)
            print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")
            for sem, c_acc in class_accuracies[t].items():
                print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")
            continue

        model.eval()
        novelty_candidates = []
        novel, false_novel, total = 0, 0, 0
        with torch.no_grad():
            for x, y in loader:
                _, z = model(x.to(DEVICE))
                y_labels = y.argmax(1)
                scores = [detector.score(z[i]).item() for i in range(len(z))]
                thr = np.percentile(scores, 30)
                for i in range(len(z)):
                    total += 1
                    if scores[i] > thr: 
                        novelty_candidates.append((scores[i], x[i].cpu(), y_labels[i].item()))
                        novel += 1
                        if y_labels[i].item() in semantic_to_internal: false_novel += 1

        print(f"🔍 [NOVELTY] Flagged Novel: {novel}/{total} | False Novelty: {false_novel}")

        novelty_candidates.sort(reverse=True, key=lambda x: x[0])
        novelty_buffer.extend([(img, y) for _, img, y in novelty_candidates[:TOPK_NOVELTY]])
        novelty_buffer = novelty_buffer[-MAX_NOVELTY_BUFFER:]

        if t % P == 0 and len(novelty_buffer) >= 20: 
            Z = []
            with torch.no_grad():
                for img, _ in novelty_buffer:
                    _, z = model(img.unsqueeze(0).to(DEVICE))
                    Z.append(z.squeeze().cpu().numpy())

            Z = np.stack(Z)
            
            try:
                if exp_config["pipeline"] == "baseline":
                    print(f"\n🧠 [CLUSTERING] Running HDBSCAN Euclidean on buffer (Size: {len(novelty_buffer)})...")
                    clusterer = hdbscan.HDBSCAN(metric='euclidean', min_cluster_size=20)
                    labels = clusterer.fit_predict(Z)
                else:
                    print(f"\n🧠 [CLUSTERING] Running VOLTRON Agglomerative Cosine on buffer (Size: {len(novelty_buffer)})...")
                    clusterer = AgglomerativeClustering(n_clusters=None, distance_threshold=0.35, metric='cosine', linkage='average')
                    labels = clusterer.fit_predict(Z)
            except Exception as e:
                print(f"  ❌ [ERROR] Clustering engine crashed: {str(e)}")
                labels = np.full(len(Z), -1)
            
            new_buffer = []
            found, promoted = 0, 0

            for cid in sorted(set(labels)):
                idxs = np.where(labels == cid)[0].tolist()
                
                if cid == -1: 
                    print(f"  🗑️  [CLUSTER -1] NOISE: Found {len(idxs)} noise samples.")
                    for i in idxs: new_buffer.append(novelty_buffer[i])
                    continue

                found += 1
                labels_true = [novelty_buffer[i][1] for i in idxs]
                sem_label, cnt = Counter(labels_true).most_common(1)[0]
                purity = cnt / len(labels_true)
                
                Xc = torch.stack([novelty_buffer[i][0] for i in idxs])
                with torch.no_grad(): _, Zc = model(Xc.to(DEVICE))
                
                mu = F.normalize(Zc.mean(0), dim=0)
                n = len(idxs)
                S_intra = torch.mean(1 - torch.matmul(Zc, mu))
                S_known = min([1 - torch.dot(mu, detector.mu[k].to(mu.device)) for k in detector.mu]) if len(detector.mu)>0 else torch.tensor(1.0)
                density = n / (S_intra.item() + 1e-6)
                margin = S_known - S_intra
                
                if sem_label in semantic_to_internal:
                    print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Dominant class '{CIFAR10_LABELS[sem_label]}' already known.")
                    for i in idxs: new_buffer.append(novelty_buffer[i])
                    continue 

                cond_intra = S_intra.item() <= ALPHA
                cond_density = density >= DELTA
                cond_known = S_known.item() >= BETA
                cond_margin = margin.item() > -0.10
                cond_size = n >= 150 # Baseline Size

                print(f"\n  📊 [CLUSTER {cid} EVALUATION] Dominant: '{CIFAR10_LABELS[sem_label]}' (Raw Purity: {purity:.2f})")
                print(f"     ➔ Size:    {n:3d}   (Req: >= 150)   {'✅' if cond_size else '❌'}")
                print(f"     ➔ S_intra: {S_intra.item():.3f} (Req: <= {ALPHA:.2f}) {'✅' if cond_intra else '❌'}")
                print(f"     ➔ S_known: {S_known.item():.3f} (Req: >= {BETA:.2f}) {'✅' if cond_known else '❌'}")
                print(f"     ➔ Density: {density:.1f} (Req: >= {DELTA})   {'✅' if cond_density else '❌'}")
                print(f"     ➔ Margin:  {margin.item():.3f} (Req: > -0.10) {'✅' if cond_margin else '❌'}")

                if cond_intra and cond_density and cond_size and cond_margin and cond_known:
                    promoted += 1
                    new_label = known_classes
                    semantic_to_internal[sem_label] = new_label
                    internal_to_semantic[new_label] = sem_label
                    known_classes += 1
                    
                    exp_init_purities.append(purity)
                    
                    model.expand_head(known_classes) 
                    model.to(DEVICE)
                    
                    print(f"     🎉 [PROMOTION SUCCESS] -> Preparing to Finetune '{CIFAR10_LABELS[sem_label]}'...")
                    
                    # 🛠️ FIXED: Pass ONLY the true background noise to the finetuner
                    idx_set = set(idxs)
                    bg_buffer = [novelty_buffer[i] for i in range(len(novelty_buffer)) if i not in idx_set]
                    finetune_master(model, memory, Xc, new_label, bg_buffer, t, locked_generator, exp_config)
                    
                    print(f"     ✨ [LEARNED] -> Network successfully adapted to '{CIFAR10_LABELS[sem_label]}'")
                    
                    memory.build_memory_herding(Xc, new_label, model)
                    detector.update(memory.get(), model)
                    promoted_classes_log.append({"task": t, "semantic": CIFAR10_LABELS[sem_label]})
                else:
                    print(f"     🛑 [PROMOTION FAILED] -> Conditions not met. Retaining samples in buffer.")
                    for i in idxs: new_buffer.append(novelty_buffer[i])
                    
            novelty_buffer = new_buffer
            task_cpr.append(promoted / max(found, 1))
        else:
            task_cpr.append(0.0)
            
        if t > 0 and t % 13 == 0: 
            novelty_buffer = [] 
            print(f"\n🧹 [BUFFER CLEAR] Task {t} marks the end of a multi-class wave! Flushed buffer.")

        learned_classes_over_time.append(set(learned_classes_over_time[-1]) if len(learned_classes_over_time) > 0 else set())
        for p in promoted_classes_log:
            if p["task"] == t and p["semantic"] not in learned_classes_over_time[-1]:
                learned_classes_over_time[-1].add(p["semantic"])

        acc, avg_forg, class_accuracies[t] = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
        task_kca.append(acc)
        average_forgetting.append(avg_forg)
        print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f} | Classes: {len(learned_classes_over_time[-1])}")
        
        for sem, c_acc in class_accuracies[t].items():
            print(f"   ➔ Class '{CIFAR10_LABELS[sem]}' Acc: {c_acc:.3f}")

        # ==========================================
        # 💾 SAVE CHECKPOINT
        # ==========================================
        next_task = t + 1
        next_exp = exp_idx
        if next_task >= NUM_TASKS:
            next_task = 0
            next_exp = exp_idx + 1

        torch.save({
            'exp_idx': next_exp,
            'task_idx': next_task,
            'model_state': model.state_dict(),
            'classifier_module': model.classifier,
            'memory_data': memory.data,
            'detector_mu': detector.mu,
            'detector_r': detector.r,
            'novelty_buffer': novelty_buffer,
            'semantic_to_internal': semantic_to_internal,
            'internal_to_semantic': internal_to_semantic,
            'known_classes': known_classes,
            'task_kca': task_kca,
            'task_cpr': task_cpr,
            'average_forgetting': average_forgetting,
            'class_accuracies': class_accuracies,
            'learned_classes_over_time': learned_classes_over_time,
            'promoted_classes_log': promoted_classes_log,
            'exp_init_purities': exp_init_purities,
            'final_results_summary': final_results_summary
        }, CHECKPOINT_FILE)

    # ============================================================
    # STORE EXPERIMENT RESULT
    # ============================================================
    final_accs = class_accuracies[NUM_TASKS - 1]
    avg_raw_pur = np.mean(exp_init_purities) if exp_init_purities else 0.0
    
    row_result = {
        "Experiment": exp_name,
        "Final KCA": f"{task_kca[-1]:.3f}",
        "Avg Forg": f"{average_forgetting[-1]:.3f}",
        "Classes": str(len(learned_classes_over_time[-1])),
        "Raw Pur": f"{avg_raw_pur:.2f}"
    }
    
    for cls_id in range(10): 
        row_result[f"C{cls_id}({CIFAR10_LABELS[cls_id][:3]})"] = f"{final_accs.get(cls_id, 0.0):.2f}"
        
    final_results_summary.append(row_result)
    start_task_idx = 0

# ============================================================
# CREATE & SAVE THE MASTER ABLATION TABLE (PNG)
# ============================================================
print("\n" + "="*160)
print("📊 MASTER ABLATION SUMMARY: FINAL VOLTRON PIPELINE")
print("="*160)

columns = list(final_results_summary[0].keys())
cell_text = []

header_format = "{:<55} | {:<9} | {:<8} | {:<7} | {:<8} | " + " | ".join([f"{{:<10}}"]*10)
print(header_format.format(*columns))
print("-" * 195)

for res in final_results_summary:
    row = [res[col] for col in columns]
    cell_text.append(row)
    print(header_format.format(*row))

fig, ax = plt.subplots(figsize=(34, len(cell_text) * 0.6 + 2))
ax.axis('off')
ax.axis('tight')

table = ax.table(cellText=cell_text, colLabels=columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 2.0)

for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#d35400')
    else:
        if i % 2 == 0: cell.set_facecolor('#f8f9fa')

plt.title("Master Ablation Summary: Final Voltron Pipeline", fontsize=18, fontweight='bold', pad=20)
plt.savefig("ablation_final_voltron_summary.png", bbox_inches='tight', dpi=300)
plt.close()

print("\n✅ Ablation complete. Master table saved to 'ablation_final_voltron_summary.png'")

### Path A (Mnn pruning, micro clustering, deferred promotion)

In [5]:
import os
# 🚨 THESE MUST BE SET BEFORE IMPORTING TORCH
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import scipy.spatial.distance as sp_dist
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from collections import defaultdict, Counter
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torchvision.transforms as T
from torchvision.models import resnet18
import hdbscan
import copy
from torchvision import datasets as tv_datasets, transforms

# LabelBench Imports
from LabelBench.skeleton.dataset_skeleton import datasets as DATASET_REGISTRY
from LabelBench.skeleton.dataset_skeleton import register_dataset, LabelType, TransformDataset

# ============================================================
# BULLETPROOF REPRODUCIBILITY LOCK
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=False)

def get_locked_generator(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# ============================================================
# STABLE METRIC LEARNING LOSSES
# ============================================================
def soft_center_loss(z, labels, margin=0.15):
    loss = torch.tensor(0.0, device=z.device)
    for c in torch.unique(labels):
        mask = (labels == c)
        if mask.sum() > 1:
            centroid = z[mask].mean(dim=0, keepdim=True).detach()
            centroid = F.normalize(centroid, dim=1)
            dist = 1 - F.cosine_similarity(z[mask], centroid)
            loss += F.relu(dist - margin).mean()
    return loss

def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

def triplet_loss_batch(z, labels, margin=1.0):
    loss = torch.tensor(0.0, device=z.device)
    valid_triplets = 0
    for i in range(len(z)):
        anchor = z[i]
        pos_mask = (labels == labels[i])
        pos_mask[i] = False
        neg_mask = (labels != labels[i])
        if pos_mask.sum() > 0 and neg_mask.sum() > 0:
            pos = z[pos_mask][torch.randint(0, pos_mask.sum(), (1,))].squeeze(0)
            neg = z[neg_mask][torch.randint(0, neg_mask.sum(), (1,))].squeeze(0)
            loss += F.triplet_margin_loss(anchor.unsqueeze(0), pos.unsqueeze(0), neg.unsqueeze(0), margin=margin)
            valid_triplets += 1
    if valid_triplets > 0:
        loss /= valid_triplets
    return loss

# ============================================================
# DIRECTORIES & CONFIGURATION 
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
global BASE_DIR

P = 1                              
TOPK_NOVELTY = 400              
MAX_NOVELTY_BUFFER = 2500        

# 🚀 PATH B CONFIGURATION (MNN + MICRO-CLUSTERS + STASH + WARMUP)
ALPHA = 0.35              
BETA = 0.01                
DELTA = 15                 
MIN_CLUSTER_SIZE = 20      
PROMOTION_THRESHOLD = 150  
MNN_K = 10                 
MERGE_THRESHOLD = 0.90     
EPSILON = 0.00
METHOD = "margin_contrastive"

CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat",
    4: "deer", 5: "dog", 6: "frog", 7: "horse",
    8: "ship", 9: "truck"
}

# ============================================================
# DYNAMIC 40-TASK DATASET DEFINITION (MULTI-CLASS WAVE LOGIC)
# ============================================================
NUM_TASKS = 40   

class CIFARStream(Dataset):
    def __init__(self, base_ds, indices):
        self.base_ds = base_ds      
        self.indices = indices      

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.base_ds[self.indices[idx]]
        return x, y

def one_hot(y, n=10):
    return F.one_hot(torch.tensor(y), num_classes=n).float()

@register_dataset("splitcifar10", LabelType.MULTI_CLASS)
def get_splitcifar10(_):
    raise RuntimeError("Use splitcifar10_<id>")

base_train_global = tv_datasets.CIFAR10(root="./data", train=True, download=True)
targets = np.array(base_train_global.targets)
class_indices = {c: np.where(targets == c)[0] for c in range(10)}

rng = np.random.default_rng(42)
for c in range(10):
    rng.shuffle(class_indices[c])

stream_splits = {}
for t in range(1, NUM_TASKS):
    stream_splits[t] = []
    
    if 1 <= t <= 13: reminders = [0, 1]
    elif 14 <= t <= 26: reminders = [0, 1, 2, 3]
    elif 27 <= t <= 39: reminders = [0, 1, 2, 3, 4, 5, 6]
    else: reminders = []
        
    for c in reminders:
        stream_splits[t].extend(rng.choice(class_indices[c], 100, replace=False))
        
    if 1 <= t <= 13:
        stream_splits[t].extend(np.array_split(class_indices[2], 13)[t-1])
        stream_splits[t].extend(np.array_split(class_indices[3], 13)[t-1])
    elif 14 <= t <= 26:
        stream_splits[t].extend(np.array_split(class_indices[4], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[5], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[6], 13)[t-14])
    elif 27 <= t <= 39:
        stream_splits[t].extend(np.array_split(class_indices[7], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[8], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[9], 13)[t-27])
        
    rng.shuffle(stream_splits[t])

for split_id in range(NUM_TASKS):
    @register_dataset(f"splitcifar10_{split_id}", LabelType.MULTI_CLASS)
    def _make_split(data_dir, split_id=split_id):
        tf = transforms.Compose([transforms.ToTensor()])
        base_train = tv_datasets.CIFAR10(root=data_dir, train=True, download=True)
        base_test  = tv_datasets.CIFAR10(root=data_dir, train=False, download=True)
        
        if split_id == 0: indices = [i for i,(x,y) in enumerate(base_train) if y in [0,1]]
        else: indices = stream_splits[split_id]

        train_ds = CIFARStream(base_train, indices)
        train_ds = TransformDataset(train_ds, transform=tf, target_transform=lambda y: one_hot(y,10))
        test_ds = TransformDataset(base_test, transform=tf, target_transform=lambda y: one_hot(y,10))

        return train_ds, test_ds, test_ds, None, None, None, 10, [str(i) for i in range(10)]

# ============================================================
# MODEL DEFINITION
# ============================================================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = resnet18(weights='DEFAULT')
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes, bias=False)
        self.scale = 20.0  

    def expand_head(self, new_classes):
        old_w = self.classifier.weight.data.clone()
        old_n = old_w.shape[0]
        new_classifier = nn.Linear(512, new_classes, bias=False).to(old_w.device)
        new_classifier.weight.data[:old_n] = old_w
        self.classifier = new_classifier

    def forward(self, x, labels=None):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = self.embed(z)
        z = F.normalize(z, dim=1)
        W = F.normalize(self.classifier.weight, dim=1)
        cosine = torch.matmul(z, W.t())
        if labels is not None: 
            m = 0.2 
            theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
            target_logits = torch.cos(theta + m)
            one_hot = F.one_hot(labels, num_classes=W.size(0)).float()
            logits = cosine * (1 - one_hot) + target_logits * one_hot
        else:
            logits = cosine
        logits = logits * self.scale
        return logits, z

# ============================================================
# TRACKERS & MEMORY BUFFER
# ============================================================
class MemoryBuffer:
    def __init__(self, max_per_class=400):
        self.data = defaultdict(list)
        self.max_per_class = max_per_class
        self.aug = T.Compose([
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip()
        ])

    @torch.no_grad()
    def build_memory_herding(self, X_all, y_label, model):
        model.eval()
        logits_all, Z_all = [], []
        for i in range(0, len(X_all), 128):
            batch = X_all[i:i+128].to(DEVICE)
            logits, z = model(batch)
            logits_all.append(logits.cpu())
            Z_all.append(z.cpu())
        logits_all = torch.cat(logits_all)
        Z_all = torch.cat(Z_all)
        class_mean = F.normalize(Z_all.mean(0), dim=0)
        selected_idx = []
        features = Z_all.clone()

        for k in range(min(self.max_per_class, len(X_all))):
            if k > 0:
                S = Z_all[selected_idx].sum(0)
            else:
                S = torch.zeros_like(class_mean)
            target = (k + 1) * class_mean - S
            distances = torch.norm(features - target, dim=1)
            for idx in selected_idx: distances[idx] = float('inf')
            best = distances.argmin().item()
            selected_idx.append(best)

        self.data[int(y_label)] = []
        for idx in selected_idx:
            self.data[int(y_label)].append((X_all[idx].detach().cpu(), logits_all[idx].detach().cpu()))
    
    def get(self): return self.data

    def sample_balanced(self, batch_size, model):
        classes = list(self.data.keys())
        if not classes: return None, None, None
        samples_per_class = max(1, batch_size // len(classes))
        X_mem, Y_mem, L_mem = [], [], []
        current_dim = model.classifier.out_features
        for cls in classes:
            samples = self.data[cls]
            if len(samples) == 0: continue
            replace = len(samples) < samples_per_class
            idx = np.random.choice(len(samples), samples_per_class, replace=replace)
            for i in idx:
                x, logit = samples[i]
                if logit.shape[0] < current_dim:
                    padded = torch.zeros(current_dim)
                    padded[:logit.shape[0]] = logit
                    logit = padded
                X_mem.append(x)
                Y_mem.append(cls)
                L_mem.append(logit)
        if not X_mem: return None, None, None
        X_tensor = torch.stack(X_mem)
        X_tensor = self.aug(X_tensor) 
        return X_tensor, torch.tensor(Y_mem), torch.stack(L_mem)

class HypersphereNovelty:
    def __init__(self, q=0.90):
        self.q = q
        self.mu, self.r = {}, {}

    def update(self, memory, model):
        self.mu, self.r = {}, {}
        for k, X_tuples in memory.items():
            if len(X_tuples) == 0: continue
            X = torch.stack([x for x, _ in X_tuples]).to(DEVICE)
            with torch.no_grad(): 
                _, Z = model(X)
            mu = F.normalize(Z.mean(0), dim=0)
            d = 1 - torch.matmul(Z, mu)
            self.mu[k] = mu
            self.r[k] = torch.quantile(d, self.q)

    def score(self, z):
        if not self.mu: return torch.tensor(0.0)
        return min([(1 - torch.dot(z, self.mu[k].to(z.device))) - self.r[k].to(z.device) for k in self.mu])

# ============================================================
# TRAINING LOOPS
# ============================================================
def train_supervised(model, loader, test_loader, task_id, method="baseline_ce"):
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    
    for epoch in range(30):
        for x, y in loader:
            x_device, y_device = x.to(DEVICE), y.argmax(1).to(DEVICE)
            x_aug = aug(x_device)
            logits, z = model(x_aug, y_device)
            loss_ce = F.cross_entropy(logits, y_device)
            z_norm = F.normalize(z, dim=1)
            
            if method == "baseline_ce": loss = loss_ce
            elif method == "soft_center_loss": loss = loss_ce + 1.0 * soft_center_loss(z_norm, y_device, margin=0.15)
            elif method == "margin_contrastive": loss = loss_ce + 1.0 * margin_contrastive_loss(z_norm, y_device)
            elif method == "triplet": loss = loss_ce + 1.0 * triplet_loss_batch(z_norm, y_device, margin=1.0)
                
            opt.zero_grad()
            loss.backward()
            opt.step()

def finetune(model, memory, X_new, new_label, task_id, test_loader, locked_gen): 
    old_model = copy.deepcopy(model).eval()
    for p in old_model.parameters(): p.requires_grad = False 
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()
            
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    Y_new = torch.full((len(X_new),), new_label, dtype=torch.long)
    
    loader = DataLoader(TensorDataset(X_new, Y_new), batch_size=32, shuffle=True, generator=locked_gen)
    opt = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

    for epoch in range(25):
        for xb, yb in loader:
            xb = aug(xb) 
            X_mem, Y_mem, L_mem = memory.sample_balanced(32, model)
            if X_mem is not None:
                xb = torch.cat([xb, X_mem], dim=0)
                yb = torch.cat([yb, Y_mem], dim=0)

            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits_margin, Z = model(xb, yb)
            loss_ce = F.cross_entropy(logits_margin, yb)
            
            if X_mem is not None:
                pure_logits, _ = model(xb) 
                logits_mem = pure_logits[-len(X_mem):]
                loss_der = F.mse_loss(logits_mem, L_mem.to(DEVICE))
            else:
                loss_der = torch.tensor(0.0, device=DEVICE)

            with torch.no_grad(): 
                logits_old, Z_old = old_model(xb)

            loss_feat = (1 - F.cosine_similarity(Z, Z_old)).mean()
            loss = loss_ce + 0.5 * loss_der + 1.0 * loss_feat
            
            opt.zero_grad()
            loss.backward()
            opt.step()
    model.eval()

def evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies):
    correct_per_class = defaultdict(int)
    total_per_class = defaultdict(int)
    model.eval()
    with torch.no_grad():
        for x_test, y_test in DataLoader(test_ds, batch_size=128):
            x_test = x_test.to(DEVICE)
            y_test = y_test.argmax(1).to(DEVICE)
            logits, _ = model(x_test)
            preds = logits.argmax(1)
            
            for i in range(len(y_test)):
                sem = int(y_test[i])
                if sem in semantic_to_internal:
                    internal_gt = semantic_to_internal[sem]
                    total_per_class[sem] += 1
                    if preds[i].item() == internal_gt:
                        correct_per_class[sem] += 1

    current_class_accs = {}
    for sem in semantic_to_internal.keys():
        current_class_accs[sem] = correct_per_class[sem] / max(total_per_class[sem], 1)
        
    class_accuracies[t] = current_class_accs
    total_correct = sum(correct_per_class.values())
    total_eval = sum(total_per_class.values())
    acc = total_correct / max(total_eval, 1)
    
    if t > 0:
        forgetting_list = []
        for sem in current_class_accs.keys():
            past_accs = [class_accuracies[k].get(sem, None) for k in range(t)]
            past_accs = [a for a in past_accs if a is not None]
            if past_accs:
                max_past = max(past_accs)
                forgetting = max_past - current_class_accs[sem]
                forgetting_list.append(forgetting)
        avg_forg = np.mean(forgetting_list) if forgetting_list else 0.0
    else:
        avg_forg = 0.0
        
    return acc, avg_forg, current_class_accs

# ============================================================
# GRAND SINGLE RUN: PATH B (WARMUP + MNN + MICRO-CLUSTER)
# ============================================================

print("\n" + "="*80)
print(f"🚀 RUNNING PATH B PIPELINE: [Pseudo-Label Warmup | Memory Anchor | VIP Stash]")
print("="*80)

set_seed(42)
locked_generator = get_locked_generator(42)

BASE_DIR = f"debug_PathB_{METHOD}"
os.makedirs(BASE_DIR, exist_ok=True)

task_kca, task_cpr = [], []
average_forgetting = [] 
class_accuracies = {}   
learned_classes_over_time = []
promoted_classes_log = []

TASKS = [f"splitcifar10_{i}" for i in range(NUM_TASKS)]
model = CNN(num_classes=2).to(DEVICE)
memory = MemoryBuffer(max_per_class=400) 
detector = HypersphereNovelty()
novelty_buffer = []
candidate_stash = [] 

semantic_to_internal = {0: 0, 1: 1}
internal_to_semantic = {0: 0, 1: 1}
known_classes = 2

for t, task in enumerate(TASKS):
    _, dataset_fn = DATASET_REGISTRY[task]
    train_ds, test_ds, *_ = dataset_fn("./data")
    loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=locked_generator) 

    print(f"\n{'='*20} 🚀 TASK {t} {'='*20}")
    all_semantics = []
    for _, y in loader: all_semantics.extend(y.argmax(1).tolist())
    unique_semantics = list(set(all_semantics))
    class_names = [CIFAR10_LABELS[sem] for sem in unique_semantics]
    print(f"📦 [STREAM] Task {t} stream contains class(es): {class_names} (Label IDs: {unique_semantics})")
    
    if t == 0:
        print(f"🎓 [INIT] Running initialization on Task 0...")
        train_supervised(model, loader, DataLoader(test_ds, batch_size=128, shuffle=False), t, method=METHOD)
        for cls in [0, 1]:
            Xc = torch.cat([x[y.argmax(1) == cls] for x, y in loader])
            memory.build_memory_herding(Xc, cls, model)
        detector.update(memory.get(), model)
        learned_classes_over_time.append({CIFAR10_LABELS[0], CIFAR10_LABELS[1]})
        
        acc, avg_forg, current_class_accs = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
        task_kca.append(acc)
        average_forgetting.append(avg_forg)
        print(f"📈 [METRIC] Known-Class Acc: {acc:.3f}")
        continue

    model.eval()
    novelty_candidates = []
    novel, false_novel, total = 0, 0, 0
    
    with torch.no_grad():
        for x, y in loader:
            _, z = model(x.to(DEVICE))
            y_labels = y.argmax(1)
            scores = [detector.score(z[i]).item() for i in range(len(z))]
            thr = np.percentile(scores, 30)
            for i in range(len(z)):
                total += 1
                if scores[i] > thr:
                    novelty_candidates.append((scores[i], x[i].cpu(), y_labels[i].item()))
                    novel += 1
                    if y_labels[i].item() in semantic_to_internal: false_novel += 1

    novelty_candidates.sort(reverse=True, key=lambda x: x[0])
    novelty_buffer.extend([(img, y) for _, img, y in novelty_candidates[:TOPK_NOVELTY]])
    novelty_buffer = novelty_buffer[-MAX_NOVELTY_BUFFER:]

    if t % P == 0 and len(novelty_buffer) >= MNN_K + 1: 
        # ============================================================
        # 🚀 PATH B: PSEUDO-LABEL WARMUP + MEMORY ANCHORING
        # ============================================================
        print(f"     🔥 [WARMUP] Applying Pseudo-Label Contrastive & Memory Anchoring...")
        model.eval()
        
        # 1. Feature Extraction for K-Means Pseudo-Labeling
        with torch.no_grad():
            Z_buffer = []
            for img, _ in novelty_buffer:
                _, z = model(img.unsqueeze(0).to(DEVICE))
                Z_buffer.append(z.squeeze().cpu().numpy())
            Z_buffer = np.stack(Z_buffer)
            
        # 2. Run K-Means to find dense, tight pseudo-clusters
        kmeans = KMeans(n_clusters=10, random_state=42, n_init=10).fit(Z_buffer)
        pseudo_labels = kmeans.labels_
        
        X_buffer = torch.stack([img for img, _ in novelty_buffer])
        Y_pseudo = torch.tensor(pseudo_labels, dtype=torch.long)
        warmup_loader = DataLoader(TensorDataset(X_buffer, Y_pseudo), batch_size=32, shuffle=True)
        
        model.train()
        warmup_opt = torch.optim.Adam(model.parameters(), lr=1e-4)
        aug_view = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ColorJitter(0.4, 0.4, 0.4, 0.1)])

        for warmup_epoch in range(5):
            for xb, yb_pseudo in warmup_loader:
                xb = aug_view(xb).to(DEVICE)
                yb_pseudo = yb_pseudo.to(DEVICE)
                
                # --- WARMUP FIX 1: Pseudo-Label Contrastive (Pulls similar unknowns together) ---
                _, z_novel = model(xb)
                loss_ss = margin_contrastive_loss(z_novel, yb_pseudo)
                
                # --- WARMUP FIX 2: Memory Anchoring (Protects Known Classes from drifting) ---
                X_mem, Y_mem, _ = memory.sample_balanced(32, model)
                if X_mem is not None:
                    X_mem, Y_mem = X_mem.to(DEVICE), Y_mem.to(DEVICE)
                    logits_mem, _ = model(X_mem)
                    loss_mem = F.cross_entropy(logits_mem, Y_mem)
                else:
                    loss_mem = torch.tensor(0.0).to(DEVICE)
                
                loss = loss_ss + loss_mem
                
                warmup_opt.zero_grad()
                loss.backward()
                warmup_opt.step()
                
        model.eval()
        print(f"     🌌 [WARMUP COMPLETE] Unknown clusters tightened. Memory space protected.")
        # ============================================================
        
        print(f"\n🧠 [CLUSTERING] Running MNN + HDBSCAN on novelty buffer (Size: {len(novelty_buffer)})...")
        
        Z = []
        with torch.no_grad():
            for img, _ in novelty_buffer:
                _, z = model(img.unsqueeze(0).to(DEVICE))
                Z.append(z.squeeze().cpu().numpy())
        Z = np.stack(Z)

        orig_dist = sp_dist.cdist(Z, Z, metric='cosine')
        nn_model = NearestNeighbors(n_neighbors=MNN_K, metric='cosine').fit(Z)
        _, indices = nn_model.kneighbors(Z)
        
        N = len(Z)
        mnn_dist = np.full((N, N), 2.0) 
        np.fill_diagonal(mnn_dist, 0.0)
        
        for i in range(N):
            for j in indices[i]:
                if i in indices[j]: 
                    mnn_dist[i, j] = orig_dist[i, j]
                    mnn_dist[j, i] = orig_dist[i, j]
                    
        labels = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=MIN_CLUSTER_SIZE, cluster_selection_epsilon=EPSILON).fit_predict(mnn_dist)
        
        new_buffer = []
        found, promoted = 0, 0

        for cid in sorted(set(labels)):
            idxs = np.where(labels == cid)[0]
            if cid == -1:
                print(f"  🗑️  [CLUSTER -1] NOISE: Found {len(idxs)} noise samples.")
                for i in idxs: new_buffer.append(novelty_buffer[i])
                continue

            found += 1
            X_list = [novelty_buffer[i][0] for i in idxs]
            Xc = torch.stack(X_list)
            with torch.no_grad(): _, Zc = model(Xc.to(DEVICE))
            
            mu = F.normalize(Zc.mean(0), dim=0)
            n = len(idxs)
            S_intra = torch.mean(1 - torch.matmul(Zc, mu))
            
            if len(detector.mu) > 0:
                S_known = min([1 - torch.dot(mu, detector.mu[k].to(mu.device)) for k in detector.mu])
            else:
                S_known = torch.tensor(1.0)

            density = n / (S_intra.item() + 1e-6)
            margin = S_known - S_intra
            
            labels_true = [novelty_buffer[i][1] for i in idxs]
            sem_label, cnt = Counter(labels_true).most_common(1)[0]
            purity = cnt / len(labels_true) 
            
            if sem_label in semantic_to_internal:
                print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Dominant class '{CIFAR10_LABELS[sem_label]}' already known.")
                continue 

            cond_intra = S_intra.item() <= ALPHA
            cond_density = density >= DELTA
            cond_known = S_known.item() >= BETA
            cond_margin = margin.item() > -0.10

            print(f"\n  📊 [MICRO-CLUSTER {cid} EVALUATION] Dominant: '{CIFAR10_LABELS[sem_label]}' (Purity: {purity:.2f})")
            print(f"     ➔ Size:    {n:3d}   (Micro-cluster) ✅")
            print(f"     ➔ S_intra: {S_intra.item():.3f} (Req: <= {ALPHA:.2f}) {'✅' if cond_intra else '❌'}")
            print(f"     ➔ Margin:  {margin.item():.3f} (Req: > -0.10) {'✅' if cond_margin else '❌'}")

            if cond_intra and cond_density and cond_known and cond_margin:
                best_stash_idx = -1
                best_sim = -1
                
                for s_idx, stash in enumerate(candidate_stash):
                    sim = torch.dot(mu, stash['centroid'].to(DEVICE)).item()
                    if sim > best_sim:
                        best_sim = sim
                        best_stash_idx = s_idx
                
                if best_stash_idx != -1 and best_sim >= MERGE_THRESHOLD:
                    print(f"     🤝 [STASH MERGE] Matched with Stash {best_stash_idx} (Sim: {best_sim:.2f})")
                    candidate_stash[best_stash_idx]['images'].extend(X_list)
                    candidate_stash[best_stash_idx]['labels'].extend(labels_true)
                    candidate_stash[best_stash_idx]['Z'].extend([Zc[j].cpu() for j in range(n)])
                    
                    all_Z = torch.stack(candidate_stash[best_stash_idx]['Z'])
                    candidate_stash[best_stash_idx]['centroid'] = F.normalize(all_Z.mean(0), dim=0)
                    stash_ref = candidate_stash[best_stash_idx]
                else:
                    print(f"     📦 [STASH CREATED] New candidate group created.")
                    candidate_stash.append({
                        'images': X_list,
                        'labels': labels_true,
                        'Z': [Zc[j].cpu() for j in range(n)],
                        'centroid': mu.cpu()
                    })
                    stash_ref = candidate_stash[-1]
                    
                total_stash_size = len(stash_ref['images'])
                print(f"     ⏳ [WAITING ROOM] Current stash size: {total_stash_size}/{PROMOTION_THRESHOLD}")
                
                if total_stash_size >= PROMOTION_THRESHOLD:
                    promoted += 1
                    new_label = known_classes
                    semantic_to_internal[sem_label] = new_label
                    internal_to_semantic[new_label] = sem_label
                    known_classes += 1
                    
                    Xc_promoted = torch.stack(stash_ref['images'])
                    torch.save(Xc_promoted.cpu(), f"{BASE_DIR}/promoted_class_{sem_label}.pt")
                    
                    model.expand_head(known_classes)
                    model.to(DEVICE)
                    
                    print(f"     🎉 [PROMOTION TRIGGERED] -> Stash full! Finetuning '{CIFAR10_LABELS[sem_label]}'...")
                    finetune(model, memory, Xc_promoted, new_label, t, DataLoader(test_ds, batch_size=128, shuffle=False), locked_generator)
                    print(f"     ✨ [LEARNED] -> Network successfully adapted to '{CIFAR10_LABELS[sem_label]}'")
                    
                    memory.build_memory_herding(Xc_promoted, new_label, model)
                    detector.update(memory.get(), model)
                    promoted_classes_log.append({"task": t, "semantic": CIFAR10_LABELS[sem_label]})
                    
                    candidate_stash.remove(stash_ref)
            else:
                print(f"     🛑 [REJECTED] -> Conditions not met. Retaining samples in buffer.")
                for i in idxs: new_buffer.append(novelty_buffer[i])
                
        novelty_buffer = new_buffer
        task_cpr.append(promoted / max(found, 1))
    else:
        task_cpr.append(0.0)

    if t > 0 and t % 13 == 0:
        novelty_buffer = [] 
        candidate_stash = [] 
        print(f"\n🧹 [BUFFER CLEAR] End of multi-class wave! Flushed buffer & waiting room.")

    learned_classes_over_time.append(set(learned_classes_over_time[-1]) if len(learned_classes_over_time) > 0 else set())
    for p in promoted_classes_log:
        if p["task"] == t and p["semantic"] not in learned_classes_over_time[-1]:
            learned_classes_over_time[-1].add(p["semantic"])

    acc, avg_forg, current_class_accs = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
    task_kca.append(acc)
    average_forgetting.append(avg_forg)
    print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")

final_accs = class_accuracies[NUM_TASKS - 1]
row_result = {
    "HPT": "Path B (Warmup+Anchor)",
    "Method": METHOD,
    "Final_KCA": f"{task_kca[-1]:.3f}",
    "Avg_Forg": f"{average_forgetting[-1]:.3f}",
    "Classes_Learned": str(len(learned_classes_over_time[-1]))
}
for cls_id in range(10):
    row_result[f"Cls{cls_id}_{CIFAR10_LABELS[cls_id][:3]}"] = f"{final_accs.get(cls_id, 0.0):.2f}"

final_results_summary = [row_result]

print("\n" + "="*120)
print("📊 FINAL OPTIMIZED RUN SUMMARY (Path B: K-Means + Anchoring + Stash)")
print("="*120)

columns = ["HPT", "Method", "Final KCA", "Avg Forg", "Classes Learned"] + [f"Cls{i}" for i in range(10)]
header_format = "{:<22} | {:<18} | {:<9} | {:<8} | {:<15} | " + " | ".join([f"{{:<4}}"]*10)
print(header_format.format(*columns))
print("-" * 140)

for res in final_results_summary:
    row = [res["HPT"], res["Method"], res["Final_KCA"], res["Avg_Forg"], res["Classes_Learned"]]
    row += [res[k] for k in list(res.keys())[5:]]
    print(header_format.format(*row))

print("\n✅ Script complete.")


🚀 RUNNING PATH B PIPELINE: [Pseudo-Label Warmup | Memory Anchor | VIP Stash]

==================== 🚀 TASK 0 ====================
📦 [STREAM] Task 0 stream contains class(es): ['airplane', 'automobile'] (Label IDs: [0, 1])
🎓 [INIT] Running initialization on Task 0...
📈 [METRIC] Known-Class Acc: 0.991

==================== 🚀 TASK 1 ====================
📦 [STREAM] Task 1 stream contains class(es): ['airplane', 'automobile', 'bird', 'cat'] (Label IDs: [0, 1, 2, 3])
     🔥 [WARMUP] Applying Pseudo-Label Contrastive & Memory Anchoring...
     🌌 [WARMUP COMPLETE] Unknown clusters tightened. Memory space protected.

🧠 [CLUSTERING] Running MNN + HDBSCAN on novelty buffer (Size: 400)...
  🗑️  [CLUSTER -1] NOISE: Found 15 noise samples.

  📊 [MICRO-CLUSTER 0 EVALUATION] Dominant: 'cat' (Purity: 0.64)
     ➔ Size:     39   (Micro-cluster) ✅
     ➔ S_intra: 0.065 (Req: <= 0.35) ✅
     ➔ Margin:  0.234 (Req: > -0.10) ✅
     📦 [STASH CREATED] New candidate group created.
     ⏳ [WAITING ROOM] Current

In [5]:
import os
# THESE MUST BE SET BEFORE IMPORTING TORCH
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.spatial.distance as sp_dist
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from collections import defaultdict, Counter
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torchvision.transforms as T
from torchvision.models import resnet18
import hdbscan
import copy
from torchvision import datasets as tv_datasets, transforms

# LabelBench Imports
from LabelBench.skeleton.dataset_skeleton import datasets as DATASET_REGISTRY
from LabelBench.skeleton.dataset_skeleton import register_dataset, LabelType, TransformDataset

# ============================================================
# BULLETPROOF REPRODUCIBILITY LOCK
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if hasattr(torch, "use_deterministic_algorithms"):
        torch.use_deterministic_algorithms(True, warn_only=True)

def get_locked_generator(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# ============================================================
# STABLE METRIC LEARNING LOSSES
# ============================================================
def soft_center_loss(z, labels, margin=0.15):
    loss = torch.tensor(0.0, device=z.device)
    for c in torch.unique(labels):
        mask = (labels == c)
        if mask.sum() > 1:
            centroid = z[mask].mean(dim=0, keepdim=True).detach()
            centroid = F.normalize(centroid, dim=1)
            dist = 1 - F.cosine_similarity(z[mask], centroid)
            loss += F.relu(dist - margin).mean()
    return loss

def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

def triplet_loss_batch(z, labels, margin=1.0):
    loss = torch.tensor(0.0, device=z.device)
    valid_triplets = 0
    for i in range(len(z)):
        anchor = z[i]
        pos_mask = (labels == labels[i])
        pos_mask[i] = False
        neg_mask = (labels != labels[i])
        if pos_mask.sum() > 0 and neg_mask.sum() > 0:
            pos = z[pos_mask][torch.randint(0, pos_mask.sum(), (1,))].squeeze(0)
            neg = z[neg_mask][torch.randint(0, neg_mask.sum(), (1,))].squeeze(0)
            loss += F.triplet_margin_loss(anchor.unsqueeze(0), pos.unsqueeze(0), neg.unsqueeze(0), margin=margin)
            valid_triplets += 1
    if valid_triplets > 0:
        loss /= valid_triplets
    return loss

# ============================================================
# DIRECTORIES & CONFIGURATION 
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
global BASE_DIR

P = 1                              
TOPK_NOVELTY = 400              
MAX_NOVELTY_BUFFER = 2500        

# 🚀 THE FINAL CONFIGURATION (K-Means Warmup + Max Discovery Gates)
ALPHA = 0.50               # Relaxed intra-distance gate
BETA = 0.01                
DELTA = 5                  # Relaxed density gate
MIN_CLUSTER_SIZE = 10      # Micro-clustering
PROMOTION_THRESHOLD = 100  # Promote early to catch more classes
MNN_K = 5                  # Strict graph fracture
MERGE_THRESHOLD = 0.90     
EPSILON = 0.00
METHOD = "margin_contrastive"

CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat",
    4: "deer", 5: "dog", 6: "frog", 7: "horse",
    8: "ship", 9: "truck"
}

# ============================================================
# DYNAMIC 40-TASK DATASET DEFINITION (MULTI-CLASS WAVE LOGIC)
# ============================================================
NUM_TASKS = 40   

class CIFARStream(Dataset):
    def __init__(self, base_ds, indices):
        self.base_ds = base_ds      
        self.indices = indices      

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.base_ds[self.indices[idx]]
        return x, y

def one_hot(y, n=10):
    return F.one_hot(torch.tensor(y), num_classes=n).float()

@register_dataset("splitcifar10", LabelType.MULTI_CLASS)
def get_splitcifar10(_):
    raise RuntimeError("Use splitcifar10_<id>")

base_train_global = tv_datasets.CIFAR10(root="./data", train=True, download=True)
targets = np.array(base_train_global.targets)
class_indices = {c: np.where(targets == c)[0] for c in range(10)}

rng = np.random.default_rng(42)
for c in range(10):
    rng.shuffle(class_indices[c])

stream_splits = {}
for t in range(1, NUM_TASKS):
    stream_splits[t] = []
    
    if 1 <= t <= 13: reminders = [0, 1]
    elif 14 <= t <= 26: reminders = [0, 1, 2, 3]
    elif 27 <= t <= 39: reminders = [0, 1, 2, 3, 4, 5, 6]
    else: reminders = []
        
    for c in reminders:
        stream_splits[t].extend(rng.choice(class_indices[c], 100, replace=False))
        
    if 1 <= t <= 13:
        stream_splits[t].extend(np.array_split(class_indices[2], 13)[t-1])
        stream_splits[t].extend(np.array_split(class_indices[3], 13)[t-1])
    elif 14 <= t <= 26:
        stream_splits[t].extend(np.array_split(class_indices[4], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[5], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[6], 13)[t-14])
    elif 27 <= t <= 39:
        stream_splits[t].extend(np.array_split(class_indices[7], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[8], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[9], 13)[t-27])
        
    rng.shuffle(stream_splits[t])

for split_id in range(NUM_TASKS):
    @register_dataset(f"splitcifar10_{split_id}", LabelType.MULTI_CLASS)
    def _make_split(data_dir, split_id=split_id):
        tf = transforms.Compose([transforms.ToTensor()])
        base_train = tv_datasets.CIFAR10(root=data_dir, train=True, download=True)
        base_test  = tv_datasets.CIFAR10(root=data_dir, train=False, download=True)
        
        if split_id == 0: indices = [i for i,(x,y) in enumerate(base_train) if y in [0,1]]
        else: indices = stream_splits[split_id]

        train_ds = CIFARStream(base_train, indices)
        train_ds = TransformDataset(train_ds, transform=tf, target_transform=lambda y: one_hot(y,10))
        test_ds = TransformDataset(base_test, transform=tf, target_transform=lambda y: one_hot(y,10))

        return train_ds, test_ds, test_ds, None, None, None, 10, [str(i) for i in range(10)]

# ============================================================
# MODEL DEFINITION
# ============================================================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = resnet18(weights='DEFAULT')
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes, bias=False)
        self.scale = 20.0  

    def expand_head(self, new_classes):
        old_w = self.classifier.weight.data.clone()
        old_n = old_w.shape[0]
        new_classifier = nn.Linear(512, new_classes, bias=False).to(old_w.device)
        new_classifier.weight.data[:old_n] = old_w
        self.classifier = new_classifier

    def forward(self, x, labels=None):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = self.embed(z)
        z = F.normalize(z, dim=1)
        W = F.normalize(self.classifier.weight, dim=1)
        cosine = torch.matmul(z, W.t())
        if labels is not None: 
            m = 0.2 
            theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
            target_logits = torch.cos(theta + m)
            one_hot = F.one_hot(labels, num_classes=W.size(0)).float()
            logits = cosine * (1 - one_hot) + target_logits * one_hot
        else:
            logits = cosine
        logits = logits * self.scale
        return logits, z

# ============================================================
# TRACKERS & MEMORY BUFFER
# ============================================================
class MemoryBuffer:
    def __init__(self, max_per_class=400):
        self.data = defaultdict(list)
        self.max_per_class = max_per_class
        self.aug = T.Compose([
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip()
        ])

    @torch.no_grad()
    def build_memory_herding(self, X_all, y_label, model):
        model.eval()
        logits_all, Z_all = [], []
        for i in range(0, len(X_all), 128):
            batch = X_all[i:i+128].to(DEVICE)
            logits, z = model(batch)
            logits_all.append(logits.cpu())
            Z_all.append(z.cpu())
        logits_all = torch.cat(logits_all)
        Z_all = torch.cat(Z_all)
        class_mean = F.normalize(Z_all.mean(0), dim=0)
        selected_idx = []
        features = Z_all.clone()

        for k in range(min(self.max_per_class, len(X_all))):
            if k > 0:
                S = Z_all[selected_idx].sum(0)
            else:
                S = torch.zeros_like(class_mean)
            target = (k + 1) * class_mean - S
            distances = torch.norm(features - target, dim=1)
            for idx in selected_idx: distances[idx] = float('inf')
            best = distances.argmin().item()
            selected_idx.append(best)

        self.data[int(y_label)] = []
        for idx in selected_idx:
            self.data[int(y_label)].append((X_all[idx].detach().cpu(), logits_all[idx].detach().cpu()))
    
    def get(self): return self.data

    def sample_balanced(self, batch_size, model):
        classes = list(self.data.keys())
        if not classes: return None, None, None
        samples_per_class = max(1, batch_size // len(classes))
        X_mem, Y_mem, L_mem = [], [], []
        current_dim = model.classifier.out_features
        for cls in classes:
            samples = self.data[cls]
            if len(samples) == 0: continue
            replace = len(samples) < samples_per_class
            idx = np.random.choice(len(samples), samples_per_class, replace=replace)
            for i in idx:
                x, logit = samples[i]
                if logit.shape[0] < current_dim:
                    padded = torch.zeros(current_dim)
                    padded[:logit.shape[0]] = logit
                    logit = padded
                X_mem.append(x)
                Y_mem.append(cls)
                L_mem.append(logit)
        if not X_mem: return None, None, None
        X_tensor = torch.stack(X_mem)
        X_tensor = self.aug(X_tensor) 
        return X_tensor, torch.tensor(Y_mem), torch.stack(L_mem)

class HypersphereNovelty:
    def __init__(self, q=0.90):
        self.q = q
        self.mu, self.r = {}, {}

    def update(self, memory, model):
        self.mu, self.r = {}, {}
        for k, X_tuples in memory.items():
            if len(X_tuples) == 0: continue
            X = torch.stack([x for x, _ in X_tuples]).to(DEVICE)
            with torch.no_grad(): 
                _, Z = model(X)
            mu = F.normalize(Z.mean(0), dim=0)
            d = 1 - torch.matmul(Z, mu)
            self.mu[k] = mu
            self.r[k] = torch.quantile(d, self.q)

    def score(self, z):
        if not self.mu: return torch.tensor(0.0)
        return min([(1 - torch.dot(z, self.mu[k].to(z.device))) - self.r[k].to(z.device) for k in self.mu])

# ============================================================
# TRAINING LOOPS
# ============================================================
def train_supervised(model, loader, test_loader, task_id, method="baseline_ce"):
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    
    for epoch in range(30):
        for x, y in loader:
            x_device, y_device = x.to(DEVICE), y.argmax(1).to(DEVICE)
            x_aug = aug(x_device)
            logits, z = model(x_aug, y_device)
            loss_ce = F.cross_entropy(logits, y_device)
            z_norm = F.normalize(z, dim=1)
            
            if method == "baseline_ce": loss = loss_ce
            elif method == "soft_center_loss": loss = loss_ce + 1.0 * soft_center_loss(z_norm, y_device, margin=0.15)
            elif method == "margin_contrastive": loss = loss_ce + 1.0 * margin_contrastive_loss(z_norm, y_device)
            elif method == "triplet": loss = loss_ce + 1.0 * triplet_loss_batch(z_norm, y_device, margin=1.0)
                
            opt.zero_grad()
            loss.backward()
            opt.step()

def finetune(model, memory, X_new, new_label, task_id, test_loader, locked_gen): 
    old_model = copy.deepcopy(model).eval()
    for p in old_model.parameters(): p.requires_grad = False 
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()
            
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    Y_new = torch.full((len(X_new),), new_label, dtype=torch.long)
    
    loader = DataLoader(TensorDataset(X_new, Y_new), batch_size=32, shuffle=True, generator=locked_gen)
    opt = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

    for epoch in range(25):
        for xb, yb in loader:
            xb = aug(xb) 
            X_mem, Y_mem, L_mem = memory.sample_balanced(32, model)
            if X_mem is not None:
                xb = torch.cat([xb, X_mem], dim=0)
                yb = torch.cat([yb, Y_mem], dim=0)

            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits_margin, Z = model(xb, yb)
            loss_ce = F.cross_entropy(logits_margin, yb)
            
            if X_mem is not None:
                pure_logits, _ = model(xb) 
                logits_mem = pure_logits[-len(X_mem):]
                loss_der = F.mse_loss(logits_mem, L_mem.to(DEVICE))
            else:
                loss_der = torch.tensor(0.0, device=DEVICE)

            with torch.no_grad(): 
                logits_old, Z_old = old_model(xb)

            loss_feat = (1 - F.cosine_similarity(Z, Z_old)).mean()
            loss = loss_ce + 0.5 * loss_der + 1.0 * loss_feat
            
            opt.zero_grad()
            loss.backward()
            opt.step()
    model.eval()

def evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies):
    correct_per_class = defaultdict(int)
    total_per_class = defaultdict(int)
    model.eval()
    with torch.no_grad():
        for x_test, y_test in DataLoader(test_ds, batch_size=128):
            x_test = x_test.to(DEVICE)
            y_test = y_test.argmax(1).to(DEVICE)
            logits, _ = model(x_test)
            preds = logits.argmax(1)
            
            for i in range(len(y_test)):
                sem = int(y_test[i])
                if sem in semantic_to_internal:
                    internal_gt = semantic_to_internal[sem]
                    total_per_class[sem] += 1
                    if preds[i].item() == internal_gt:
                        correct_per_class[sem] += 1

    current_class_accs = {}
    for sem in semantic_to_internal.keys():
        current_class_accs[sem] = correct_per_class[sem] / max(total_per_class[sem], 1)
        
    class_accuracies[t] = current_class_accs
    total_correct = sum(correct_per_class.values())
    total_eval = sum(total_per_class.values())
    acc = total_correct / max(total_eval, 1)
    
    if t > 0:
        forgetting_list = []
        for sem in current_class_accs.keys():
            past_accs = [class_accuracies[k].get(sem, None) for k in range(t)]
            past_accs = [a for a in past_accs if a is not None]
            if past_accs:
                max_past = max(past_accs)
                forgetting = max_past - current_class_accs[sem]
                forgetting_list.append(forgetting)
        avg_forg = np.mean(forgetting_list) if forgetting_list else 0.0
    else:
        avg_forg = 0.0
        
    return acc, avg_forg, current_class_accs

# ============================================================
# GRAND SINGLE RUN: THE FINAL PIPELINE (K-Means Warmup + Max Discovery)
# ============================================================

print("\n" + "="*80)
print(f"🚀 RUNNING FINAL PIPELINE: [K-Means Warmup | Memory Anchor | Relaxed Gates]")
print("="*80)

set_seed(42)
locked_generator = get_locked_generator(42)

BASE_DIR = f"debug_FinalPipeline_{METHOD}"
os.makedirs(BASE_DIR, exist_ok=True)

task_kca, task_cpr = [], []
average_forgetting = [] 
class_accuracies = {}   
learned_classes_over_time = []
promoted_classes_log = []

TASKS = [f"splitcifar10_{i}" for i in range(NUM_TASKS)]
model = CNN(num_classes=2).to(DEVICE)
memory = MemoryBuffer(max_per_class=400) 
detector = HypersphereNovelty()
novelty_buffer = []
candidate_stash = [] 

semantic_to_internal = {0: 0, 1: 1}
internal_to_semantic = {0: 0, 1: 1}
known_classes = 2

for t, task in enumerate(TASKS):
    _, dataset_fn = DATASET_REGISTRY[task]
    train_ds, test_ds, *_ = dataset_fn("./data")
    loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=locked_generator) 

    print(f"\n{'='*20} 🚀 TASK {t} {'='*20}")
    all_semantics = []
    for _, y in loader: all_semantics.extend(y.argmax(1).tolist())
    unique_semantics = list(set(all_semantics))
    class_names = [CIFAR10_LABELS[sem] for sem in unique_semantics]
    print(f"📦 [STREAM] Task {t} stream contains class(es): {class_names} (Label IDs: {unique_semantics})")
    
    if t == 0:
        print(f"🎓 [INIT] Running initialization on Task 0...")
        train_supervised(model, loader, DataLoader(test_ds, batch_size=128, shuffle=False), t, method=METHOD)
        for cls in [0, 1]:
            Xc = torch.cat([x[y.argmax(1) == cls] for x, y in loader])
            memory.build_memory_herding(Xc, cls, model)
        detector.update(memory.get(), model)
        learned_classes_over_time.append({CIFAR10_LABELS[0], CIFAR10_LABELS[1]})
        
        acc, avg_forg, current_class_accs = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
        task_kca.append(acc)
        average_forgetting.append(avg_forg)
        print(f"📈 [METRIC] Known-Class Acc: {acc:.3f}")
        continue

    model.eval()
    novelty_candidates = []
    novel, false_novel, total = 0, 0, 0
    
    with torch.no_grad():
        for x, y in loader:
            _, z = model(x.to(DEVICE))
            y_labels = y.argmax(1)
            scores = [detector.score(z[i]).item() for i in range(len(z))]
            thr = np.percentile(scores, 30)
            for i in range(len(z)):
                total += 1
                if scores[i] > thr:
                    novelty_candidates.append((scores[i], x[i].cpu(), y_labels[i].item()))
                    novel += 1
                    if y_labels[i].item() in semantic_to_internal: false_novel += 1

    novelty_candidates.sort(reverse=True, key=lambda x: x[0])
    novelty_buffer.extend([(img, y) for _, img, y in novelty_candidates[:TOPK_NOVELTY]])
    novelty_buffer = novelty_buffer[-MAX_NOVELTY_BUFFER:]

    if t % P == 0 and len(novelty_buffer) >= MNN_K + 1: 
        # ============================================================
        # 🚀 THE SUCCESSFUL WARMUP: K-Means Pseudo-Labels + Anchor
        # ============================================================
        print(f"     🔥 [WARMUP] Applying Pseudo-Label Contrastive & Memory Anchoring...")
        model.eval()
        
        with torch.no_grad():
            Z_buffer = []
            for img, _ in novelty_buffer:
                _, z = model(img.unsqueeze(0).to(DEVICE))
                Z_buffer.append(z.squeeze().cpu().numpy())
            Z_buffer = np.stack(Z_buffer)
            
        kmeans = KMeans(n_clusters=10, random_state=42, n_init=10).fit(Z_buffer)
        pseudo_labels = kmeans.labels_
        
        X_buffer = torch.stack([img for img, _ in novelty_buffer])
        Y_pseudo = torch.tensor(pseudo_labels, dtype=torch.long)
        warmup_loader = DataLoader(TensorDataset(X_buffer, Y_pseudo), batch_size=32, shuffle=True)
        
        model.train()
        warmup_opt = torch.optim.Adam(model.parameters(), lr=1e-4)
        aug_view = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ColorJitter(0.4, 0.4, 0.4, 0.1)])

        for warmup_epoch in range(5):
            for xb, yb_pseudo in warmup_loader:
                xb = aug_view(xb).to(DEVICE)
                yb_pseudo = yb_pseudo.to(DEVICE)
                
                _, z_novel = model(xb)
                loss_ss = margin_contrastive_loss(z_novel, yb_pseudo)
                
                X_mem, Y_mem, _ = memory.sample_balanced(32, model)
                if X_mem is not None:
                    X_mem, Y_mem = X_mem.to(DEVICE), Y_mem.to(DEVICE)
                    logits_mem, _ = model(X_mem)
                    loss_mem = F.cross_entropy(logits_mem, Y_mem)
                else:
                    loss_mem = torch.tensor(0.0).to(DEVICE)
                
                loss = loss_ss + loss_mem
                
                warmup_opt.zero_grad()
                loss.backward()
                warmup_opt.step()
                
        model.eval()
        print(f"     🌌 [WARMUP COMPLETE] Unknown clusters tightened. Memory space protected.")
        # ============================================================
        
        print(f"\n🧠 [CLUSTERING] Running MNN (K={MNN_K}) + HDBSCAN on novelty buffer (Size: {len(novelty_buffer)})...")
        
        Z = []
        with torch.no_grad():
            for img, _ in novelty_buffer:
                _, z = model(img.unsqueeze(0).to(DEVICE))
                Z.append(z.squeeze().cpu().numpy())
        Z = np.stack(Z)

        orig_dist = sp_dist.cdist(Z, Z, metric='cosine')
        nn_model = NearestNeighbors(n_neighbors=MNN_K, metric='cosine').fit(Z)
        _, indices = nn_model.kneighbors(Z)
        
        N = len(Z)
        mnn_dist = np.full((N, N), 2.0) 
        np.fill_diagonal(mnn_dist, 0.0)
        
        for i in range(N):
            for j in indices[i]:
                if i in indices[j]: 
                    mnn_dist[i, j] = orig_dist[i, j]
                    mnn_dist[j, i] = orig_dist[i, j]
                    
        labels = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=MIN_CLUSTER_SIZE, cluster_selection_epsilon=EPSILON).fit_predict(mnn_dist)
        
        new_buffer = []
        found, promoted = 0, 0

        for cid in sorted(set(labels)):
            idxs = np.where(labels == cid)[0]
            if cid == -1:
                print(f"  🗑️  [CLUSTER -1] NOISE: Found {len(idxs)} noise samples.")
                for i in idxs: new_buffer.append(novelty_buffer[i])
                continue

            found += 1
            X_list = [novelty_buffer[i][0] for i in idxs]
            Xc = torch.stack(X_list)
            with torch.no_grad(): _, Zc = model(Xc.to(DEVICE))
            
            mu = F.normalize(Zc.mean(0), dim=0)
            n = len(idxs)
            S_intra = torch.mean(1 - torch.matmul(Zc, mu))
            
            if len(detector.mu) > 0:
                S_known = min([1 - torch.dot(mu, detector.mu[k].to(mu.device)) for k in detector.mu])
            else:
                S_known = torch.tensor(1.0)

            density = n / (S_intra.item() + 1e-6)
            margin = S_known - S_intra
            
            labels_true = [novelty_buffer[i][1] for i in idxs]
            sem_label, cnt = Counter(labels_true).most_common(1)[0]
            purity = cnt / len(labels_true) 
            
            if sem_label in semantic_to_internal:
                print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Dominant class '{CIFAR10_LABELS[sem_label]}' already known.")
                continue 

            cond_intra = S_intra.item() <= ALPHA
            cond_density = density >= DELTA
            cond_known = S_known.item() >= BETA
            # 🚀 RELAXED MARGIN FOR MAX DISCOVERY
            cond_margin = margin.item() > -0.30

            print(f"\n  📊 [MICRO-CLUSTER {cid} EVALUATION] Dominant: '{CIFAR10_LABELS[sem_label]}' (Purity: {purity:.2f})")
            print(f"     ➔ Size:    {n:3d}   (Micro-cluster) ✅")
            print(f"     ➔ S_intra: {S_intra.item():.3f} (Req: <= {ALPHA:.2f}) {'✅' if cond_intra else '❌'}")
            print(f"     ➔ Margin:  {margin.item():.3f} (Req: > -0.30) {'✅' if cond_margin else '❌'}")

            if cond_intra and cond_density and cond_known and cond_margin:
                best_stash_idx = -1
                best_sim = -1
                
                for s_idx, stash in enumerate(candidate_stash):
                    sim = torch.dot(mu, stash['centroid'].to(DEVICE)).item()
                    if sim > best_sim:
                        best_sim = sim
                        best_stash_idx = s_idx
                
                if best_stash_idx != -1 and best_sim >= MERGE_THRESHOLD:
                    print(f"     🤝 [STASH MERGE] Matched with Stash {best_stash_idx} (Sim: {best_sim:.2f})")
                    candidate_stash[best_stash_idx]['images'].extend(X_list)
                    candidate_stash[best_stash_idx]['labels'].extend(labels_true)
                    candidate_stash[best_stash_idx]['Z'].extend([Zc[j].cpu() for j in range(n)])
                    
                    all_Z = torch.stack(candidate_stash[best_stash_idx]['Z'])
                    candidate_stash[best_stash_idx]['centroid'] = F.normalize(all_Z.mean(0), dim=0)
                    stash_ref = candidate_stash[best_stash_idx]
                else:
                    print(f"     📦 [STASH CREATED] New candidate group created.")
                    candidate_stash.append({
                        'images': X_list,
                        'labels': labels_true,
                        'Z': [Zc[j].cpu() for j in range(n)],
                        'centroid': mu.cpu()
                    })
                    stash_ref = candidate_stash[-1]
                    
                total_stash_size = len(stash_ref['images'])
                print(f"     ⏳ [WAITING ROOM] Current stash size: {total_stash_size}/{PROMOTION_THRESHOLD}")
                
                if total_stash_size >= PROMOTION_THRESHOLD:
                    promoted += 1
                    new_label = known_classes
                    semantic_to_internal[sem_label] = new_label
                    internal_to_semantic[new_label] = sem_label
                    known_classes += 1
                    
                    Xc_promoted = torch.stack(stash_ref['images'])
                    torch.save(Xc_promoted.cpu(), f"{BASE_DIR}/promoted_class_{sem_label}.pt")
                    
                    model.expand_head(known_classes)
                    model.to(DEVICE)
                    
                    print(f"     🎉 [PROMOTION TRIGGERED] -> Stash full! Finetuning '{CIFAR10_LABELS[sem_label]}'...")
                    finetune(model, memory, Xc_promoted, new_label, t, DataLoader(test_ds, batch_size=128, shuffle=False), locked_generator)
                    print(f"     ✨ [LEARNED] -> Network successfully adapted to '{CIFAR10_LABELS[sem_label]}'")
                    
                    memory.build_memory_herding(Xc_promoted, new_label, model)
                    detector.update(memory.get(), model)
                    promoted_classes_log.append({"task": t, "semantic": CIFAR10_LABELS[sem_label]})
                    
                    candidate_stash.remove(stash_ref)
            else:
                print(f"     🛑 [REJECTED] -> Conditions not met. Retaining samples in buffer.")
                for i in idxs: new_buffer.append(novelty_buffer[i])
                
        novelty_buffer = new_buffer
        task_cpr.append(promoted / max(found, 1))
    else:
        task_cpr.append(0.0)

    if t > 0 and t % 13 == 0:
        novelty_buffer = [] 
        candidate_stash = [] 
        print(f"\n🧹 [BUFFER CLEAR] End of multi-class wave! Flushed buffer & waiting room.")

    learned_classes_over_time.append(set(learned_classes_over_time[-1]) if len(learned_classes_over_time) > 0 else set())
    for p in promoted_classes_log:
        if p["task"] == t and p["semantic"] not in learned_classes_over_time[-1]:
            learned_classes_over_time[-1].add(p["semantic"])

    acc, avg_forg, current_class_accs = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
    task_kca.append(acc)
    average_forgetting.append(avg_forg)
    print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")

final_accs = class_accuracies[NUM_TASKS - 1]
row_result = {
    "HPT": "The Final Synthesis",
    "Method": METHOD,
    "Final_KCA": f"{task_kca[-1]:.3f}",
    "Avg_Forg": f"{average_forgetting[-1]:.3f}",
    "Classes_Learned": str(len(learned_classes_over_time[-1]))
}
for cls_id in range(10):
    row_result[f"Cls{cls_id}_{CIFAR10_LABELS[cls_id][:3]}"] = f"{final_accs.get(cls_id, 0.0):.2f}"

final_results_summary = [row_result]

print("\n" + "="*120)
print("📊 FINAL OPTIMIZED RUN SUMMARY (K-Means Anchor + Max Discovery Gates)")
print("="*120)

columns = ["HPT", "Method", "Final KCA", "Avg Forg", "Classes Learned"] + [f"Cls{i}" for i in range(10)]
header_format = "{:<32} | {:<18} | {:<9} | {:<8} | {:<15} | " + " | ".join([f"{{:<4}}"]*10)
print(header_format.format(*columns))
print("-" * 140)

for res in final_results_summary:
    row = [res["HPT"], res["Method"], res["Final_KCA"], res["Avg_Forg"], res["Classes_Learned"]]
    row += [res[k] for k in list(res.keys())[5:]]
    print(header_format.format(*row))

print("\n✅ Script complete.")

100%|██████████| 170M/170M [00:02<00:00, 66.4MB/s] 



🚀 RUNNING FINAL PIPELINE: [K-Means Warmup | Memory Anchor | Relaxed Gates]
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 187MB/s]



==================== 🚀 TASK 0 ====================
📦 [STREAM] Task 0 stream contains class(es): ['airplane', 'automobile'] (Label IDs: [0, 1])
🎓 [INIT] Running initialization on Task 0...
📈 [METRIC] Known-Class Acc: 0.991

==================== 🚀 TASK 1 ====================
📦 [STREAM] Task 1 stream contains class(es): ['airplane', 'automobile', 'bird', 'cat'] (Label IDs: [0, 1, 2, 3])
     🔥 [WARMUP] Applying Pseudo-Label Contrastive & Memory Anchoring...
     🌌 [WARMUP COMPLETE] Unknown clusters tightened. Memory space protected.

🧠 [CLUSTERING] Running MNN (K=5) + HDBSCAN on novelty buffer (Size: 400)...
  🗑️  [CLUSTER -1] NOISE: Found 15 noise samples.

  📊 [MICRO-CLUSTER 0 EVALUATION] Dominant: 'cat' (Purity: 0.64)
     ➔ Size:     39   (Micro-cluster) ✅
     ➔ S_intra: 0.065 (Req: <= 0.50) ✅
     ➔ Margin:  0.234 (Req: > -0.30) ✅
     📦 [STASH CREATED] New candidate group created.
     ⏳ [WAITING ROOM] Current stash size: 39/100

  📊 [MICRO-CLUSTER 1 EVALUATION] Dominant: 'cat' (P

## 60.2 with tsne

In [4]:
import os
# THESE MUST BE SET BEFORE IMPORTING TORCH
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.spatial.distance as sp_dist
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from collections import defaultdict, Counter
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torchvision.transforms as T
from torchvision.models import resnet18
import hdbscan
import copy
from torchvision import datasets as tv_datasets, transforms

# --- NEW IMPORTS FOR VISUALIZATION ---
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# LabelBench Imports
from LabelBench.skeleton.dataset_skeleton import datasets as DATASET_REGISTRY
from LabelBench.skeleton.dataset_skeleton import register_dataset, LabelType, TransformDataset

# ============================================================
# BULLETPROOF REPRODUCIBILITY LOCK
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if hasattr(torch, "use_deterministic_algorithms"):
        torch.use_deterministic_algorithms(True, warn_only=True)

def get_locked_generator(seed=42):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# ============================================================
# STABLE METRIC LEARNING LOSSES
# ============================================================
def soft_center_loss(z, labels, margin=0.15):
    loss = torch.tensor(0.0, device=z.device)
    for c in torch.unique(labels):
        mask = (labels == c)
        if mask.sum() > 1:
            centroid = z[mask].mean(dim=0, keepdim=True).detach()
            centroid = F.normalize(centroid, dim=1)
            dist = 1 - F.cosine_similarity(z[mask], centroid)
            loss += F.relu(dist - margin).mean()
    return loss

def margin_contrastive_loss(z, labels, pos_margin=0.85, neg_margin=0.1):
    sim = torch.matmul(z, z.T)
    mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask
    labels_diff = ~labels_equal & ~mask
    pos_loss = F.relu(pos_margin - sim)[labels_equal].mean() if labels_equal.sum() > 0 else torch.tensor(0.0)
    neg_loss = F.relu(sim - neg_margin)[labels_diff].mean() if labels_diff.sum() > 0 else torch.tensor(0.0)
    return pos_loss + neg_loss

def triplet_loss_batch(z, labels, margin=1.0):
    loss = torch.tensor(0.0, device=z.device)
    valid_triplets = 0
    for i in range(len(z)):
        anchor = z[i]
        pos_mask = (labels == labels[i])
        pos_mask[i] = False
        neg_mask = (labels != labels[i])
        if pos_mask.sum() > 0 and neg_mask.sum() > 0:
            pos = z[pos_mask][torch.randint(0, pos_mask.sum(), (1,))].squeeze(0)
            neg = z[neg_mask][torch.randint(0, neg_mask.sum(), (1,))].squeeze(0)
            loss += F.triplet_margin_loss(anchor.unsqueeze(0), pos.unsqueeze(0), neg.unsqueeze(0), margin=margin)
            valid_triplets += 1
    if valid_triplets > 0:
        loss /= valid_triplets
    return loss

# ============================================================
# DIRECTORIES & CONFIGURATION 
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
global BASE_DIR

P = 1                              
TOPK_NOVELTY = 400              
MAX_NOVELTY_BUFFER = 2500        

# 🚀 THE FINAL CONFIGURATION (K-Means Warmup + Max Discovery Gates)
ALPHA = 0.50               # Relaxed intra-distance gate
BETA = 0.01                
DELTA = 5                  # Relaxed density gate
MIN_CLUSTER_SIZE = 10      # Micro-clustering
PROMOTION_THRESHOLD = 100  # Promote early to catch more classes
MNN_K = 5                  # Strict graph fracture
MERGE_THRESHOLD = 0.90     
EPSILON = 0.00
METHOD = "margin_contrastive"

CIFAR10_LABELS = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat",
    4: "deer", 5: "dog", 6: "frog", 7: "horse",
    8: "ship", 9: "truck"
}

# ============================================================
# DYNAMIC 40-TASK DATASET DEFINITION 
# ============================================================
NUM_TASKS = 40   

class CIFARStream(Dataset):
    def __init__(self, base_ds, indices):
        self.base_ds = base_ds      
        self.indices = indices      

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.base_ds[self.indices[idx]]
        return x, y

def one_hot(y, n=10):
    return F.one_hot(torch.tensor(y), num_classes=n).float()

@register_dataset("splitcifar10", LabelType.MULTI_CLASS)
def get_splitcifar10(_):
    raise RuntimeError("Use splitcifar10_<id>")

base_train_global = tv_datasets.CIFAR10(root="./data", train=True, download=True)
targets = np.array(base_train_global.targets)
class_indices = {c: np.where(targets == c)[0] for c in range(10)}

rng = np.random.default_rng(42)
for c in range(10):
    rng.shuffle(class_indices[c])

stream_splits = {}
for t in range(1, NUM_TASKS):
    stream_splits[t] = []
    
    if 1 <= t <= 13: reminders = [0, 1]
    elif 14 <= t <= 26: reminders = [0, 1, 2, 3]
    elif 27 <= t <= 39: reminders = [0, 1, 2, 3, 4, 5, 6]
    else: reminders = []
        
    for c in reminders:
        stream_splits[t].extend(rng.choice(class_indices[c], 100, replace=False))
        
    if 1 <= t <= 13:
        stream_splits[t].extend(np.array_split(class_indices[2], 13)[t-1])
        stream_splits[t].extend(np.array_split(class_indices[3], 13)[t-1])
    elif 14 <= t <= 26:
        stream_splits[t].extend(np.array_split(class_indices[4], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[5], 13)[t-14])
        stream_splits[t].extend(np.array_split(class_indices[6], 13)[t-14])
    elif 27 <= t <= 39:
        stream_splits[t].extend(np.array_split(class_indices[7], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[8], 13)[t-27])
        stream_splits[t].extend(np.array_split(class_indices[9], 13)[t-27])
        
    rng.shuffle(stream_splits[t])

for split_id in range(NUM_TASKS):
    @register_dataset(f"splitcifar10_{split_id}", LabelType.MULTI_CLASS)
    def _make_split(data_dir, split_id=split_id):
        tf = transforms.Compose([transforms.ToTensor()])
        base_train = tv_datasets.CIFAR10(root=data_dir, train=True, download=True)
        base_test  = tv_datasets.CIFAR10(root=data_dir, train=False, download=True)
        
        if split_id == 0: indices = [i for i,(x,y) in enumerate(base_train) if y in [0,1]]
        else: indices = stream_splits[split_id]

        train_ds = CIFARStream(base_train, indices)
        train_ds = TransformDataset(train_ds, transform=tf, target_transform=lambda y: one_hot(y,10))
        test_ds = TransformDataset(base_test, transform=tf, target_transform=lambda y: one_hot(y,10))

        return train_ds, test_ds, test_ds, None, None, None, 10, [str(i) for i in range(10)]

# ============================================================
# MODEL DEFINITION
# ============================================================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = resnet18(weights='DEFAULT')
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.embed = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes, bias=False)
        self.scale = 20.0  

    def expand_head(self, new_classes):
        old_w = self.classifier.weight.data.clone()
        old_n = old_w.shape[0]
        new_classifier = nn.Linear(512, new_classes, bias=False).to(old_w.device)
        new_classifier.weight.data[:old_n] = old_w
        self.classifier = new_classifier

    def forward(self, x, labels=None):
        z = self.encoder(x)
        z = z.view(z.size(0), -1)
        z = self.embed(z)
        z = F.normalize(z, dim=1)
        W = F.normalize(self.classifier.weight, dim=1)
        cosine = torch.matmul(z, W.t())
        if labels is not None: 
            m = 0.2 
            theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
            target_logits = torch.cos(theta + m)
            one_hot = F.one_hot(labels, num_classes=W.size(0)).float()
            logits = cosine * (1 - one_hot) + target_logits * one_hot
        else:
            logits = cosine
        logits = logits * self.scale
        return logits, z

# ============================================================
# TSNE VISUALIZATIONS (PORTED & ENHANCED)
# ============================================================
# def visualize_tsne(model, loader, task_id, known_semantics, folder_name, tag=""):
#     model.eval()
#     Z_all, Y_all = [], []
#     with torch.no_grad():
#         for x, y in loader:
#             _, z = model(x.to(DEVICE))
#             y = y.argmax(1)
#             mask = torch.tensor([int(label.item()) in known_semantics for label in y])
#             if mask.sum() > 0: 
#                 Z_all.append(z[mask].cpu())
#                 Y_all.append(y[mask].cpu())
#     if not Z_all: return
    
#     Z_all, Y_all = torch.cat(Z_all).numpy(), torch.cat(Y_all).numpy()
#     n_samples = len(Z_all)
#     if n_samples < 2: return 
    
#     # if n_samples > 3000: 
#     #     idx = np.random.choice(n_samples, 3000, replace=False)
#     #     Z_all, Y_all = Z_all[idx], Y_all[idx]
#     #     n_samples = 3000
#     if n_samples > 3000: 
#         # FIREWALL: Use a local generator so we don't touch global numpy state
#         local_rng = np.random.default_rng(42)
#         idx = local_rng.choice(n_samples, 3000, replace=False)
#         Z_all, Y_all = Z_all[idx], Y_all[idx]
#         n_samples = 3000
    
#     perp = min(30, n_samples - 1)
#     tsne = TSNE(n_components=2, perplexity=perp, metric="cosine", random_state=42, init="pca")
#     Z_2d = tsne.fit_transform(Z_all)
    
#     plt.figure(figsize=(7,6))
#     for label in sorted(np.unique(Y_all)):
#         plt.scatter(Z_2d[Y_all == label, 0], Z_2d[Y_all == label, 1], s=12, alpha=0.6, label=CIFAR10_LABELS[int(label)])
#     plt.title(f"Task {task_id} ({tag})")
#     plt.legend(fontsize=8)
#     plt.grid(True)
#     plt.tight_layout()
#     filename = f"task_{task_id}_{tag}" if tag else f"task_{task_id}"
#     plt.savefig(f"{BASE_DIR}/{folder_name}/{filename}.png")
#     plt.close()

# def visualize_epoch_tsne(model, test_loader, task_id, epoch, known_semantics, folder_name):
#     model.eval()
#     Z_all, Y_all = [], []
#     with torch.no_grad():
#         for x, y in test_loader:
#             _, z = model(x.to(DEVICE))
#             y = y.argmax(1)
#             mask = torch.tensor([int(label.item()) in known_semantics for label in y])
#             if mask.sum() > 0: 
#                 Z_all.append(z[mask].cpu())
#                 Y_all.append(y[mask].cpu())
#     if not Z_all: return
    
#     Z_all, Y_all = torch.cat(Z_all).numpy(), torch.cat(Y_all).numpy()
#     n_samples = len(Z_all)
#     if n_samples < 2: return
    
#     # if n_samples > 3000: 
#     #     idx = np.random.choice(n_samples, 3000, replace=False)
#     #     Z_all, Y_all = Z_all[idx], Y_all[idx]
#     #     n_samples = 3000
#     if n_samples > 3000: 
#         # FIREWALL: Use a local generator
#         local_rng = np.random.default_rng(42)
#         idx = local_rng.choice(n_samples, 3000, replace=False)
#         Z_all, Y_all = Z_all[idx], Y_all[idx]
#         n_samples = 3000
        
#     perp = min(30, n_samples - 1)
#     tsne = TSNE(n_components=2, perplexity=perp, metric="cosine", random_state=42, init="pca")
#     Z_2d = tsne.fit_transform(Z_all)
    
#     plt.figure(figsize=(7,6))
#     for label in sorted(np.unique(Y_all)):
#         plt.scatter(Z_2d[Y_all == label, 0], Z_2d[Y_all == label, 1], s=12, alpha=0.6, label=CIFAR10_LABELS[int(label)])
#     plt.title(f"Finetune | Task {task_id} | Epoch {epoch}")
#     plt.legend(fontsize=8)
#     plt.grid(True)
#     plt.tight_layout()
#     save_path = f"{BASE_DIR}/{folder_name}/task{task_id}_epoch{epoch}.png"
#     plt.savefig(save_path)
#     plt.close()

# def visualize_clusters(Z, labels, task_id, folder_name):
#     n_samples = len(Z)
#     if n_samples < 2: 
#         print(f"⚠️  [DEBUG] Only {n_samples} points in buffer. Skipping cluster visualization.")
#         return
        
#     perp = min(30, n_samples - 1)
#     tsne = TSNE(n_components=2, perplexity=perp, random_state=42)
#     Z_2d = tsne.fit_transform(Z)

#     plt.figure(figsize=(6,5))
#     for cid in set(labels):
#         mask = labels == cid
#         if cid == -1:
#             plt.scatter(Z_2d[mask,0], Z_2d[mask,1], s=10, alpha=0.3, label="Noise")
#         else:
#             plt.scatter(Z_2d[mask,0], Z_2d[mask,1], s=15, alpha=0.7, label=f"Cluster {cid}")

#     plt.legend()
#     plt.title(f"HDBSCAN Structure Task {task_id}")
#     plt.grid(True)
#     plt.savefig(f"{BASE_DIR}/{folder_name}/task_{task_id}.png")
#     plt.close()


# ============================================================
# TSNE VISUALIZATIONS (PORTED & ENHANCED WITH STATE LOCKS)
# ============================================================
def visualize_tsne(model, loader, task_id, known_semantics, folder_name, tag=""):
    # 🛑 FIREWALL: Freeze global RNG and individual submodule states
    torch_state = torch.get_rng_state()
    np_state = np.random.get_state()
    py_state = random.getstate()
    if torch.cuda.is_available(): cuda_state = torch.cuda.get_rng_state()
    
    # Save the exact training state of EVERY layer (preserves your BatchNorm freeze)
    module_states = {m: m.training for m in model.modules()}

    model.eval()
    Z_all, Y_all = [], []
    with torch.no_grad():
        for x, y in loader:
            _, z = model(x.to(DEVICE))
            y = y.argmax(1)
            mask = torch.tensor([int(label.item()) in known_semantics for label in y])
            if mask.sum() > 0: 
                Z_all.append(z[mask].cpu())
                Y_all.append(y[mask].cpu())
    if not Z_all: 
        # Must restore states even if returning early
        for m, is_training in module_states.items(): m.training = is_training
        return
    
    Z_all, Y_all = torch.cat(Z_all).numpy(), torch.cat(Y_all).numpy()
    n_samples = len(Z_all)
    if n_samples < 2: 
        for m, is_training in module_states.items(): m.training = is_training
        return 
    
    if n_samples > 3000: 
        local_rng = np.random.default_rng(42)
        idx = local_rng.choice(n_samples, 3000, replace=False)
        Z_all, Y_all = Z_all[idx], Y_all[idx]
        n_samples = 3000
    
    perp = min(30, n_samples - 1)
    tsne = TSNE(n_components=2, perplexity=perp, metric="cosine", random_state=42, init="pca")
    Z_2d = tsne.fit_transform(Z_all)
    
    plt.figure(figsize=(7,6))
    for label in sorted(np.unique(Y_all)):
        plt.scatter(Z_2d[Y_all == label, 0], Z_2d[Y_all == label, 1], s=12, alpha=0.6, label=CIFAR10_LABELS[int(label)])
    plt.title(f"Task {task_id} ({tag})")
    plt.legend(fontsize=8)
    plt.grid(True)
    plt.tight_layout()
    filename = f"task_{task_id}_{tag}" if tag else f"task_{task_id}"
    plt.savefig(f"{BASE_DIR}/{folder_name}/{filename}.png")
    plt.close()

    # 🛑 FIREWALL: Restore global RNG and layer states exactly as they were
    for m, is_training in module_states.items():
        m.training = is_training # Bypasses PyTorch's recursive override
        
    torch.set_rng_state(torch_state)
    np.random.set_state(np_state)
    random.setstate(py_state)
    if torch.cuda.is_available(): torch.cuda.set_rng_state(cuda_state)


def visualize_epoch_tsne(model, test_loader, task_id, epoch, known_semantics, folder_name):
    # 🛑 FIREWALL: Freeze global RNG and individual submodule states
    torch_state = torch.get_rng_state()
    np_state = np.random.get_state()
    py_state = random.getstate()
    if torch.cuda.is_available(): cuda_state = torch.cuda.get_rng_state()
    
    # Save the exact training state of EVERY layer
    module_states = {m: m.training for m in model.modules()}

    model.eval()
    Z_all, Y_all = [], []
    with torch.no_grad():
        for x, y in test_loader:
            _, z = model(x.to(DEVICE))
            y = y.argmax(1)
            mask = torch.tensor([int(label.item()) in known_semantics for label in y])
            if mask.sum() > 0: 
                Z_all.append(z[mask].cpu())
                Y_all.append(y[mask].cpu())
    if not Z_all: 
        for m, is_training in module_states.items(): m.training = is_training
        return
    
    Z_all, Y_all = torch.cat(Z_all).numpy(), torch.cat(Y_all).numpy()
    n_samples = len(Z_all)
    if n_samples < 2: 
        for m, is_training in module_states.items(): m.training = is_training
        return
    
    if n_samples > 3000: 
        local_rng = np.random.default_rng(42)
        idx = local_rng.choice(n_samples, 3000, replace=False)
        Z_all, Y_all = Z_all[idx], Y_all[idx]
        n_samples = 3000
        
    perp = min(30, n_samples - 1)
    tsne = TSNE(n_components=2, perplexity=perp, metric="cosine", random_state=42, init="pca")
    Z_2d = tsne.fit_transform(Z_all)
    
    plt.figure(figsize=(7,6))
    for label in sorted(np.unique(Y_all)):
        plt.scatter(Z_2d[Y_all == label, 0], Z_2d[Y_all == label, 1], s=12, alpha=0.6, label=CIFAR10_LABELS[int(label)])
    plt.title(f"Finetune | Task {task_id} | Epoch {epoch}")
    plt.legend(fontsize=8)
    plt.grid(True)
    plt.tight_layout()
    save_path = f"{BASE_DIR}/{folder_name}/task{task_id}_epoch{epoch}.png"
    plt.savefig(save_path)
    plt.close()

    # 🛑 FIREWALL: Restore global RNG and layer states exactly as they were
    for m, is_training in module_states.items():
        m.training = is_training 
        
    torch.set_rng_state(torch_state)
    np.random.set_state(np_state)
    random.setstate(py_state)
    if torch.cuda.is_available(): torch.cuda.set_rng_state(cuda_state)


def visualize_clusters(Z, labels, task_id, folder_name):
    # 🛑 FIREWALL: Freeze global RNG
    np_state = np.random.get_state()
    py_state = random.getstate()

    n_samples = len(Z)
    if n_samples < 2: 
        print(f"⚠️  [DEBUG] Only {n_samples} points in buffer. Skipping cluster visualization.")
    else:
        perp = min(30, n_samples - 1)
        tsne = TSNE(n_components=2, perplexity=perp, random_state=42)
        Z_2d = tsne.fit_transform(Z)

        plt.figure(figsize=(6,5))
        for cid in set(labels):
            mask = labels == cid
            if cid == -1:
                plt.scatter(Z_2d[mask,0], Z_2d[mask,1], s=10, alpha=0.3, label="Noise")
            else:
                plt.scatter(Z_2d[mask,0], Z_2d[mask,1], s=15, alpha=0.7, label=f"Cluster {cid}")

        plt.legend()
        plt.title(f"HDBSCAN Structure Task {task_id}")
        plt.grid(True)
        plt.savefig(f"{BASE_DIR}/{folder_name}/task_{task_id}.png")
        plt.close()

    # 🛑 FIREWALL: Restore global RNG exactly as it was
    np.random.set_state(np_state)
    random.setstate(py_state)

# ============================================================
# TRACKERS & MEMORY BUFFER
# ============================================================
class MemoryBuffer:
    def __init__(self, max_per_class=400):
        self.data = defaultdict(list)
        self.max_per_class = max_per_class
        self.aug = T.Compose([
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip()
        ])

    @torch.no_grad()
    def build_memory_herding(self, X_all, y_label, model):
        model.eval()
        logits_all, Z_all = [], []
        for i in range(0, len(X_all), 128):
            batch = X_all[i:i+128].to(DEVICE)
            logits, z = model(batch)
            logits_all.append(logits.cpu())
            Z_all.append(z.cpu())
        logits_all = torch.cat(logits_all)
        Z_all = torch.cat(Z_all)
        class_mean = F.normalize(Z_all.mean(0), dim=0)
        selected_idx = []
        features = Z_all.clone()

        for k in range(min(self.max_per_class, len(X_all))):
            if k > 0:
                S = Z_all[selected_idx].sum(0)
            else:
                S = torch.zeros_like(class_mean)
            target = (k + 1) * class_mean - S
            distances = torch.norm(features - target, dim=1)
            for idx in selected_idx: distances[idx] = float('inf')
            best = distances.argmin().item()
            selected_idx.append(best)

        self.data[int(y_label)] = []
        for idx in selected_idx:
            self.data[int(y_label)].append((X_all[idx].detach().cpu(), logits_all[idx].detach().cpu()))
    
    def get(self): return self.data

    def sample_balanced(self, batch_size, model):
        classes = list(self.data.keys())
        if not classes: return None, None, None
        samples_per_class = max(1, batch_size // len(classes))
        X_mem, Y_mem, L_mem = [], [], []
        current_dim = model.classifier.out_features
        for cls in classes:
            samples = self.data[cls]
            if len(samples) == 0: continue
            replace = len(samples) < samples_per_class
            idx = np.random.choice(len(samples), samples_per_class, replace=replace)
            for i in idx:
                x, logit = samples[i]
                if logit.shape[0] < current_dim:
                    padded = torch.zeros(current_dim)
                    padded[:logit.shape[0]] = logit
                    logit = padded
                X_mem.append(x)
                Y_mem.append(cls)
                L_mem.append(logit)
        if not X_mem: return None, None, None
        X_tensor = torch.stack(X_mem)
        X_tensor = self.aug(X_tensor) 
        return X_tensor, torch.tensor(Y_mem), torch.stack(L_mem)

class HypersphereNovelty:
    def __init__(self, q=0.90):
        self.q = q
        self.mu, self.r = {}, {}

    def update(self, memory, model):
        self.mu, self.r = {}, {}
        for k, X_tuples in memory.items():
            if len(X_tuples) == 0: continue
            X = torch.stack([x for x, _ in X_tuples]).to(DEVICE)
            with torch.no_grad(): 
                _, Z = model(X)
            mu = F.normalize(Z.mean(0), dim=0)
            d = 1 - torch.matmul(Z, mu)
            self.mu[k] = mu
            self.r[k] = torch.quantile(d, self.q)

    def score(self, z):
        if not self.mu: return torch.tensor(0.0)
        return min([(1 - torch.dot(z, self.mu[k].to(z.device))) - self.r[k].to(z.device) for k in self.mu])

# ============================================================
# TRAINING LOOPS
# ============================================================
def train_supervised(model, loader, test_loader, task_id, method="baseline_ce"):
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    model.train()
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    
    for epoch in range(30):
        for x, y in loader:
            x_device, y_device = x.to(DEVICE), y.argmax(1).to(DEVICE)
            x_aug = aug(x_device)
            logits, z = model(x_aug, y_device)
            loss_ce = F.cross_entropy(logits, y_device)
            z_norm = F.normalize(z, dim=1)
            
            if method == "baseline_ce": loss = loss_ce
            elif method == "soft_center_loss": loss = loss_ce + 1.0 * soft_center_loss(z_norm, y_device, margin=0.15)
            elif method == "margin_contrastive": loss = loss_ce + 1.0 * margin_contrastive_loss(z_norm, y_device)
            elif method == "triplet": loss = loss_ce + 1.0 * triplet_loss_batch(z_norm, y_device, margin=1.0)
                
            opt.zero_grad()
            loss.backward()
            opt.step()

def finetune(model, memory, X_new, new_label, task_id, test_loader, locked_gen): 
    old_model = copy.deepcopy(model).eval()
    for p in old_model.parameters(): p.requires_grad = False 
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d): m.eval()
            
    aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
    Y_new = torch.full((len(X_new),), new_label, dtype=torch.long)
    
    loader = DataLoader(TensorDataset(X_new, Y_new), batch_size=32, shuffle=True, generator=locked_gen)
    opt = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

    # Visualization: Before Finetuning
    visualize_epoch_tsne(model, test_loader, task_id, epoch="before", known_semantics=set(semantic_to_internal.keys()), folder_name="tsne_05_finetune_progression")

    for epoch in range(25):
        for xb, yb in loader:
            xb = aug(xb) 
            X_mem, Y_mem, L_mem = memory.sample_balanced(32, model)
            if X_mem is not None:
                xb = torch.cat([xb, X_mem], dim=0)
                yb = torch.cat([yb, Y_mem], dim=0)

            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits_margin, Z = model(xb, yb)
            loss_ce = F.cross_entropy(logits_margin, yb)
            
            if X_mem is not None:
                pure_logits, _ = model(xb) 
                logits_mem = pure_logits[-len(X_mem):]
                loss_der = F.mse_loss(logits_mem, L_mem.to(DEVICE))
            else:
                loss_der = torch.tensor(0.0, device=DEVICE)

            with torch.no_grad(): 
                logits_old, Z_old = old_model(xb)

            loss_feat = (1 - F.cosine_similarity(Z, Z_old)).mean()
            loss = loss_ce + 0.5 * loss_der + 1.0 * loss_feat
            
            opt.zero_grad()
            loss.backward()
            opt.step()
            
        # Visualizations during Finetuning
        if epoch == 4: 
            visualize_epoch_tsne(model, test_loader, task_id, epoch="after_epoch_5", known_semantics=set(semantic_to_internal.keys()), folder_name="tsne_05_finetune_progression")
        elif epoch == 14: 
            visualize_epoch_tsne(model, test_loader, task_id, epoch="after_epoch_15", known_semantics=set(semantic_to_internal.keys()), folder_name="tsne_05_finetune_progression")

    model.eval()
# def finetune(model, memory, X_new, new_label, task_id, test_loader, locked_gen): 
#     old_model = copy.deepcopy(model).eval()
#     for p in old_model.parameters(): p.requires_grad = False 
#     model.train()
#     for m in model.modules():
#         if isinstance(m, nn.BatchNorm2d): m.eval()
            
#     aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()])
#     Y_new = torch.full((len(X_new),), new_label, dtype=torch.long)
    
#     loader = DataLoader(TensorDataset(X_new, Y_new), batch_size=32, shuffle=True, generator=locked_gen)
#     opt = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)

#     # Visualization: Before Finetuning
#     visualize_epoch_tsne(model, test_loader, task_id, epoch="before", known_semantics=set(semantic_to_internal.keys()), folder_name="tsne_05_finetune_progression")
    
#     model.train() # <--- 🛑 ADD THIS HERE! Otherwise the loop runs in eval mode.

#     for epoch in range(25):
#         for xb, yb in loader:
#             xb = aug(xb) 
#             X_mem, Y_mem, L_mem = memory.sample_balanced(32, model)
#             if X_mem is not None:
#                 xb = torch.cat([xb, X_mem], dim=0)
#                 yb = torch.cat([yb, Y_mem], dim=0)

#             xb, yb = xb.to(DEVICE), yb.to(DEVICE)
#             logits_margin, Z = model(xb, yb)
#             loss_ce = F.cross_entropy(logits_margin, yb)
            
#             if X_mem is not None:
#                 pure_logits, _ = model(xb) 
#                 logits_mem = pure_logits[-len(X_mem):]
#                 loss_der = F.mse_loss(logits_mem, L_mem.to(DEVICE))
#             else:
#                 loss_der = torch.tensor(0.0, device=DEVICE)

#             with torch.no_grad(): 
#                 logits_old, Z_old = old_model(xb)

#             loss_feat = (1 - F.cosine_similarity(Z, Z_old)).mean()
#             loss = loss_ce + 0.5 * loss_der + 1.0 * loss_feat
            
#             opt.zero_grad()
#             loss.backward()
#             opt.step()
            
#         # Visualizations during Finetuning
#         if epoch == 4: 
#             visualize_epoch_tsne(model, test_loader, task_id, epoch="after_epoch_5", known_semantics=set(semantic_to_internal.keys()), folder_name="tsne_05_finetune_progression")
#             model.train() # <--- 🛑 ADD THIS HERE!
#         elif epoch == 14: 
#             visualize_epoch_tsne(model, test_loader, task_id, epoch="after_epoch_15", known_semantics=set(semantic_to_internal.keys()), folder_name="tsne_05_finetune_progression")
#             model.train() # <--- 🛑 AND ADD THIS HERE!

#     model.eval()

def evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies):
    correct_per_class = defaultdict(int)
    total_per_class = defaultdict(int)
    model.eval()
    with torch.no_grad():
        for x_test, y_test in DataLoader(test_ds, batch_size=128):
            x_test = x_test.to(DEVICE)
            y_test = y_test.argmax(1).to(DEVICE)
            logits, _ = model(x_test)
            preds = logits.argmax(1)
            
            for i in range(len(y_test)):
                sem = int(y_test[i])
                if sem in semantic_to_internal:
                    internal_gt = semantic_to_internal[sem]
                    total_per_class[sem] += 1
                    if preds[i].item() == internal_gt:
                        correct_per_class[sem] += 1

    current_class_accs = {}
    for sem in semantic_to_internal.keys():
        current_class_accs[sem] = correct_per_class[sem] / max(total_per_class[sem], 1)
        
    class_accuracies[t] = current_class_accs
    total_correct = sum(correct_per_class.values())
    total_eval = sum(total_per_class.values())
    acc = total_correct / max(total_eval, 1)
    
    if t > 0:
        forgetting_list = []
        for sem in current_class_accs.keys():
            past_accs = [class_accuracies[k].get(sem, None) for k in range(t)]
            past_accs = [a for a in past_accs if a is not None]
            if past_accs:
                max_past = max(past_accs)
                forgetting = max_past - current_class_accs[sem]
                forgetting_list.append(forgetting)
        avg_forg = np.mean(forgetting_list) if forgetting_list else 0.0
    else:
        avg_forg = 0.0
        
    return acc, avg_forg, current_class_accs

# ============================================================
# GRAND SINGLE RUN: THE FINAL PIPELINE 
# ============================================================

print("\n" + "="*80)
print(f"🚀 RUNNING FINAL PIPELINE: [K-Means Warmup | Memory Anchor | Relaxed Gates]")
print("="*80)

set_seed(42)
locked_generator = get_locked_generator(42)

BASE_DIR = f"debug_FinalPipeline_{METHOD}"

# --- DESCRIPTIVE FOLDER CREATION ---
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/tsne_01_initial_state", exist_ok=True)
os.makedirs(f"{BASE_DIR}/tsne_02_pre_warmup", exist_ok=True)
os.makedirs(f"{BASE_DIR}/tsne_03_post_warmup", exist_ok=True)
os.makedirs(f"{BASE_DIR}/hdbscan_04_cluster_assignments", exist_ok=True)
os.makedirs(f"{BASE_DIR}/tsne_05_finetune_progression", exist_ok=True)
os.makedirs(f"{BASE_DIR}/tsne_06_task_end", exist_ok=True)

task_kca, task_cpr = [], []
average_forgetting = [] 
class_accuracies = {}   
learned_classes_over_time = []
promoted_classes_log = []

TASKS = [f"splitcifar10_{i}" for i in range(NUM_TASKS)]
model = CNN(num_classes=2).to(DEVICE)
memory = MemoryBuffer(max_per_class=400) 
detector = HypersphereNovelty()
novelty_buffer = []
candidate_stash = [] 

semantic_to_internal = {0: 0, 1: 1}
internal_to_semantic = {0: 0, 1: 1}
known_classes = 2

for t, task in enumerate(TASKS):
    _, dataset_fn = DATASET_REGISTRY[task]
    train_ds, test_ds, *_ = dataset_fn("./data")
    loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=locked_generator) 
    test_loader_fixed = DataLoader(test_ds, batch_size=128, shuffle=False)

    print(f"\n{'='*20} 🚀 TASK {t} {'='*20}")
    all_semantics = []
    for _, y in loader: all_semantics.extend(y.argmax(1).tolist())
    unique_semantics = list(set(all_semantics))
    class_names = [CIFAR10_LABELS[sem] for sem in unique_semantics]
    print(f"📦 [STREAM] Task {t} stream contains class(es): {class_names} (Label IDs: {unique_semantics})")
    
    if t == 0:
        print(f"🎓 [INIT] Running initialization on Task 0...")
        train_supervised(model, loader, test_loader_fixed, t, method=METHOD)
        for cls in [0, 1]:
            Xc = torch.cat([x[y.argmax(1) == cls] for x, y in loader])
            memory.build_memory_herding(Xc, cls, model)
        detector.update(memory.get(), model)
        learned_classes_over_time.append({CIFAR10_LABELS[0], CIFAR10_LABELS[1]})
        
        acc, avg_forg, current_class_accs = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
        task_kca.append(acc)
        average_forgetting.append(avg_forg)
        print(f"📈 [METRIC] Known-Class Acc: {acc:.3f}")
        
        # Vis Hook: Initial State
        visualize_tsne(model, test_loader_fixed, t, known_semantics=set(semantic_to_internal.keys()), folder_name="tsne_01_initial_state", tag="Task0_End")
        continue

    model.eval()
    novelty_candidates = []
    novel, false_novel, total = 0, 0, 0
    
    with torch.no_grad():
        for x, y in loader:
            _, z = model(x.to(DEVICE))
            y_labels = y.argmax(1)
            scores = [detector.score(z[i]).item() for i in range(len(z))]
            thr = np.percentile(scores, 30)
            for i in range(len(z)):
                total += 1
                if scores[i] > thr:
                    novelty_candidates.append((scores[i], x[i].cpu(), y_labels[i].item()))
                    novel += 1
                    if y_labels[i].item() in semantic_to_internal: false_novel += 1

    novelty_candidates.sort(reverse=True, key=lambda x: x[0])
    novelty_buffer.extend([(img, y) for _, img, y in novelty_candidates[:TOPK_NOVELTY]])
    novelty_buffer = novelty_buffer[-MAX_NOVELTY_BUFFER:]

    if t % P == 0 and len(novelty_buffer) >= MNN_K + 1: 
        # ============================================================
        # 🚀 THE SUCCESSFUL WARMUP
        # ============================================================
        print(f"     🔥 [WARMUP] Applying Pseudo-Label Contrastive & Memory Anchoring...")
        model.eval()
        
        # Vis Hook: Pre Warmup
        visualize_tsne(model, test_loader_fixed, t, known_semantics=set(semantic_to_internal.keys()), folder_name="tsne_02_pre_warmup", tag="Before_Warmup")
        
        with torch.no_grad():
            Z_buffer = []
            for img, _ in novelty_buffer:
                _, z = model(img.unsqueeze(0).to(DEVICE))
                Z_buffer.append(z.squeeze().cpu().numpy())
            Z_buffer = np.stack(Z_buffer)
            
        kmeans = KMeans(n_clusters=10, random_state=42, n_init=10).fit(Z_buffer)
        pseudo_labels = kmeans.labels_
        
        X_buffer = torch.stack([img for img, _ in novelty_buffer])
        Y_pseudo = torch.tensor(pseudo_labels, dtype=torch.long)
        warmup_loader = DataLoader(TensorDataset(X_buffer, Y_pseudo), batch_size=32, shuffle=True)
        
        model.train()
        warmup_opt = torch.optim.Adam(model.parameters(), lr=1e-4)
        aug_view = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ColorJitter(0.4, 0.4, 0.4, 0.1)])

        for warmup_epoch in range(5):
            for xb, yb_pseudo in warmup_loader:
                xb = aug_view(xb).to(DEVICE)
                yb_pseudo = yb_pseudo.to(DEVICE)
                
                _, z_novel = model(xb)
                loss_ss = margin_contrastive_loss(z_novel, yb_pseudo)
                
                X_mem, Y_mem, _ = memory.sample_balanced(32, model)
                if X_mem is not None:
                    X_mem, Y_mem = X_mem.to(DEVICE), Y_mem.to(DEVICE)
                    logits_mem, _ = model(X_mem)
                    loss_mem = F.cross_entropy(logits_mem, Y_mem)
                else:
                    loss_mem = torch.tensor(0.0).to(DEVICE)
                
                loss = loss_ss + loss_mem
                
                warmup_opt.zero_grad()
                loss.backward()
                warmup_opt.step()
                
        model.eval()
        print(f"     🌌 [WARMUP COMPLETE] Unknown clusters tightened. Memory space protected.")
        
        # Vis Hook: Post Warmup
        visualize_tsne(model, test_loader_fixed, t, known_semantics=set(semantic_to_internal.keys()), folder_name="tsne_03_post_warmup", tag="After_Warmup")
        # ============================================================
        
        print(f"\n🧠 [CLUSTERING] Running MNN (K={MNN_K}) + HDBSCAN on novelty buffer (Size: {len(novelty_buffer)})...")
        
        Z = []
        with torch.no_grad():
            for img, _ in novelty_buffer:
                _, z = model(img.unsqueeze(0).to(DEVICE))
                Z.append(z.squeeze().cpu().numpy())
        Z = np.stack(Z)

        orig_dist = sp_dist.cdist(Z, Z, metric='cosine')
        nn_model = NearestNeighbors(n_neighbors=MNN_K, metric='cosine').fit(Z)
        _, indices = nn_model.kneighbors(Z)
        
        N = len(Z)
        mnn_dist = np.full((N, N), 2.0) 
        np.fill_diagonal(mnn_dist, 0.0)
        
        for i in range(N):
            for j in indices[i]:
                if i in indices[j]: 
                    mnn_dist[i, j] = orig_dist[i, j]
                    mnn_dist[j, i] = orig_dist[i, j]
                    
        labels = hdbscan.HDBSCAN(metric='precomputed', min_cluster_size=MIN_CLUSTER_SIZE, cluster_selection_epsilon=EPSILON).fit_predict(mnn_dist)
        
        # Vis Hook: HDBSCAN Clustering
        visualize_clusters(Z, labels, t, folder_name="hdbscan_04_cluster_assignments")
        
        new_buffer = []
        found, promoted = 0, 0

        for cid in sorted(set(labels)):
            idxs = np.where(labels == cid)[0]
            if cid == -1:
                print(f"  🗑️  [CLUSTER -1] NOISE: Found {len(idxs)} noise samples.")
                for i in idxs: new_buffer.append(novelty_buffer[i])
                continue

            found += 1
            X_list = [novelty_buffer[i][0] for i in idxs]
            Xc = torch.stack(X_list)
            with torch.no_grad(): _, Zc = model(Xc.to(DEVICE))
            
            mu = F.normalize(Zc.mean(0), dim=0)
            n = len(idxs)
            S_intra = torch.mean(1 - torch.matmul(Zc, mu))
            
            if len(detector.mu) > 0:
                S_known = min([1 - torch.dot(mu, detector.mu[k].to(mu.device)) for k in detector.mu])
            else:
                S_known = torch.tensor(1.0)

            density = n / (S_intra.item() + 1e-6)
            margin = S_known - S_intra
            
            labels_true = [novelty_buffer[i][1] for i in idxs]
            sem_label, cnt = Counter(labels_true).most_common(1)[0]
            purity = cnt / len(labels_true) 
            
            if sem_label in semantic_to_internal:
                print(f"\n  ⚠️  [CLUSTER {cid}] SKIPPED: Dominant class '{CIFAR10_LABELS[sem_label]}' already known.")
                continue 

            cond_intra = S_intra.item() <= ALPHA
            cond_density = density >= DELTA
            cond_known = S_known.item() >= BETA
            cond_margin = margin.item() > -0.30

            print(f"\n  📊 [MICRO-CLUSTER {cid} EVALUATION] Dominant: '{CIFAR10_LABELS[sem_label]}' (Purity: {purity:.2f})")
            print(f"     ➔ Size:    {n:3d}   (Micro-cluster) ✅")
            print(f"     ➔ S_intra: {S_intra.item():.3f} (Req: <= {ALPHA:.2f}) {'✅' if cond_intra else '❌'}")
            print(f"     ➔ Margin:  {margin.item():.3f} (Req: > -0.30) {'✅' if cond_margin else '❌'}")

            if cond_intra and cond_density and cond_known and cond_margin:
                best_stash_idx = -1
                best_sim = -1
                
                for s_idx, stash in enumerate(candidate_stash):
                    sim = torch.dot(mu, stash['centroid'].to(DEVICE)).item()
                    if sim > best_sim:
                        best_sim = sim
                        best_stash_idx = s_idx
                
                if best_stash_idx != -1 and best_sim >= MERGE_THRESHOLD:
                    print(f"     🤝 [STASH MERGE] Matched with Stash {best_stash_idx} (Sim: {best_sim:.2f})")
                    candidate_stash[best_stash_idx]['images'].extend(X_list)
                    candidate_stash[best_stash_idx]['labels'].extend(labels_true)
                    candidate_stash[best_stash_idx]['Z'].extend([Zc[j].cpu() for j in range(n)])
                    
                    all_Z = torch.stack(candidate_stash[best_stash_idx]['Z'])
                    candidate_stash[best_stash_idx]['centroid'] = F.normalize(all_Z.mean(0), dim=0)
                    stash_ref = candidate_stash[best_stash_idx]
                else:
                    print(f"     📦 [STASH CREATED] New candidate group created.")
                    candidate_stash.append({
                        'images': X_list,
                        'labels': labels_true,
                        'Z': [Zc[j].cpu() for j in range(n)],
                        'centroid': mu.cpu()
                    })
                    stash_ref = candidate_stash[-1]
                    
                total_stash_size = len(stash_ref['images'])
                print(f"     ⏳ [WAITING ROOM] Current stash size: {total_stash_size}/{PROMOTION_THRESHOLD}")
                
                if total_stash_size >= PROMOTION_THRESHOLD:
                    promoted += 1
                    new_label = known_classes
                    semantic_to_internal[sem_label] = new_label
                    internal_to_semantic[new_label] = sem_label
                    known_classes += 1
                    
                    Xc_promoted = torch.stack(stash_ref['images'])
                    torch.save(Xc_promoted.cpu(), f"{BASE_DIR}/promoted_class_{sem_label}.pt")
                    
                    model.expand_head(known_classes)
                    model.to(DEVICE)
                    
                    print(f"     🎉 [PROMOTION TRIGGERED] -> Stash full! Finetuning '{CIFAR10_LABELS[sem_label]}'...")
                    
                    # Passing test_loader to finetune for intermediate visualizations
                    finetune(model, memory, Xc_promoted, new_label, t, test_loader_fixed, locked_generator)
                    
                    print(f"     ✨ [LEARNED] -> Network successfully adapted to '{CIFAR10_LABELS[sem_label]}'")
                    
                    memory.build_memory_herding(Xc_promoted, new_label, model)
                    detector.update(memory.get(), model)
                    promoted_classes_log.append({"task": t, "semantic": CIFAR10_LABELS[sem_label]})
                    
                    candidate_stash.remove(stash_ref)
            else:
                print(f"     🛑 [REJECTED] -> Conditions not met. Retaining samples in buffer.")
                for i in idxs: new_buffer.append(novelty_buffer[i])
                
        novelty_buffer = new_buffer
        task_cpr.append(promoted / max(found, 1))
    else:
        task_cpr.append(0.0)

    if t > 0 and t % 13 == 0:
        novelty_buffer = [] 
        candidate_stash = [] 
        print(f"\n🧹 [BUFFER CLEAR] End of multi-class wave! Flushed buffer & waiting room.")

    learned_classes_over_time.append(set(learned_classes_over_time[-1]) if len(learned_classes_over_time) > 0 else set())
    for p in promoted_classes_log:
        if p["task"] == t and p["semantic"] not in learned_classes_over_time[-1]:
            learned_classes_over_time[-1].add(p["semantic"])

    acc, avg_forg, current_class_accs = evaluate_metrics(model, test_ds, semantic_to_internal, t, class_accuracies)
    task_kca.append(acc)
    average_forgetting.append(avg_forg)
    print(f"📈 [METRIC] Known-Class Acc: {acc:.3f} | Avg Forgetting: {avg_forg:.3f}")
    
    # Vis Hook: Task End
    visualize_tsne(model, test_loader_fixed, t, known_semantics=set(semantic_to_internal.keys()), folder_name="tsne_06_task_end", tag="Task_Completed")

final_accs = class_accuracies[NUM_TASKS - 1]
row_result = {
    "HPT": "The Final Synthesis",
    "Method": METHOD,
    "Final_KCA": f"{task_kca[-1]:.3f}",
    "Avg_Forg": f"{average_forgetting[-1]:.3f}",
    "Classes_Learned": str(len(learned_classes_over_time[-1]))
}
for cls_id in range(10):
    row_result[f"Cls{cls_id}_{CIFAR10_LABELS[cls_id][:3]}"] = f"{final_accs.get(cls_id, 0.0):.2f}"

final_results_summary = [row_result]

print("\n" + "="*120)
print("📊 FINAL OPTIMIZED RUN SUMMARY (K-Means Anchor + Max Discovery Gates)")
print("="*120)

columns = ["HPT", "Method", "Final KCA", "Avg Forg", "Classes Learned"] + [f"Cls{i}" for i in range(10)]
header_format = "{:<32} | {:<18} | {:<9} | {:<8} | {:<15} | " + " | ".join([f"{{:<4}}"]*10)
print(header_format.format(*columns))
print("-" * 140)

for res in final_results_summary:
    row = [res["HPT"], res["Method"], res["Final_KCA"], res["Avg_Forg"], res["Classes_Learned"]]
    row += [res[k] for k in list(res.keys())[5:]]
    print(header_format.format(*row))

print("\n✅ Script complete. Visualizations exported to chronological directories.")


🚀 RUNNING FINAL PIPELINE: [K-Means Warmup | Memory Anchor | Relaxed Gates]

==================== 🚀 TASK 0 ====================
📦 [STREAM] Task 0 stream contains class(es): ['airplane', 'automobile'] (Label IDs: [0, 1])
🎓 [INIT] Running initialization on Task 0...
📈 [METRIC] Known-Class Acc: 0.991

==================== 🚀 TASK 1 ====================
📦 [STREAM] Task 1 stream contains class(es): ['airplane', 'automobile', 'bird', 'cat'] (Label IDs: [0, 1, 2, 3])
     🔥 [WARMUP] Applying Pseudo-Label Contrastive & Memory Anchoring...
     🌌 [WARMUP COMPLETE] Unknown clusters tightened. Memory space protected.

🧠 [CLUSTERING] Running MNN (K=5) + HDBSCAN on novelty buffer (Size: 400)...
  🗑️  [CLUSTER -1] NOISE: Found 15 noise samples.

  📊 [MICRO-CLUSTER 0 EVALUATION] Dominant: 'cat' (Purity: 0.64)
     ➔ Size:     39   (Micro-cluster) ✅
     ➔ S_intra: 0.065 (Req: <= 0.50) ✅
     ➔ Margin:  0.234 (Req: > -0.30) ✅
     📦 [STASH CREATED] New candidate group created.
     ⏳ [WAITING ROOM] Cur

In [5]:
import shutil

# This must match the BASE_DIR from your main script
folder_to_zip = "debug_FinalPipeline_margin_contrastive" 
zip_filename = "All_TSNE_Visualizations"

print(f"📦 Zipping all folders inside '{folder_to_zip}'...")

# This creates a file named 'All_TSNE_Visualizations.zip'
shutil.make_archive(zip_filename, 'zip', folder_to_zip)
print("✅ Zipping complete!")

# ---------------------------------------------------------
# IF YOU ARE USING GOOGLE COLAB, RUN THIS TO DOWNLOAD:
# ---------------------------------------------------------
try:
    from google.colab import files
    print(f"⬇️ Downloading {zip_filename}.zip to your machine...")
    files.download(f"{zip_filename}.zip")
except ImportError:
    print(f"ℹ️ Not in Colab. You can manually download '{zip_filename}.zip' from your current working directory.")

📦 Zipping all folders inside 'debug_FinalPipeline_margin_contrastive'...
✅ Zipping complete!
⬇️ Downloading All_TSNE_Visualizations.zip to your machine...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>